# Self-Healing Multimodal RAG for Bangladeshi Food — v7 (fast, Kaggle/Colab-ready)

Ask about a Bangladeshi dish in **English, Bangla script or Banglish** ("biriyani er calorie koto?"), or give a **food photo**. The system answers *what it is*, *plate nutrition*, or *shows verified pictures*, and it **repairs itself** when a step fails instead of guessing.

### Run it on Kaggle (recommended) — what to put in **Add input**
Accelerator *GPU T4 x2* (retrievers/vision on GPU 0, LLMs and judges on GPU 1), Internet *On*. Then **Add input**:

| Add input → | what | needed? |
|---|---|---|
| **Datasets** | **FoodBD** (whole folder: `MealNutrition1837_All.csv`, `data.yaml`, `train/valid/test`, `FoodBD_Meta_data.csv`) | yes |
| **Datasets** | **DeshiFoodBD** (`food_data_english/train`, `test`) | recommended |
| **Datasets** | **FoodNetBD** (`FoodNetBD*.csv`, Mendeley 23f7yxnvy7) — per-100 g nutrients + diet tags | optional |
| **Datasets** | extra folder-per-class galleries (BD-Food33, street food, desserts …) matched by `EXTRA_IMAGE_GLOBS` | optional |
| **Models** | any of the Hugging Face models below, e.g. Kaggle Models *Qwen2.5 / Transformers / 3b-instruct*, or a dataset containing a downloaded model folder — `model_src()` uses an attached copy instead of downloading (faster, no HF rate limit) | optional |
| **Notebook output** | **the output of your previous run of this notebook** — its `cache/*.pkl` (FoodBD parse, KB, embeddings, evaluator scores, benchmark features, judge scores) are reused, so a re-run skips ~20–30 min | recommended after the first run |

Section 1 prints the input index; section 13 prints which data files and which model sources (attached vs. downloaded) were used.

**Other food datasets (v8):** set `DOMAIN = "indian"` or `"food101"` in section 1 and Run All — the same pipeline, benchmark generator, threshold tuning on that domain's dev split, and the same metric tables run on another cuisine (section 16 lists what to attach; without attachments the photos are downloaded from Hugging Face). Attach the outputs of your runs to get one cross-domain table.
* **Colab**: Runtime → T4 GPU. Upload the dataset zips to `/content` (or your Drive) — section 2 unzips and finds them. Then *Run All*.
* Nothing found? The notebook switches to **SMOKE mode** (synthetic data + light fallback models) so every cell still runs on CPU in a few minutes. `MODEL_STATUS` at the end says exactly what ran.
* `PROFILE = "fast"` (default, ≈ 40–55 min on a T4 x2 including model downloads; ≈ 15–20 min when the previous run's output is attached) or `"full"` (bigger benchmark and judge sample). Every section prints its own run time; expensive results are cached in `WORK/cache` (and in the notebook output, which is why attaching it works).
* **Colab tip**: copy `/content/shrag_work/cache` to Drive after a run and back before the next one for the same speed-up.

### v6 Kaggle run: analysis, critique, fixes and alternatives

**What the v6 run showed (real data, 844 test questions, 984 test images).**
* Text: FULL v6 **0.947** accuracy vs CRAG-style **0.842** (McNemar p = 2e-16), wrong-dish (hallucination) rate **2.3 %** vs 7.9 %; phonetic spellings 0.929 vs 0.776, typos 0.845 vs 0.618, Bangla variants 1.000.
* Images: DeshiFoodBD 0.930 top-1; FoodBD crops only **0.661** (shak 0.16, potato / vegetable 0.25, rice 0.55); picture coverage 19 → 77 dishes; plate item detector micro-F1 **0.742** (rice 0.98, ruti 0.98, lau 0.36); plate router 0.98.
* Nutrition ranges covered **84.6 %** of test plates (target 80 %, v5: 65.9 %). Judge cascade **0.882** agreement vs Prometheus-only 0.824 at 21 % escalation.

| # | v6 problem (evidence from the run) | cause | v7 fix (implemented) | alternative ways to solve it |
|---|---|---|---|---|
| 1 | **Diet safety**: "is khichuri ok for diabetes?" → declined; "ডায়াবেটিস রোগী কি বিরিয়ানি খেতে পারে?" → the LLM wrote *"Yes, diabetic patients can eat biriyani"* | fuzzy span "khichuri **ok**" absorbed a word and was then rejected; the generator answered the health part; FoodNetBD tags were not parsed (layout undocumented) | a fuzzy/phonetic match must explain every word of its span; diet/recipe words are stop words; diet questions: extractive description + **only** the FoodNetBD tag; a health-claim guard in the verifier; format-agnostic tag parser that prints the raw format | answer diet questions from a structured table only (no RAG text at all); ask a clarifying question ("which condition?"); add expert-reviewed guidance text as a separate, cited KB |
| 2 | **Invented recipes**: korola vaji "made by boiling small bamboo shoots", lau "sliced and served cold" — both passed the verifier | the KB has ingredient/method lines, not recipes; NLI over a concatenated premise is lenient | recipe questions are answered extractively and say *no step-by-step recipe in the KB*; verifier = NLI **and** a lexical-support check (content words must occur in the evidence) | add a Bangla recipe corpus (Kaggle/HF recipe datasets) as a second, cited source; stronger faithfulness checkers (larger multilingual NLI, AlignScore-style); constrained decoding that copies spans |
| 3 | "0 % unsupported after healing" | measured by the same verifier that heals → circular | lexical-fail count reported separately; the external judge and the hand-written set (11c) check behaviour | label 50–100 answers by hand (ARES needs ~150 labels); a second, different verifier model as the meter |
| 4 | **Near-miss foreign dishes** declined only 68 % (baselines 89–95 %) | repairs accepted on "retrieval rank ≤ 1" with no evaluator floor; ~6 near-miss questions in dev | two new link signals tuned on dev: evaluator score of the **repaired question** and **coverage** (share of the question the repaired name explains) | an in-KB vs out-of-KB classifier on embeddings (KB names vs a foreign-dish lexicon such as Food-101 names); conformal abstention with a risk guarantee (TRAQ, C-RAG) |
| 5 | short misspellings rejected: *addo→adda, dewo→dewa, bheef→beef* | the cross-encoder scored the misspelled question | repaired-question score (see 4) | keyboard / phonetic edit models; a small learned spelling corrector trained on generated typos |
| 6 | "বাঁধাকপি কি?" → *cabbage* scored wrong (gold *bandhakopi*) | two KB entries for one food | merge by identical Bangla name and via FoodNetBD's bilingual rows (English → A, Bengali → B ⇒ same food) | an explicit synonym table reviewed by a native speaker |
| 7 | ilish per 100 g = *Hilsa (Smoked/Dried Slice)*, 415 kcal; "ফুচকা কি" answered with a per-100 g line | first matching FoodNetBD row taken; dataset facts appended to the description | best-row selection (exact name, no processing qualifier, string similarity); dataset facts kept separately and shown only when asked | let the user choose the preparation; link FoodNetBD rows by hand for the 100 most common dishes |
| 8 | plate photo: "no nutrition data for Bhat, Carrot" | no FoodBD plate with exactly that item set | new tier: presence-only model over all FoodBD plates, with its own conformal interval | YOLOv8-seg trained on FoodBD → polygon areas → the polygon nutrition model (MAE ≈ 60 kcal vs 75 for the photo regression) |
| 9 | recogniser A (tanziro) 4.6 % on FoodBD crops; CLIP prototypes 0.65 | label names not mapped to KB dishes; prototypes are weak; box crops include neighbouring food | label-mapping diagnostics + Bangla folding/transliteration; **linear probe** on CLIP-L features; **polygon-masked** crops | fine-tune an image model on FoodBD crops (EfficientNetV2 / ConvNeXt / SigLIP-2, ~20 min on a T4); a VLM (Qwen2.5-VL-3B, 4-bit) choosing among the top-5 candidates; DINOv2 kNN |
| 10 | your meat-curry photo → "chicken + beef" plate; your vegetable-curry photo → flat refusal | fixed router threshold 0.5 (p = 0.66); no middle ground between naming a dish and refusing | router threshold tuned on held-out photos and ≥ 2 items required; **category back-off** ("looks like a vegetable dish, most likely …") calibrated for precision | hierarchical classifier (category → dish); ask the user to pick from 3 thumbnails |
| 11 | `FoodBD_Meta_data.csv` has only `filename`, `instances`, `types`; v6 treated the split as a photo condition | assumed a richer schema | `instances` detected as a per-image item list and cross-checked against the polygons; split columns ignored; condition-aware intervals switch off when no condition exists | — (the file has no lighting / device metadata to use) |
| 12 | 217 s to load FoodBD, 683 s of Prometheus, nothing reused across sessions | recursive globbing per file, label files read twice, no judge cache | one input index; one parallel read; caches (incl. judge scores, evaluator pairs) are reused from an attached **previous notebook output**; models are taken from attached inputs when present | precompute everything in a separate "indexing" notebook and attach its output to the analysis notebook |
| 13 | evaluation is templated with programmatic gold labels; ~60 judged items | cheap to generate | hand-written question set (11c), bootstrap CIs on judge agreement | collect real questions from Bangladeshi users; human labels for the judge |

**Still open:** answers are written in English (Bangla generation is not evaluated); the hand-written set is small and written by the author; recogniser A may have seen DeshiFoodBD test images.

## Related work: papers, repos, benchmarks, goals, limitations, and how this notebook answers them

Numbers below are the ones each work reports **on its own benchmark**. Section 15 at the end prints them next to this run's results.

### A. Self-correcting ("self-healing") RAG, text
| Work (code) | Goal | Benchmark → reported result | Limitation (for this project) | How this notebook addresses it |
|---|---|---|---|---|
| **RAG**, Lewis et al., NeurIPS 2020 | Condition generation on retrieved passages | Open-domain QA | Retrieves blindly, no failure detection, never abstains | Evaluator-graded retrieval, explicit heal actions, abstain / clarify |
| **Self-RAG**, Asai et al., ICLR 2024 ([AkariAsai/self-rag](https://github.com/AkariAsai/self-rag)) | One LM emits reflection tokens: retrieve? relevant? supported? useful? | 7B: PopQA 54.9, PubHealth 72.4, ARC-C 67.3, Biography FactScore 81.2. 13B: PopQA 55.8, TriviaQA 69.3, PubHealth 74.5, ARC-C 73.1 | Fine-tunes a 7B/13B LM on GPT-4-labelled critic data; English; text only | Training-free, modular critics: multilingual reranker ≈ *IsRel*, mDeBERTa NLI ≈ *IsSup*, deterministic number/entity checks; runs 4-bit on a free T4 |
| **CRAG**, Yan et al., 2024 ([HuskyInSalt/CRAG](https://github.com/HuskyInSalt/CRAG)) | A T5-large (0.77B) evaluator grades retrieval → Correct / Ambiguous / Incorrect; refine; web search on failure | With SelfRAG-LLaMA2-7b, RAG → CRAG: PopQA 52.8 → 59.8, Biography 59.2 → 74.1, PubHealth 39.0 → 75.6, ARC-C 53.2 → 68.6; Self-CRAG PopQA 61.8, Biography 86.2 | Fine-tuned evaluator ("inevitable", per the authors); thresholds set by hand per dataset; web search can bring unreliable pages; 0.51 s vs 0.36 s per query | Off-the-shelf multilingual reranker; thresholds **tuned on dev**; no web: offline repair chain (rewrite → transliteration → phonetic repair) then **abstain / clarify** |
| **CRAG reproduction + SHAP** (arXiv 2603.16169) | Open-source reproduction and explanation of CRAG | The evaluator leans mainly on named-entity alignment | Entity alignment breaks on Bangla script, Banglish and typos | Explicit Bangla/Banglish **entity linker**: spelling folding, suffix stripping, fuzzy + phonetic repair, **link verification**, entity guard |
| **Adaptive-RAG**, Jeong et al., NAACL 2024 ([starsuzi/Adaptive-RAG](https://github.com/starsuzi/Adaptive-RAG)) | T5-large classifier routes queries: no retrieval / single-step / multi-step | 6 QA sets (SQuAD, NQ, TriviaQA, MuSiQue, HotpotQA, 2Wiki) | Trained router; labels from model outputs | Closed domain → cheap rule-based intent router (describe / nutrition / pictures) |
| **LangGraph CRAG / Self-RAG / Adaptive-RAG tutorials**; "self-healing RAG" posts (e.g. LangGraph + LLM-as-judge: similarity gate 0.72, rewrite, GPT-4o grounding judge, one retry, fallback, failure log) | Practical heal loops | Usually **no benchmark** (anecdotal) | Paid API judges (GPT-4o), token limits; English, text only | Same loop with **free 4-bit models**, measured: re-implemented baselines, ablations, bootstrap CIs, McNemar tests |

### B. Multimodal RAG
| Work | Goal | Benchmark → reported result | Limitation | How this notebook addresses it |
|---|---|---|---|---|
| **MRAG-Bench**, Hu et al., ICLR 2025 | Vision-centric RAG benchmark: 1,353 questions, 16,130 images, 9 scenarios | GPT-4o 68.68 → **68.96** with retrieved images (+0.28) vs 74.50 with ground-truth images; most open models get *worse* with retrieved images | Retrieved images are noisy and models cannot filter them | **Label-consistency filter** on picture retrieval (v5 run: P@5 0.41 raw → 1.00 verified); shortfalls reported, never padded |
| **mR²AG** (arXiv 2411.15041) | *Retrieval-reflection* (retrieve or not) + *relevance-reflection* (find the evidence) for knowledge-based VQA | INFOSEEK, Encyclopedic-VQA; reports beating GPT-4V/4o and RAG-MLLMs | Instruction-tunes an MLLM; English encyclopedic entities | Reflection by separate free parts: recogniser disagreement → kNN vote; calibrated τ decides "name it or not"; evidence verifier |
| **MMed-RAG**, Xia et al., ICLR 2025 | Domain-aware retrieval, adaptive context selection, RAG preference fine-tuning | 5 medical datasets: **+43.8 %** average factual accuracy | Preference fine-tuning of a Med-LVLM; medical domain | Adaptive context = **entity guard** (evidence scoped to linked dishes); no fine-tuning |
| Multimodal-RAG surveys (arXiv 2502.08826, 2504.08748) | Taxonomy: selection / context / response hallucinations | — | English documents; regional cuisines absent | Bangladeshi food KB plus Bangla food recognisers; heals selection (picture filter) and recognition errors (disagreement + calibrated abstention) |

### C. Judges and evaluation
| Work | Goal | Reported | Limitation | Here |
|---|---|---|---|---|
| **RAGAS** (arXiv 2309.15217), **ARES** (arXiv 2311.09476) | Reference-free LLM-judge RAG metrics | — | RAGAS judge unvalidated; ARES needs ~150 human labels | Judges validated against programmatic gold (agreement, Cohen's κ); CSV exported for a small human check |
| **Prometheus 2**, Kim et al., EMNLP 2024 ([prometheus-eval](https://github.com/prometheus-eval/prometheus-eval)) | Open 7B / 8x7B evaluator, rubric scoring and pairwise | Highest human/GPT-4 correlation among open evaluators; pairwise gap to GPT-4 roughly halved | Slow on a T4 (v5: ~13 s per item, unbatched); not tuned for Bangla | Escalation judge only, batched, 4-bit |
| **JEV-as-a-Judge**, Li, Miao, Krishnan, Padman, 2026 (arXiv 2609.26550) | Decision-only judge returns label probabilities; accept when confident, escalate when unsure | RewardBench, JudgeBench, HaluEval, RM-Bench, PPE: within ~3 pts of GPT-6 on preference/factuality (RewardBench 92.5 = tie; HaluEval 87.3 vs 88.4), ~277x cheaper (0.15 s median); cascade at τ = 0.9: 93.4 % vs 92.5 % on 1,610 held-out pairs, escalating 31.5 %; live test 90.2 % = 90.2 % | Hosted API (fees / tokens); 7–28 pts behind where the verdict must be derived (JudgeBench 78.6 vs 93.1); confidence weaker on style-adversarial pairs (AUROC 0.76 vs 0.84); τ needs ~100 local labels | Free re-creation: Phi-3.5-mini P(Yes)/P(No) + Prometheus-2; τ tuned on local labels (calibration half) and reported on the other half |

### D. Bangladeshi food data and models
| Work | Content | Reported | Limitation | Here |
|---|---|---|---|---|
| **FoodBD**, BMC Research Notes 2025 (Mendeley xh3ghf3jbg v2) | 3,523 meal photos, 12,575 polygons, 67 classes (Bangla + English), 1,837 with expert nutrition (CHO, protein, fat, fiber, kcal, GL), `FoodBD_Meta_data.csv` (instances, conditions, preprocessing) | Dataset paper | No QA; nutrition per plate | Polygons → nutrition model + picture crops; nutrition → grounded numbers with conformal ranges; metadata → Bangla names, condition-aware intervals, KB facts |
| **Polygon-aware meal nutrition**, Yasin, Yasin & Koklu, J. Food Science 2026 | YOLOv8n polygons → shape / colour / texture features → regressors (XGBoost best) | FoodBD v2: mask mAP@0.5 0.423, instance F1 0.87, polygon features +11.63 % on average | No uncertainty; no QA | Split-conformal intervals; nutrition is a grounded tool inside RAG, never generated |
| **DeshiFoodBD** (Mendeley tczzndbprx, 2021) | 5,425 images, 19 dishes, English + Bengali labels | Classification dataset | Recognition only | Recognition / calibration / test images + picture gallery |
| **FoodNetBD** (Mendeley 23f7yxnvy7, 2026), optional | 600 foods, 33 nutrients per 100 g, English + Bengali names, diet tags for 12 conditions | Static table | No QA, no linking | Linked to KB dishes → per-100 g answers instead of a FoodBD proxy class (raw ingredients never mapped to cooked dishes) |
| **tanziro/bangladeshi-food** + EfficientNetV2-S recogniser (Hugging Face) | Dish metadata (description, ingredients, method, region) + an 841-class recogniser | — | Recogniser trained on pooled DeshiFoodBD, so test leakage is possible | KB text source; recogniser A, with the leakage caveat printed |

### E. Newer or stronger work found while reviewing the v6 run, and what this notebook takes from it
| Work / resource | What it does | Reported | Relevance and limitation | Used here? |
|---|---|---|---|---|
| **Test-time Corpus Feedback** survey (Rathee et al., Findings EACL 2026) | Taxonomy of query-, retrieval- and generation-level feedback in RAG (PRF, rewriting, FLARE/DRAGIN/CRAG triggers, Self-RAG, Search-R1) | open problems: cost, noisy feedback, **no explicit policy for when to rewrite / re-retrieve**, feedback attribution | this notebook's controller *is* an explicit, logged policy; each heal action is an ablation (attribution) | design rationale |
| **TRAQ** (NAACL 2024) / **C-RAG** (ICML 2024) | conformal prediction for RAG: correctness / risk guarantees with high probability | TRAQ: guarantee met, prediction sets 16.2 % smaller than an ablation; C-RAG: certified risk bounds hold under bounded shift | the principled version of "abstain when unsure"; needs a calibration set | conformal intervals for nutrition + calibrated image threshold; full conformal abstention = future work |
| **FoodieQA** (EMNLP 2024) | multimodal benchmark of regional Chinese food culture (1,364 questions) | humans 91.7 % (multi-image VQA); GPT-4o 86.4 %; best open model 50.9 % | open VLMs are weak on regional dishes → do not trust a VLM alone for Bangladeshi food | motivates retrieval + calibrated recognisers instead of a VLM-only answer |
| **DiningBench** (2026) | 3,021 dishes, 15,928 images: fine-grained classification, nutrition estimation, VQA | best classification 81.8 %; nutrition is hardest (best 24.5 % MAPE); CoT often hurts perception | nutrition from photos is hard even for frontier VLMs → keep data-grounded numbers with intervals | yes (design) |
| **TigerLLM** (ACL 2025, Hugging Face `md-nishat-008/TigerLLM-1B-it`) | Bangla LLM family | claims to beat open Bangla models and GPT-3.5 | candidate generator for **Bangla-language answers** (not evaluated yet) | option (`GEN_MODEL`) |
| **Qwen2.5-VL-3B** (Hugging Face / Kaggle Models), **SigLIP-2**, **DINOv2** | open VLM / image encoders | — | alternatives for recogniser B or a constrained "pick one of these 5 dishes" check | listed as alternatives (row 9 above) |
| **YOLOv8-seg** on FoodBD (Yasin et al. 2026: mask mAP@0.5 0.42, instance F1 0.87) | per-item masks → polygon features | +11.63 % nutrition accuracy from polygon features | would let photo nutrition use the polygon model (MAE ≈ 60 kcal vs ≈ 75) | future work (needs ~30–45 min training on a T4; save weights as a Kaggle Model and attach) |
| GitHub: `AkariAsai/self-rag`, `HuskyInSalt/CRAG`, `starsuzi/Adaptive-RAG`, `prometheus-eval/prometheus-eval`, `explodinggradients/ragas`, LangGraph CRAG / Self-RAG tutorials | reference implementations | — | English, text-only, mostly paid-API judges | mechanisms re-implemented with free 4-bit models (B0–B3 baselines) |

### Goal (testable)
For questions about Bangladeshi dishes, in English, Bangla script or Banglish, from text or a photo:
1. Answer correctly whenever the knowledge supports it.
2. **Never invent a dish, number or fact**: abstain or ask instead.
3. Stay robust to spelling variants.
4. Name a dish from a photo only when calibrated confidence reaches a target precision.
5. Give numbers only from data, with calibrated ranges.

All of this runs on free Kaggle/Colab GPUs with 4-bit and fp16 models.

**Metrics**: accuracy, hallucination rate, abstention F1, Recall@k / MRR, unsupported-claim rate, image selective accuracy and ECE, picture P@5, nutrition MAE + conformal coverage, external-judge agreement. All are measured on a held-out **test** split; thresholds are tuned only on **dev**.

**Gap this fills**: none of the works above combines Bangla / Banglish / English queries, text + image, self-correction with abstention, free quantized models, and a measured comparison against re-implemented baselines.

### Limitations of this project
* Benchmark questions are generated from templates over the KB, and the gold labels are programmatic. A small human-labelled set is the next step (`judge_items_for_human_check.csv` is exported for this).
* Recogniser A may have seen DeshiFoodBD test images (pooled training). Fresh phone photos are the clean test.
* Answers are written in English; Bangla-language answers are not evaluated.
* FoodBD nutrition is per plate (portion-dependent); per-100 g values need FoodNetBD.
* The judges are validated against programmatic labels, not people.

In [ ]:
# 0. Install only what is missing (no "-U": upgrading transformers/huggingface-hub on Kaggle breaks preinstalled packages)
import importlib.util, subprocess, sys
_need = {"rapidfuzz": "rapidfuzz", "sentence_transformers": "sentence-transformers", "bitsandbytes": "bitsandbytes",
         "datasets": "datasets", "timm": "timm", "pyarrow": "pyarrow"}
_missing = [pip for mod, pip in _need.items() if importlib.util.find_spec(mod) is None]
if _missing:
    print("installing:", _missing)
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + _missing, capture_output=True, text=True)
    print("pip ok" if r.returncode == 0 else "pip failed (offline?) -> fallbacks will be used\n" + r.stderr[-400:])
else:
    print("all packages present")

## 1. Setup, run profile, caching, timers

In [ ]:
import os, re, json, math, time, glob, random, zipfile, pickle, hashlib, warnings, unicodedata, itertools, collections, contextlib
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
SEED = 0; random.seed(SEED); np.random.seed(SEED)

ENV = "kaggle" if os.path.exists("/kaggle/input") else ("colab" if os.path.exists("/content") else "local")
def _hf_token():
    """Optional: a free HF read token (Kaggle: Add-ons > Secrets > HF_TOKEN; Colab: key icon > HF_TOKEN) gives faster, rate-limit-free downloads."""
    if os.environ.get("HF_TOKEN"): return
    try:
        if ENV == "kaggle":
            from kaggle_secrets import UserSecretsClient; os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
        elif ENV == "colab":
            from google.colab import userdata; os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception: pass                      # no secret set -> anonymous downloads still work
_hf_token()
WORK = {"kaggle": "/kaggle/working", "colab": "/content/shrag_work"}.get(ENV, os.path.join(os.getcwd(), "shrag_work"))
CACHE = os.path.join(WORK, "cache"); os.makedirs(CACHE, exist_ok=True)
ROOTS = list(dict.fromkeys(os.path.realpath(r) for r in ("/kaggle/input", "/content", "/content/drive/MyDrive", WORK, ".") if os.path.exists(r)))

# ---- v7: ONE index of everything attached ("Add input" on Kaggle: datasets, models, notebook outputs; uploads/Drive on Colab) ----
# v6 ran a recursive glob over all inputs for every file it looked for (thousands of images each time; 'load FoodBD' took 217 s).
import fnmatch
_IMG_EXT = (".jpg", ".jpeg", ".png", ".webp", ".bmp", ".gif")
def _build_index(roots):
    files, dirs, seen = [], [], set()
    for r in roots:
        for dp, dn, fn in os.walk(r):
            rp = os.path.realpath(dp)
            if rp in seen: dn[:] = []; continue
            seen.add(rp); dirs.append(dp)
            deep = r.endswith("MyDrive") and dp[len(r):].count(os.sep) >= 4             # Google Drive: do not crawl the whole drive
            dn[:] = [] if deep else sorted(d for d in dn if not d.startswith(".") and d not in ("__pycache__", "sample_data") and
                                           not (dp == "/content" and d == "drive") and os.path.realpath(os.path.join(dp, d)) != os.path.realpath(CACHE))
            files += [os.path.join(dp, f) for f in fn if not f.lower().endswith(_IMG_EXT)]
    return files, dirs
_t0 = time.time(); IDX_FILES, IDX_DIRS = _build_index(ROOTS)
def _match(paths, pattern):
    pat = pattern.replace("**/", "")
    return sorted(p for p in paths if (fnmatch.fnmatch(p, "*/" + pat) if "/" in pat else fnmatch.fnmatch(os.path.basename(p), pat)))
def find_first(pattern):
    hits = _match(IDX_FILES, pattern) or _match(IDX_DIRS, pattern)
    return hits[0] if hits else None
def find_dirs(pattern): return _match(IDX_DIRS, pattern)

# models attached from Kaggle Models / a dataset: any folder with config.json whose path names the model ("qwen2.5/transformers/3b-instruct")
MODEL_DIRS = sorted({os.path.dirname(p) for p in IDX_FILES if os.path.basename(p) in ("config.json", "config_sentence_transformers.json")})
def model_src(hf_id):
    """Local copy of a Hugging Face model if one is attached as an input (no download, no HF rate limit), else the HF id."""
    name = hf_id.split("/")[-1].lower(); toks = [t for t in re.split(r"[-_]", name) if t]
    for d in sorted(MODEL_DIRS, key=len):
        low = d.lower()
        if "/kaggle/working" in low: continue
        if name in low or (len(toks) >= 2 and all(t in low for t in toks)):
            MODEL_SOURCES[hf_id] = d; return d
    MODEL_SOURCES[hf_id] = "huggingface"; return hf_id
MODEL_SOURCES = {}
# caches from an earlier run: attach that notebook's output (Add input -> Notebook output) and its cache/*.pkl files are reused
INPUT_CACHES = sorted({os.path.dirname(p) for p in IDX_FILES if p.endswith(".pkl") and os.path.basename(os.path.dirname(p)) == "cache"
                       and os.path.realpath(os.path.dirname(p)) != os.path.realpath(CACHE)})
def cache_file(fname):
    """Path of a cache file: this run's cache, else a cache attached from a previous run's output."""
    p = os.path.join(CACHE, fname)
    if os.path.exists(p): return p
    for d in INPUT_CACHES:
        if os.path.exists(os.path.join(d, fname)): return os.path.join(d, fname)
    return None
print(f"input index: {len(IDX_FILES)} files, {len(IDX_DIRS)} folders in {ROOTS} ({time.time() - _t0:.1f}s) | attached model folders: {len(MODEL_DIRS)} | "
      f"previous-run caches: {INPUT_CACHES or 'none'}")

try:
    import torch
    HAS_TORCH = True; DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    HAS_TORCH = False; DEVICE = "cpu"
GPU = DEVICE == "cuda"
N_GPU = torch.cuda.device_count() if GPU else 0
LLM_GPU = 1 if N_GPU > 1 else 0                         # T4 x2: retrievers/vision on GPU 0, LLMs + judges on GPU 1
HAS_ST = importlib.util.find_spec("sentence_transformers") is not None
HAS_TF = importlib.util.find_spec("transformers") is not None

PROFILE = os.environ.get("SHRAG_PROFILE", "fast")      # "fast" | "full"
FORCE_SMOKE = os.environ.get("SHRAG_SMOKE") == "1"

# ---- v8: run the SAME pipeline on another food domain (text KB + image set + pretrained recogniser) ----
# "bd"      Bangladeshi food: FoodBD + DeshiFoodBD + tanziro KB + FoodNetBD (the main study)
# "indian"  Indian Food 101 (Kaggle nehaprabhavalkar/indian-food-101, indian_food.csv: 255 dishes) + Indian Food Images
#           (Kaggle iamsouravbanerjee/indian-food-images-dataset: 80 classes) or, if not attached, Hugging Face rajistics/indian_food_images
#           (20 classes); recogniser A = rajistics/finetuned-indian-food (ViT, 93.3 % on its own eval split; trained on that HF set -> leakage caveat)
# "food101" Food-101 (Kaggle dansbecker/food-101 or kmader/food41: 101 classes, meta/test.txt split); KB text = Wikipedia summaries
#           of the class names (Internet on); recogniser A = nateraw/food (ViT fine-tuned on Food-101 train; we test on its test list)
# "custom"  your own: CUSTOM_KB_CSV (a name column + description or any descriptive columns) and CUSTOM_IMG_ROOT (folder per class)
DOMAIN = os.environ.get("SHRAG_DOMAIN") or "bd"          # <<< change to "indian", "food101" or "custom" to test another dataset (section 16)
CUSTOM_KB_CSV, CUSTOM_IMG_ROOT, CUSTOM_CLF, CUSTOM_LABEL = None, None, None, "custom food"   # used when DOMAIN = "custom"
DOMAIN_SPEC = {
    "bd":      dict(label="Bangladeshi food", clf="tanziro/bangla-food-recognizer-efficientnetv2s-a100-3h"),
    "indian":  dict(label="Indian food", kb=["**/indian_food.csv", "**/*ndian*ood*.csv"], img=["**/Indian Food Images/Indian Food Images", "**/Indian Food Images", "**/*ndian*ood*mages*"], clf="rajistics/finetuned-indian-food"),
    "food101": dict(label="Food-101 dishes", kb=None, img=["**/food-101/images", "**/food101/images", "**/images"], test_list=["**/food-101/meta/test.txt", "**/meta/test.txt"], clf="nateraw/food"),
    "custom":  dict(label=CUSTOM_LABEL, kb=[CUSTOM_KB_CSV] if CUSTOM_KB_CSV else [], img=[CUSTOM_IMG_ROOT] if CUSTOM_IMG_ROOT else [], clf=CUSTOM_CLF),
}[DOMAIN]
DOMAIN_LABEL = DOMAIN_SPEC["label"]
def _first_path(pats, want_dir=False):
    for p in pats or []:
        if p and os.path.isabs(p) and os.path.exists(p): return p
        hit = (find_dirs(p)[:1] if want_dir else [find_first(p)]) if p else [None]
        if hit and hit[0] and (not want_dir or any(os.path.isdir(os.path.join(hit[0], d)) for d in os.listdir(hit[0])[:50])): return hit[0]
    return None
DOMAIN_KB_PATH = _first_path(DOMAIN_SPEC.get("kb")) if DOMAIN != "bd" else None
DOMAIN_IMG_ROOT = _first_path(DOMAIN_SPEC.get("img"), want_dir=True) if DOMAIN != "bd" else None
DOMAIN_TEST_LIST = _first_path(DOMAIN_SPEC.get("test_list")) if DOMAIN == "food101" else None
# no Kaggle image folder attached -> download a per-class sample from Hugging Face (needs Internet On) into a class-folder layout
HF_IMG = {"food101": [("ethz/food101", "validation"), ("food101", "validation")],                 # nateraw/food was trained on 'train'
          "indian": [("rajistics/indian_food_images", "test"), ("rajistics/indian_food_images", "validation"), ("rajistics/indian_food_images", "train")]}
DOMAIN_IMG_PER_CLASS = int(os.environ.get("SHRAG_IMG_PER_CLASS", 40))
def hf_images_to_folder(repo, split, per_class, out):
    from datasets import load_dataset
    ds = load_dataset(repo, split=split, streaming=True)
    names = ds.features["label"].names if ds.features and "label" in ds.features else None
    cnt, t0 = collections.Counter(), time.time()
    for ex in ds:
        lab = names[ex["label"]] if names else str(ex["label"])
        if cnt[lab] >= per_class:
            if names and all(cnt[n] >= per_class for n in names): break
            continue
        d = os.path.join(out, lab); os.makedirs(d, exist_ok=True)
        ex["image"].convert("RGB").save(os.path.join(d, f"{cnt[lab]:04d}.jpg"), quality=90); cnt[lab] += 1
    print(f"  {repo}[{split}]: {sum(cnt.values())} images, {len(cnt)} classes in {time.time() - t0:.0f}s -> {out}")
    return out if cnt else None
DOMAIN_SYNTH, DOMAIN_IMG_SOURCE = False, None
if DOMAIN != "bd" and DOMAIN_IMG_ROOT is None and not FORCE_SMOKE:
    for repo, sp in HF_IMG.get(DOMAIN, []):
        out = os.path.join(WORK, "hf_images", repo.replace("/", "__") + "_" + sp)
        try:
            if os.path.isdir(out) and len(os.listdir(out)) > 1: DOMAIN_IMG_ROOT = out
            else: DOMAIN_IMG_ROOT = hf_images_to_folder(repo, sp, DOMAIN_IMG_PER_CLASS, out)
        except Exception as e: print(f"  HF images {repo}[{sp}] not available: {type(e).__name__}: {str(e)[:120]}")
        if DOMAIN_IMG_ROOT: DOMAIN_IMG_SOURCE = f"Hugging Face {repo} [{sp}]"; break
elif DOMAIN_IMG_ROOT: DOMAIN_IMG_SOURCE = f"attached input {DOMAIN_IMG_ROOT}"
DOMAIN_OK = DOMAIN == "bd" or (DOMAIN_IMG_ROOT is not None and (DOMAIN_KB_PATH is not None or DOMAIN == "food101"))
if DOMAIN != "bd" and not DOMAIN_OK:
    # synthetic stand-in for the domain (SMOKE): a small KB csv in the Indian-Food-101 layout + coloured-blob images, incl. 2 classes
    # that are NOT in the KB (they must be declined), so every new code path runs on CPU
    DOMAIN_SYNTH = True; _sd = os.path.join(WORK, f"synth_{DOMAIN}"); os.makedirs(_sd, exist_ok=True)
    _dishes = [("Masala dosa", "rice, urad dal, potato", "vegetarian", "main course", "Karnataka", "South", "spicy"),
               ("Idli", "rice, urad dal", "vegetarian", "snack", "Tamil Nadu", "South", "-1"), ("Samosa", "flour, potato, peas", "vegetarian", "snack", "Punjab", "North", "spicy"),
               ("Jalebi", "maida, sugar, ghee", "vegetarian", "dessert", "Uttar Pradesh", "North", "sweet"),
               ("Chole bhature", "chickpeas, maida, spices", "vegetarian", "main course", "Punjab", "North", "spicy"),
               ("Pav bhaji", "bread, potato, butter, vegetables", "vegetarian", "main course", "Maharashtra", "West", "spicy"),
               ("Kulfi", "milk, sugar, pistachio", "vegetarian", "dessert", "Delhi", "North", "sweet"), ("Dhokla", "gram flour, yogurt", "vegetarian", "snack", "Gujarat", "West", "sweet"),
               ("Rogan josh", "lamb, yogurt, chillies", "non vegetarian", "main course", "Jammu & Kashmir", "North", "spicy"),
               ("Gulab jamun", "milk powder, sugar syrup", "vegetarian", "dessert", "West Bengal", "East", "sweet"),
               ("Butter chicken", "chicken, butter, tomato, cream", "non vegetarian", "main course", "Punjab", "North", "spicy"),
               ("Pani puri", "semolina, potato, tamarind water", "vegetarian", "snack", "Maharashtra", "West", "spicy"),
               ("Kadai paneer", "paneer, capsicum, tomato", "vegetarian", "main course", "Punjab", "North", "spicy"),
               ("Chapati", "whole wheat flour, water", "vegetarian", "main course", "-1", "North", "-1")]
    if DOMAIN in ("indian", "custom"):
        DOMAIN_KB_PATH = os.path.join(_sd, "indian_food.csv")
        pd.DataFrame([(n, ing, d, 15, 30, fl, c, st, rg) for n, ing, d, c, st, rg, fl in _dishes],
                     columns=["name", "ingredients", "diet", "prep_time", "cook_time", "flavor_profile", "course", "state", "region"]).to_csv(DOMAIN_KB_PATH, index=False)
    from PIL import Image as _Im, ImageDraw as _Dr
    _cls = [n.lower().replace(" ", "_") for n, *_ in _dishes[:10]] + ["pizza", "burger"]          # pizza/burger: not in the KB -> must decline
    DOMAIN_IMG_ROOT = os.path.join(_sd, "images"); _rng = np.random.RandomState(11)
    for ci, c in enumerate(_cls):
        d = os.path.join(DOMAIN_IMG_ROOT, c); os.makedirs(d, exist_ok=True); base = _rng.randint(40, 215, size=3)
        for i in range(24):
            p = os.path.join(d, f"{i}.png")
            if os.path.exists(p): continue
            col = np.clip(base + _rng.randint(-25, 26, 3), 0, 255); im = _Im.new("RGB", (64, 64), tuple(int(v) for v in col)); dr = _Dr.Draw(im)
            for _ in range(3 + ci % 4):
                x, y, r = _rng.randint(0, 64), _rng.randint(0, 64), _rng.randint(3, 8)
                dr.ellipse([x - r, y - r, x + r, y + r], fill=tuple(int(v) for v in np.clip(255 - col + _rng.randint(-50, 50, 3), 0, 255)))
            im.save(p)
    DOMAIN_IMG_SOURCE = "synthetic (SMOKE)"
if DOMAIN != "bd":
    print(f"DOMAIN={DOMAIN} ({DOMAIN_LABEL}) | KB file: {DOMAIN_KB_PATH} | image root: {DOMAIN_IMG_ROOT} | test list: {DOMAIN_TEST_LIST} | "
          + ("synthetic stand-in (SMOKE)" if DOMAIN_SYNTH else f"found | images from {DOMAIN_IMG_SOURCE}"))
CFG = dict(
    EMBED_MODEL="BAAI/bge-m3",                         # multilingual dense retriever (Bangla included)
    BASELINE_EMBED="BAAI/bge-small-en-v1.5",           # v1/v2 English embedder, kept as a baseline
    RERANK_MODEL="BAAI/bge-reranker-v2-m3",            # CRAG-style retrieval evaluator
    NLI_MODEL="MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7",
    GEN_MODEL="Qwen/Qwen2.5-3B-Instruct" if N_GPU > 1 else "Qwen/Qwen2.5-1.5B-Instruct",   # generator, 4-bit
    DECISION_JUDGE="microsoft/Phi-3.5-mini-instruct",  # JEV-style decision-only judge, 4-bit, different family
    JUDGE_MODEL="prometheus-eval/prometheus-7b-v2.0",  # escalation judge, 4-bit
    JUDGE_FALLBACK="Qwen/Qwen2.5-3B-Instruct",
    CLF_MODEL=DOMAIN_SPEC["clf"],                       # recogniser A (None -> CLIP zero-shot over the class names)
    CLIP_MODEL="openai/clip-vit-large-patch14",          # v6: ViT-L/14 (fp16 autocast); v5 used ViT-B/32
    USE_LLM=True, USE_JUDGE=True,
    BENCH_MAX=1200 if PROFILE == "fast" else 3000,     # benchmark size (stratified sample)
    BENCH_GEN_N=80 if PROFILE == "fast" else 200,      # test answers generated with the LLM
    JUDGE_N=120 if PROFILE == "fast" else 240,         # answers sent to the judges (Prometheus is batched)
    IMG_CALIB_N=300, IMG_TEST_N=400 if PROFILE == "fast" else 1000,
    MAX_HEAL_STEPS=2, TARGET_IMG_PRECISION=0.90, JUDGE_TARGET_ACC=0.90,
)
MODEL_STATUS, TIMINGS = {}, {}

@contextlib.contextmanager
def section(name):
    t = time.time(); print(f"--- {name} ...", flush=True)
    yield
    TIMINGS[name] = round(time.time() - t, 1); print(f"--- {name} done in {TIMINGS[name]:.1f}s", flush=True)

def cached(name, fn, key="", store_if=lambda obj: True):
    """Pickle cache in WORK/cache; key should change whenever the inputs/models change."""
    fname = f"{name}_{hashlib.md5(str(key).encode()).hexdigest()[:10]}.pkl"; p = os.path.join(CACHE, fname); src = cache_file(fname)
    if src:
        with open(src, "rb") as f: obj = pickle.load(f)
        if src != p:
            print(f"  (reused {name} from an attached previous run: {src})")
            with open(p, "wb") as f: pickle.dump(obj, f)                 # keep it in this run's output too
        return obj
    obj = fn()
    if store_if(obj):                                     # e.g. never cache a heuristic stand-in that replaced a model which failed to load
        with open(p, "wb") as f: pickle.dump(obj, f)
    return obj

def try_load(tag, fn):
    t = time.time()
    try:
        obj = fn(); MODEL_STATUS[tag] = f"loaded ({time.time() - t:.0f}s)"; return obj
    except Exception as e:
        MODEL_STATUS[tag] = "FALLBACK - " + type(e).__name__ + ": " + str(e).split("\n")[0][:110]; return None

def gpu_free_gb(i=None):
    if not GPU: return 0.0
    free, _ = torch.cuda.mem_get_info(LLM_GPU if i is None else i); return free / 2**30

def load_causal_lm(name, gpu=None):
    """4-bit NF4 on GPU (bitsandbytes); fp16 only if bitsandbytes is unusable (never after an OOM); fp32 on CPU."""
    from transformers import AutoTokenizer, AutoModelForCausalLM
    name = model_src(name); tok = AutoTokenizer.from_pretrained(name); gpu = LLM_GPU if gpu is None else gpu
    if GPU:
        print(f"loading {name} on cuda:{gpu} ({gpu_free_gb(gpu):.1f} GB free)")
        try:
            from transformers import BitsAndBytesConfig
            q = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
            return tok, AutoModelForCausalLM.from_pretrained(name, quantization_config=q, device_map={"": gpu}).eval()
        except torch.cuda.OutOfMemoryError:
            free_gpu(); raise                                    # fp16 would need ~4x the memory: give up instead
        except Exception as e:
            print("4-bit load failed -> fp16:", repr(e)[:150])
        return tok, AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.float16, device_map={"": gpu}).eval()
    return tok, AutoModelForCausalLM.from_pretrained(name).eval()

def encode_chat(tok, messages):
    if getattr(tok, "chat_template", None):
        return tok.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True)
    return tok("[INST] " + "\n\n".join(m["content"] for m in messages) + " [/INST]", return_tensors="pt")

def chat_generate(tok, model, messages, max_new_tokens=200):
    enc = encode_chat(tok, messages).to(model.device)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][enc["input_ids"].shape[-1]:], skip_special_tokens=True).strip()

def chat_generate_batch(tok, model, prompts, max_new_tokens=200, bs=8):
    """Left-padded batched greedy generation (v5 judged one item at a time: ~13 s/item for Prometheus-7B on a T4)."""
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    side = tok.padding_side; tok.padding_side = "left"; outs = []
    def render(p):
        m = [{"role": "user", "content": p}]
        return tok.apply_chat_template(m, add_generation_prompt=True, tokenize=False) if getattr(tok, "chat_template", None) else f"[INST] {p} [/INST]"
    try:
        for i in range(0, len(prompts), bs):
            enc = tok([render(p) for p in prompts[i:i + bs]], return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
            with torch.no_grad():
                out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.pad_token_id)
            outs += [tok.decode(o[enc["input_ids"].shape[-1]:], skip_special_tokens=True).strip() for o in out]
    finally: tok.padding_side = side
    return outs

import contextlib
def amp():
    """fp16 mixed precision on GPU (T4 tensor cores: ~2x faster, half the activation memory); no-op on CPU."""
    return torch.autocast("cuda", dtype=torch.float16) if GPU else contextlib.nullcontext()

def free_gpu():
    import gc; gc.collect()
    if GPU: torch.cuda.empty_cache()

try:
    from rapidfuzz import fuzz as _fz, process as _proc
    def fratio(a, b): return _fz.ratio(a, b)
    def best_match(q, choices, cutoff):
        m = _proc.extractOne(q, choices, scorer=_fz.ratio, score_cutoff=cutoff)
        return (m[0], m[1]) if m else None
except ImportError:
    import difflib
    def fratio(a, b):
        sm = difflib.SequenceMatcher(None, a, b)
        return 100.0 * sm.ratio() if sm.real_quick_ratio() * 100 >= 50 else 0.0
    def best_match(q, choices, cutoff):
        best, bs = None, cutoff
        for c in choices:
            s = fratio(q, c)
            if s >= bs: best, bs = c, s
        return (best, bs) if best is not None else None

print(f"env={ENV} | device={DEVICE} x{N_GPU} (LLMs on cuda:{LLM_GPU}) | profile={PROFILE} | torch={HAS_TORCH} sentence-transformers={HAS_ST} transformers={HAS_TF} | work={WORK}")

## 2. Data: FoodBD plates (portion polygons + nutrition) and dataset discovery

**Where to get the data (all free):**
* FoodBD — Mendeley Data (polygon-annotated plates + `MealNutrition1837_All.csv`), paper: PMC12801922.
* DeshiFoodBD — Kaggle / Mendeley (`food_data_english/train|test`, folder-per-class).
* Optional extra image galleries (any folder-per-class dataset works, class names are linked to KB dishes automatically): BD-Food33 (Mendeley 8wt48bjmx3), Bangladeshi Street Food (Mendeley pjkrx3kk8h), Traditional desserts of Bangladesh (Mendeley j2pnx2mwwk). Put their folders/zips in the input roots and list a glob in `EXTRA_IMAGE_GLOBS`.

On Colab, zips uploaded to `/content` or `MyDrive` are extracted automatically. If FoodBD is missing, a synthetic stand-in is built (SMOKE mode).

In [ ]:
EXTRA_IMAGE_GLOBS = ["**/BD-Food33*", "**/*treet*ood*", "**/*esserts*"]    # edit freely; folders with class subfolders

if ENV in ("colab", "local"):
    for z in glob.glob("/content/*.zip") + glob.glob("/content/drive/MyDrive/*.zip") + glob.glob(os.path.join(WORK, "*.zip")):
        out = os.path.join(WORK, "data", os.path.basename(z)[:-4])
        if not os.path.exists(out):
            try: zipfile.ZipFile(z).extractall(out); print("extracted", z)
            except Exception as e: print("could not extract", z, e)

NUTR = ["CHO", "Protein", "Fat", "Fiber", "Calorie", "GL"]

def poly_area(coords):
    c = np.asarray(coords, dtype=float)
    if len(c) % 2: c = c[:-1]
    x, y = c[0::2], c[1::2]
    if len(x) < 3: return 0.0
    return 0.5 * abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))

def load_foodbd(csv_path):
    import yaml
    root = os.path.dirname(csv_path)
    names = yaml.safe_load(open(os.path.join(root, "data.yaml")))["names"]
    if isinstance(names, dict): names = [names[k] for k in sorted(names)]
    nut = pd.read_csv(csv_path); label_map = {}
    for lf in glob.glob(f"{root}/*/labels/*.txt"):
        label_map[os.path.splitext(os.path.basename(lf))[0]] = (lf, os.path.basename(os.path.dirname(os.path.dirname(lf))))
    img_map = {os.path.splitext(os.path.basename(p))[0]: p for p in glob.glob(f"{root}/*/images/*") if p.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))}
    # v7: every label file is read ONCE, in parallel (v6 opened each file twice, one after another: 217 s on Kaggle's network-mounted input)
    from concurrent.futures import ThreadPoolExecutor
    def _rd(lf):
        with open(lf) as fh: return fh.read().splitlines()
    with ThreadPoolExecutor(32) as ex: LINES = dict(zip(label_map, ex.map(_rd, [v[0] for v in label_map.values()])))
    inst = []                                  # every polygon in every annotated image: used for the picture gallery and the metadata join
    for stem, (lf, sp) in label_map.items():
        if stem not in img_map: continue
        for line in LINES[stem]:
            p = line.split()
            if len(p) < 7: continue
            xy = np.array([float(v) for v in p[1:]][: (len(p) - 1) // 2 * 2]).reshape(-1, 2)
            inst.append({"stem": stem, "split": sp, "path": img_map[stem], "cls": names[int(p[0])], "area": poly_area(xy.ravel()),
                         "bbox": (float(xy[:, 0].min()), float(xy[:, 1].min()), float(xy[:, 0].max()), float(xy[:, 1].max())),
                         "poly": np.round(xy, 4).astype(np.float32)})
    rows = []
    for r in nut.itertuples(index=False):
        r = r._asdict(); stem = r["FoodImage"].rsplit(".", 1)[0]
        if stem not in label_map: continue
        lf, sp = label_map[stem]; areas = np.zeros(len(names))
        for line in LINES[stem]:
            p = line.split()
            if len(p) >= 7: areas[int(p[0])] += poly_area([float(v) for v in p[1:]])
        if areas.sum() == 0: continue
        rows.append({"image": r["FoodImage"], "split": sp, "areas": areas,
                     "items": frozenset(names[i] for i in np.nonzero(areas)[0]), **{c: float(r[c]) for c in NUTR}})
    return root, list(names), pd.DataFrame(rows), pd.DataFrame(inst)

def synth_foodbd(n=900, seed=0):
    rng = np.random.RandomState(seed)
    spec = {"rice": (205, .88, .08, .04, .6, 73), "daal": (115, .55, .27, .18, 4, 30), "khichuri": (170, .62, .14, .24, 3, 55),
            "egg-boiled": (78, .03, .35, .62, 0, 0), "egg": (92, .03, .30, .67, 0, 0), "fish": (140, .02, .62, .36, 0, 0),
            "chicken": (175, .05, .55, .40, .5, 0), "beef": (225, .05, .45, .50, .5, 0), "ruti": (120, .75, .14, .11, 3, 62),
            "sweets": (150, .70, .08, .22, 0, 70), "potato-mashed": (120, .62, .06, .32, 2, 78), "chola": (150, .58, .22, .20, 6, 33),
            "vegetables": (60, .55, .15, .30, 3, 35), "begun-vaji": (95, .35, .06, .59, 2, 20), "polao": (235, .70, .07, .23, .6, 68),
            "salad": (20, .75, .15, .10, 1.5, 15)}
    names = list(spec); pop = np.array([6, 4, 2, 2, 1.5, 2, 2, 1.5, 1.5, 1, 1, 1, 2, 1, 1, 1.5]); pop /= pop.sum()
    rows = []
    for i in range(n):
        k = rng.choice([1, 2, 3], p=[.3, .45, .25]); idx = rng.choice(len(names), size=k, replace=False, p=pop)
        areas = np.zeros(len(names)); tot = dict.fromkeys(NUTR, 0.0)
        for j in idx:
            a = rng.uniform(.06, .32); areas[j] = a; kc, cp, pp, fp, fib, gi = spec[names[j]]
            e = kc * (a / .2) * rng.lognormal(0, .18)
            tot["Calorie"] += e; tot["CHO"] += e * cp / 4; tot["Protein"] += e * pp / 4; tot["Fat"] += e * fp / 9
            tot["Fiber"] += fib * a / .2; tot["GL"] += (e * cp / 4) * gi / 100
        rows.append({"image": f"synth_{i}.jpg", "split": rng.choice(["train", "valid", "test"], p=[.7, .15, .15]),
                     "areas": areas, "items": frozenset(names[j] for j in idx), **tot})
    # tiny synthetic plate photos (one coloured blob per item) so the crop gallery, recognition and plate-item detector code paths run in SMOKE mode
    from PIL import Image as _Im, ImageDraw as _Dr
    out_dir = os.path.join(WORK, "synth_plates"); os.makedirs(out_dir, exist_ok=True); inst = []
    col = {c: tuple(int(v) for v in np.random.RandomState(100 + i).randint(30, 226, 3)) for i, c in enumerate(names)}
    for i, r in enumerate(rows):
        p = os.path.join(out_dir, r["image"].replace(".jpg", ".png")); js = list(np.nonzero(r["areas"])[0])
        im = _Im.new("RGB", (96, 96), (235, 232, 225)); d = _Dr.Draw(im); boxes = {}
        for t, j in enumerate(js):
            w = 0.25 + 0.5 * r["areas"][j]; x0 = [0.05, 0.5, 0.25][t % 3]; y0 = [0.05, 0.3, 0.55][t % 3]
            b = (x0, y0, min(x0 + w, 0.98), min(y0 + w, 0.98)); boxes[j] = b
            cj = np.clip(np.array(col[names[j]]) + rng.randint(-18, 19, 3), 0, 255)
            d.ellipse([int(b[0] * 96), int(b[1] * 96), int(b[2] * 96), int(b[3] * 96)], fill=tuple(int(v) for v in cj))
        if not os.path.exists(p): im.save(p)
        inst += [{"stem": r["image"][:-4], "split": r["split"], "path": p, "cls": names[j], "area": float(r["areas"][j]), "bbox": boxes[j],
                  "poly": np.array([[boxes[j][0], boxes[j][1]], [boxes[j][2], boxes[j][1]], [boxes[j][2], boxes[j][3]], [boxes[j][0], boxes[j][3]]], np.float32)} for j in js]
    return out_dir, names, pd.DataFrame(rows), pd.DataFrame(inst)

with section("load FoodBD"):
    # other domains: FoodBD is not part of the evaluation, so its fast synthetic stand-in keeps the nutrition code running
    csv_path = None if (FORCE_SMOKE or DOMAIN != "bd") else find_first("**/MealNutrition1837_All.csv")
    REAL_FOODBD = csv_path is not None
    if REAL_FOODBD: FOODBD_ROOT, NAMES, plates, INST = cached("foodbd", lambda: load_foodbd(csv_path), (csv_path, "v7"))
    else: FOODBD_ROOT, NAMES, plates, INST = synth_foodbd()
NC = len(NAMES); SMOKE = FORCE_SMOKE or (not REAL_FOODBD if DOMAIN == "bd" else DOMAIN_SYNTH)
if DOMAIN != "bd": INST = INST.iloc[0:0]          # no FoodBD crops / plate detector outside the Bangladeshi domain
print(("REAL FoodBD" if REAL_FOODBD else "SYNTHETIC FoodBD stand-in (SMOKE mode)"), "|", len(plates), "plates with nutrition |",
      plates["split"].value_counts().to_dict(), "|", NC, "item classes |", INST["stem"].nunique(), "annotated images,", len(INST), "food instances")

## 2b. FoodBD metadata (`FoodBD_Meta_data.csv`): profiled, joined, used

The FoodBD paper lists this file as *"metadata on food instances, environmental conditions and preprocessing"* but does not document its columns, so the loader **profiles the CSV and detects column roles itself**, then prints what it found and used:
* **image key** → joined to all annotated images (3,523), not only the 1,837 plates with nutrition;
* **food class** + **Bangla name** columns → Bangla aliases for the 67 FoodBD classes (better linking of Bangla questions);
* **condition** columns (few distinct values, e.g. lighting / background / plate / angle) → per-condition error analysis and **condition-aware (Mondrian) conformal intervals** for photo-based nutrition (section 9);
* **numeric per-instance** columns whose name looks like weight / portion / calories → typical value per dish, added to the KB as a citable fact.

If the auto-detection picks the wrong columns, override with `META_ROLES`, e.g. `{"key": "Image_Name", "cls": "Food_Item", "bn": "Bangla_Name", "cond": ["Lighting"], "num": ["Weight"]}`.

In [ ]:
META_ROLES = {}                                            # optional manual override of the detected roles
BN_RE = re.compile(r"[ঀ-৿]")
def img_key(x):
    return re.sub(r"\.(jpe?g|png|webp|txt)$", "", os.path.basename(str(x).strip().lower()))
def _norm_cls(x): return re.sub(r"[\s_\-]+", " ", str(x).strip().lower())
def _mode(s):
    s = s.dropna(); return s.mode().iloc[0] if len(s) else np.nan

def read_csv_any(p):
    for enc in ("utf-8", "utf-8-sig", "cp1252", "latin-1"):
        try: return pd.read_csv(p, encoding=enc)
        except Exception: pass
    return pd.read_excel(p)

def synth_meta():
    """SMOKE stand-in with plausible columns, only to exercise the code path."""
    rng = np.random.RandomState(5)
    bn = {"rice": "ভাত", "daal": "ডাল", "khichuri": "খিচুড়ি", "egg-boiled": "সেদ্ধ ডিম", "egg": "ডিম ভাজা", "fish": "মাছ", "chicken": "মুরগি",
          "beef": "গরুর মাংস", "ruti": "রুটি", "sweets": "মিষ্টি", "potato-mashed": "আলু ভর্তা", "chola": "ছোলা", "vegetables": "সবজি",
          "begun-vaji": "বেগুন ভাজি", "polao": "পোলাও", "salad": "সালাদ"}
    light = {s: rng.choice(["Natural", "Indoor", "Low light"], p=[.5, .35, .15]) for s in INST["stem"].unique()}
    bg = {s: rng.choice(["Table", "Cloth", "Tray"]) for s in INST["stem"].unique()}
    # same layout as the real FoodBD_Meta_data.csv seen in the v6 Kaggle run: filename | instances ("fish,rice,shak") | types (split)
    rows = []
    for s, g in INST.groupby("stem"):
        its = sorted(set(g["cls"]))
        if rng.rand() < 0.05: its = its + ["shak"]                              # a little label disagreement, as in real data
        rows.append({"filename": s + ".jpg", "instances": ",".join(its), "types": g["split"].iloc[0]})
    return pd.DataFrame(rows)

def detect_meta_roles(df):
    roles = {"key": None, "cls": None, "bn": None, "cond": [], "num": [], "multi": None}
    known = set(INST["stem"].map(img_key)) | set(plates["image"].map(img_key))
    sc = {c: df[c].dropna().astype(str).head(4000).map(img_key).isin(known).mean() for c in df.columns if df[c].notna().any()}
    if sc and max(sc.values()) >= 0.2: roles["key"] = max(sc, key=sc.get)
    names_n = {_norm_cls(n) for n in NAMES}
    sc = {c: df[c].dropna().astype(str).head(4000).map(_norm_cls).isin(names_n).mean() for c in df.columns if c != roles["key"] and df[c].notna().any()}
    if sc and max(sc.values()) >= 0.3: roles["cls"] = max(sc, key=sc.get)
    # v7: a comma-separated list of FoodBD classes per image (the real file has 'instances': "fish,rice,shak,vaji")
    def _multi_share(c):
        v = df[c].dropna().astype(str).head(2000); v = v[v.str.contains(",")]
        if len(v) < 0.2 * min(len(df), 2000): return 0.0
        toks = [_norm_cls(t) for s in v for t in s.split(",") if t.strip()]
        return sum(t in names_n for t in toks) / max(len(toks), 1)
    sc = {c: _multi_share(c) for c in df.columns if c not in (roles["key"], roles["cls"]) and not pd.api.types.is_numeric_dtype(df[c])}
    if sc and max(sc.values()) >= 0.5: roles["multi"] = max(sc, key=sc.get)
    SPLITS_ = {"train", "valid", "val", "test", "validation"}
    for c in df.columns:
        if c in (roles["key"], roles["cls"], roles["multi"]) or not df[c].notna().any(): continue
        if set(df[c].dropna().astype(str).str.lower().unique()) <= SPLITS_: continue      # the split column is not a photo condition
        v = df[c].dropna().astype(str).head(4000); nun = df[c].nunique(dropna=True)
        if v.map(lambda s: bool(BN_RE.search(s))).mean() > 0.5:
            roles["bn"] = roles["bn"] or c; continue
        if nun > 0.9 * len(df): continue                                     # ids / free text
        if pd.api.types.is_numeric_dtype(df[c]):
            (roles["num"] if nun > 12 else roles["cond"]).append(c) if nun >= 2 else None
        elif 2 <= nun <= 25: roles["cond"].append(c)
    return roles

META_PATH = None if SMOKE else (find_first("**/FoodBD_Meta_data*.csv") or find_first("**/*FoodBD*[Mm]eta*.csv") or find_first("**/*[Mm]eta_[Dd]ata*.csv"))
META_RAW, META_INFO, META_BN, META_CLASS_NUM, IMG_META = None, {}, {}, None, None
if DOMAIN == "bd" and (META_PATH or SMOKE):
    META_RAW = synth_meta() if SMOKE else read_csv_any(META_PATH)
    print(f"metadata: {META_PATH or 'synthetic (SMOKE)'} | {META_RAW.shape[0]} rows x {META_RAW.shape[1]} columns")
    prof = pd.DataFrame({"dtype": META_RAW.dtypes.astype(str), "non-null": META_RAW.notna().sum(), "distinct": META_RAW.nunique(),
                         "examples": [", ".join(map(str, META_RAW[c].dropna().unique()[:4]))[:70] for c in META_RAW.columns]})
    print(prof.to_string())
    roles = {**detect_meta_roles(META_RAW), **META_ROLES}; META_INFO["roles"] = roles
    print("\ndetected roles:", roles)
    if roles["key"]:
        META_RAW["_key"] = META_RAW[roles["key"]].map(img_key)
        per_img = META_RAW.groupby("_key").size().mean(); META_INFO["rows_per_image"] = round(float(per_img), 2)
        if roles["cond"]:
            IMG_META = META_RAW.groupby("_key")[roles["cond"]].agg(_mode)
            pk = plates["image"].map(img_key)
            for c in roles["cond"]: plates["meta_" + c] = pk.map(IMG_META[c])
            META_INFO["plates_matched"] = float(pk.isin(IMG_META.index).mean())
        print(f"granularity: {per_img:.2f} rows per image ({'instance' if per_img > 1.2 else 'image'} level) | "
              f"plates matched: {META_INFO.get('plates_matched', 0):.1%} | images: {META_RAW['_key'].nunique()}")
    if roles["cls"]:
        by_norm = {_norm_cls(n): n for n in NAMES}; META_RAW["_cls"] = META_RAW[roles["cls"]].map(_norm_cls).map(by_norm)
        if roles["bn"]:
            META_BN = {k: v for k, v in META_RAW.dropna(subset=["_cls"]).groupby("_cls")[roles["bn"]].agg(_mode).items() if isinstance(v, str) and v.strip()}
        fact_num = [c for c in roles["num"] if re.search(r"weight|gram|\bgm?\b|portion|serv|volume|kcal|calor|_g$", str(c), re.I)]
        if fact_num: META_CLASS_NUM = META_RAW.dropna(subset=["_cls"]).groupby("_cls")[fact_num].median()
        print(f"Bangla names for {len(META_BN)} / {NC} FoodBD classes" + (f" | per-dish medians of {fact_num}" if fact_num else ""))
        if META_BN: print("  e.g.", dict(list(META_BN.items())[:6]))
    if roles["multi"] and roles["key"]:
        # data-quality check: per-image item list in the metadata vs the classes of the polygon annotations
        by_norm = {_norm_cls(n): n for n in NAMES}
        meta_items = META_RAW.groupby("_key")[roles["multi"]].first().map(lambda s: frozenset(by_norm[_norm_cls(t)] for t in str(s).split(",") if _norm_cls(t) in by_norm))
        poly_items = INST.groupby(INST["stem"].map(img_key))["cls"].agg(frozenset)
        both = [k for k in meta_items.index if k in poly_items.index]
        jac = [len(meta_items[k] & poly_items[k]) / max(len(meta_items[k] | poly_items[k]), 1) for k in both]
        unk = collections.Counter(_norm_cls(t) for s in META_RAW[roles["multi"]].dropna() for t in str(s).split(",") if t.strip() and _norm_cls(t) not in by_norm)
        META_INFO.update({"multi_label_column": roles["multi"], "images_compared": len(both), "mean_jaccard_meta_vs_polygons": round(float(np.mean(jac)), 3) if jac else None,
                          "exact_agreement": round(float(np.mean([j == 1 for j in jac])), 3) if jac else None,
                          "images_without_polygons": int(len(set(meta_items.index) - set(poly_items.index))), "unknown_item_names": dict(unk.most_common(8))})
        print(f"item lists in '{roles['multi']}' vs polygon classes: {len(both)} images, mean Jaccard {META_INFO['mean_jaccard_meta_vs_polygons']}, "
              f"identical sets {META_INFO['exact_agreement']:.1%} | images without polygons {META_INFO['images_without_polygons']} | names not in data.yaml: {META_INFO['unknown_item_names']}")
    if roles["cond"] and IMG_META is not None:
        for c in roles["cond"][:6]: print(f"  condition '{c}':", META_RAW.drop_duplicates("_key")[c].value_counts().head(6).to_dict())
else:
    print("FoodBD_Meta_data.csv not found (attach the full FoodBD folder) - metadata features are skipped.")
META_COND = [c for c in plates.columns if c.startswith("meta_")]

## 3. Nutrition estimator with split-conformal intervals

Model per nutrient chosen on validation; the 80 % interval half-width is the ⌈(n+1)(1−α)⌉-th smallest validation residual (finite-sample coverage ≥ 1−α under exchangeability); coverage is then checked on test plates.

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

A = np.stack(plates["areas"].values); Y = plates[NUTR].values.astype(float); split = plates["split"].values
tr, va, te = split == "train", split == "valid", split == "test"
def _rich(a):
    a = np.atleast_2d(a); p = (a > 0).astype(float)
    return np.hstack([p, a, np.sqrt(a), a.sum(1, keepdims=True), p.sum(1, keepdims=True)])
mu = Y[tr].mean(0)
lin = LinearRegression(positive=True, fit_intercept=False).fit(A[tr], Y[tr])
ridge_p = make_pipeline(StandardScaler(), Ridge(alpha=30.0)).fit(_rich(A[tr])[:, :NC], Y[tr])
ridge = make_pipeline(StandardScaler(), Ridge(alpha=30.0)).fit(_rich(A[tr]), Y[tr])
hgb = [HistGradientBoostingRegressor(max_depth=3, learning_rate=.05, max_iter=150, min_samples_leaf=15, random_state=0)
       .fit(_rich(A[tr]), Y[tr][:, j]) for j in range(len(NUTR))]
MODELS = {"mean": lambda a: np.tile(mu, (len(np.atleast_2d(a)), 1)),
          "area-linear": lambda a: lin.predict(np.atleast_2d(a)),
          "ridge presence-only": lambda a: ridge_p.predict(_rich(a)[:, :NC]),
          "ridge presence+area": lambda a: ridge.predict(_rich(a)),
          "boosted presence+area": lambda a: np.column_stack([m.predict(_rich(a)) for m in hgb])}
mae = lambda y, p: np.abs(y - p).mean(0)
BEST = pd.DataFrame({k: mae(Y[va], f(A[va])) for k, f in MODELS.items()}, index=NUTR).idxmin(axis=1)
def predict_all(a):
    a = np.atleast_2d(a); out = np.zeros((len(a), len(NUTR)))
    for j, c in enumerate(NUTR): out[:, j] = MODELS[BEST[c]](a)[:, j]
    return out
def conformal_q(resid, alpha=0.2):
    n = len(resid); k = min(n, int(math.ceil((n + 1) * (1 - alpha))))
    return np.sort(resid, axis=0)[k - 1]
Q = conformal_q(np.abs(Y[va] - predict_all(A[va])), 0.2)
test_mae = pd.DataFrame({k: mae(Y[te], f(A[te])) for k, f in MODELS.items()}, index=NUTR)
test_mae["SELECTED"] = mae(Y[te], predict_all(A[te]))
cov = (np.abs(Y[te] - predict_all(A[te])) <= Q).mean(0)
print("Test MAE per nutrient (ground-truth polygons):"); print(test_mae.round(2).to_string())
print("\nSelected per nutrient (validation):", BEST.to_dict())
print("\nConformal 80% interval: half-width and empirical test coverage"); print(pd.DataFrame({"half_width": Q.round(1), "coverage": cov.round(3)}, index=NUTR).to_string())
NUTRITION_RESULTS = {"test_mae": test_mae.round(3).to_dict(), "coverage": dict(zip(NUTR, cov.round(3).tolist()))}

COND_ANALYSIS = {}                    # FoodBD metadata: does the error depend on the photo conditions?
if META_COND:
    kc = NUTR.index("Calorie"); err_te = np.abs(Y[te][:, kc] - predict_all(A[te])[:, kc])
    print("\nCalorie MAE on test plates by metadata condition (polygon model):")
    for c in META_COND:
        t = pd.DataFrame({"group": plates.loc[te, c].fillna("unknown").astype(str).values, "err": err_te}).groupby("group")["err"].agg(["mean", "size"]).round(1)
        COND_ANALYSIS[c] = t.to_dict(); print(f"  {c[5:]}: " + " | ".join(f"{g}: {r['mean']} kcal (n={int(r['size'])})" for g, r in t.iterrows()))

_PLIST = {}
def nutrition_for_items(items, pool=None):
    pool = plates if pool is None else pool
    if id(pool) not in _PLIST: _PLIST[id(pool)] = (pool, pool["items"].tolist())
    plist = _PLIST[id(pool)][1]
    def stats(mask):
        sub = pool[mask]; st = {}
        for c in NUTR:
            v = sub[c].values.astype(float); med = float(np.median(v)); lv = np.log1p(np.maximum(v, 0))
            ml = float(np.median(lv)); hw = max((np.quantile(lv, .9) - np.quantile(lv, .1)) / 2, 0.1)   # half-width in log space (skewed data)
            s = (LOOKUP_S or {}).get(c, 1.0)                                                # conformal scale, fitted on validation plates
            st[c] = (med, float(np.expm1(ml - s * hw)), float(np.expm1(ml + s * hw)), ml, hw)
        st["_raw_kcal"] = (float(np.quantile(sub["Calorie"], .1)), float(np.quantile(sub["Calorie"], .9)), float(sub["Calorie"].min()), float(sub["Calorie"].max()))
        return st, int(mask.sum())
    def finish(tier, st, n, conf):
        q10, q90 = st["_raw_kcal"][:2]; spread = (q90 - q10) / max(st["Calorie"][0], 1.0)
        if spread > 2.0: conf = "low"
        return {"tier": tier, "n": n, "stats": st, "conf": conf, "spread": round(float(spread), 1)}
    S = frozenset(items)
    m = np.array([s == S for s in plist])
    if m.sum() >= 1: st, n = stats(m); return finish("exact", st, n, "high" if n >= 5 else "low")
    m = np.array([S <= s for s in plist])
    if m.sum() >= 3: st, n = stats(m); return finish("superset", st, n, "medium")
    # v7: no plate with this combination (common for photo-detected item sets: v6 then said "no nutrition data for rice") ->
    # the presence-only ridge model trained on all FoodBD train plates, with its own split-conformal 80 % interval (validation plates).
    if not S or not S <= set(NAMES): return None
    a = np.zeros((1, NC)); a[0, [NAMES.index(it) for it in S]] = 1.0
    p = ridge_p.predict(_rich(a)[:, :NC])[0]
    return {"tier": "model", "n": 0, "stats": {c: (float(max(p[j], 0)), float(max(p[j] - Q_PRES[j], 0)), float(p[j] + Q_PRES[j])) for j, c in enumerate(NUTR)},
            "conf": "low", "spread": 0}
Q_PRES = conformal_q(np.abs(Y[va] - ridge_p.predict(_rich(A[va])[:, :NC])), 0.2)

# Lookup ranges: v5 printed min-max of the matching plates, which covered only ~65 % of real test plates.
# Now: median +- s * half-width, with s chosen by split-conformal calibration on validation plates (lookups from train plates).
LOOKUP_S = None; _sc = {c: [] for c in NUTR}; _ptr = plates[tr]
for _, p in plates[va].iterrows():
    r = nutrition_for_items(p["items"], _ptr)
    if r is None or r["tier"] == "model": continue
    for c in NUTR:
        ml, hw = r["stats"][c][3], r["stats"][c][4]; _sc[c].append(abs(np.log1p(max(p[c], 0)) - ml) / hw)
LOOKUP_S = {c: float(conformal_q(np.array(v), 0.2)) for c, v in _sc.items() if len(v) >= 20}
print("lookup-range conformal scale (80 %):", {c: round(v, 2) for c, v in LOOKUP_S.items()})

## 4. Knowledge base

1. `tanziro/bangladeshi-food` metadata (descriptions, Bangla names, ingredients, region) — fetched with **ranged parquet reads** (text columns only, a few MB instead of ~6 GB) and cached.
2. A **seed KB** of 45 common dishes (short, qualitative, no numbers) with Bangla names and Banglish aliases; it covers every DeshiFoodBD class and gives SMOKE mode real text. *Proofread it — it is ground truth for descriptions.*
3. Optional: any extra CSV of dishes (`EXTRA_KB_CSV`, columns `name, name_bn, description`), e.g. exported from FoodNetBD (Mendeley 23f7yxnvy7, bilingual nutrition data).

`FOODBD_MAP` links a dish to a FoodBD plate item and says whether numbers are `exact` or a `proxy`.

In [ ]:
SEED_KB = [
 ("biriyani", "Biriyani", "বিরিয়ানি", ["biryani", "briyani", "kacchi", "kacchi biriyani", "কাচ্চি"], "Biriyani is a layered rice dish cooked with meat such as mutton, beef or chicken, aromatic spices, ghee and often potatoes. Dhaka's kacchi biriyani cooks raw marinated mutton together with the rice.", "rice dish", "polao", "proxy"),
 ("tehari", "Tehari", "তেহারি", ["tehri"], "Tehari is a Dhaka-style spiced rice dish cooked with small pieces of beef and mustard oil. It is a simpler, cheaper relative of biriyani.", "rice dish", "polao", "proxy"),
 ("morog polao", "Morog polao", "মোরগ পোলাও", ["murog polao", "chicken polao"], "Morog polao is fragrant polao rice cooked with chicken, ghee and whole spices. It is a common wedding dish.", "rice dish", "chicken", "proxy"),
 ("polao", "Polao", "পোলাও", ["pulao", "pilaf"], "Polao is aromatic short-grain rice such as chinigura or kalijira cooked in ghee with whole spices. It is served at festive meals.", "rice dish", "polao", "exact"),
 ("khichuri", "Khichuri", "খিচুড়ি", ["kichuri", "khichdi"], "Khichuri is rice and lentils cooked together with turmeric and spices. It is popular on rainy days and is often served with egg, beef or fried eggplant.", "rice dish", "khichuri", "exact"),
 ("rice", "Bhat (plain rice)", "ভাত", ["bhat", "plain rice", "steamed rice"], "Bhat is plain boiled rice. It is the staple food of Bangladesh and is eaten with daal, fish, vegetables or meat curry.", "staple", "rice", "exact"),
 ("panta bhat", "Panta bhat", "পান্তা ভাত", ["panta"], "Panta bhat is cooked rice soaked overnight in water. It is traditionally eaten with green chilli, onion and fried hilsa on Pohela Boishakh.", "rice dish", None, None),
 ("haleem", "Haleem", "হালিম", ["halim"], "Haleem is a thick, slow-cooked stew of wheat, lentils and meat. It is especially popular during Ramadan.", "stew", None, None),
 ("nehari", "Nehari", "নেহারি", ["nihari"], "Nehari is a slow-cooked spicy stew of beef or goat shank and trotters. It is usually eaten with naan or ruti for breakfast.", "stew", "beef", "proxy"),
 ("kala bhuna", "Kala bhuna", "কালা ভুনা", ["kalabhuna", "kalo bhuna"], "Kala bhuna is a Chittagonian beef dish slow-fried with spices and mustard oil until it turns dark. It is served with rice or porota.", "meat", "beef", "proxy"),
 ("beef curry", "Gorur mangsho (beef curry)", "গরুর মাংস", ["gorur mangsho", "beef bhuna"], "Gorur mangsho is a spicy Bangladeshi beef curry cooked with onion, garlic, ginger and garam masala. It is a staple at Eid feasts.", "meat", "beef", "exact"),
 ("chicken curry", "Murgir torkari (chicken curry)", "মুরগির তরকারি", ["murgir mangsho", "chicken bhuna", "murgi"], "Murgir torkari is a chicken curry cooked with onion, garlic, ginger and spices. It is usually eaten with rice.", "meat", "chicken", "exact"),
 ("kabab", "Kabab", "কাবাব", ["kebab", "sheekh kabab", "shami kabab"], "Kabab refers to grilled or fried spiced meat, such as sheekh kabab on skewers or shami kabab patties.", "meat", None, None),
 ("hilsha fish", "Ilish (hilsa)", "ইলিশ মাছ", ["ilish", "hilsa", "hilsha", "ilish mach", "ইলিশ"], "Ilish, the hilsa fish, is the national fish of Bangladesh. It is oily and bony and is cooked as shorshe ilish, ilish bhaja or ilish paturi.", "fish", "fish", "proxy"),
 ("shorshe ilish", "Shorshe ilish", "সর্ষে ইলিশ", ["sorshe ilish", "mustard hilsa"], "Shorshe ilish is hilsa fish cooked in a gravy of mustard paste and mustard oil with green chillies.", "fish", "fish", "proxy"),
 ("rui fish curry", "Rui macher jhol", "রুই মাছের ঝোল", ["rui mach", "rohu curry", "macher jhol"], "Rui macher jhol is a light curry of rohu fish, often with potatoes, cooked with turmeric and cumin.", "fish", "fish", "proxy"),
 ("egg curry", "Dimer torkari (egg curry)", "ডিমের তরকারি", ["dimer torkari", "dim bhuna"], "Dimer torkari is boiled eggs cooked in a spiced onion and tomato gravy, sometimes with potatoes.", "egg", "egg-boiled", "proxy"),
 ("egg omlete", "Dim bhaji (omelette)", "ডিম ভাজি", ["omelette", "omelet", "omlet", "dim bhaji", "egg omelette"], "Dim bhaji is a Bangladeshi omelette made with beaten eggs, onion and green chilli fried in oil.", "egg", "egg", "proxy"),
 ("daal", "Daal (lentil soup)", "ডাল", ["dal", "dahl", "lentil soup", "masoor dal"], "Daal is a lentil soup, most often made with red lentils and tempered with garlic, onion or cumin. It is eaten daily with rice.", "lentil", "daal", "exact"),
 ("chickpeas", "Chola bhuna", "ছোলা ভুনা", ["chola", "chana", "chola bhuna", "chickpea curry"], "Chola bhuna is spiced cooked chickpeas. It is a common iftar item during Ramadan.", "lentil", "chola", "exact"),
 ("mashed potato", "Aloo bhorta", "আলু ভর্তা", ["aloo bhorta", "alu vorta", "alu bhorta", "mashed potatoes"], "Aloo bhorta is mashed potato mixed with mustard oil, fried onion, green chilli and coriander.", "bhorta", "potato-mashed", "exact"),
 ("begun bhorta", "Begun bhorta", "বেগুন ভর্তা", ["begun vorta", "eggplant mash", "baingan bharta"], "Begun bhorta is roasted, mashed eggplant mixed with mustard oil, onion and chilli.", "bhorta", None, None),
 ("begun bhaja", "Begun bhaja", "বেগুন ভাজা", ["begun vaji", "fried eggplant"], "Begun bhaja is slices of eggplant rubbed with turmeric and salt and shallow-fried.", "vegetable", "begun-vaji", "exact"),
 ("mixed vegetables", "Shobji (mixed vegetables)", "সবজি", ["shobji", "sobji", "vegetable curry", "labra"], "Shobji is a dry or lightly spiced curry of seasonal vegetables such as potato, cauliflower, beans and pumpkin.", "vegetable", "vegetables", "proxy"),
 ("beguni", "Beguni", "বেগুনি", ["baguni"], "Beguni is eggplant slices dipped in gram-flour batter and deep fried. It is a popular iftar snack.", "snack", "begun-vaji", "proxy"),
 ("piyaju", "Piyaju", "পিঁয়াজু", ["peyaju", "piaju", "onion fritter"], "Piyaju is a deep-fried fritter made of lentil paste and onion, eaten at iftar and as a street snack.", "snack", None, None),
 ("singara", "Singara", "সিঙ্গারা", ["shingara", "singhara"], "Singara is a fried pastry stuffed with spiced potato, sometimes with liver or peanuts.", "snack", None, None),
 ("somucha", "Somucha", "সমুচা", ["samosa", "somosa"], "Somucha is a triangular fried pastry filled with spiced minced meat or vegetables.", "snack", None, None),
 ("fuchka", "Fuchka", "ফুচকা", ["phuchka", "puchka", "fuska", "phuska", "panipuri", "golgappa"], "Fuchka is a street snack of crisp hollow shells filled with a spiced chickpea and potato mix and tangy tamarind water.", "street food", None, None),
 ("chotpoti", "Chotpoti", "চটপটি", ["chatpati"], "Chotpoti is a street food of cooked yellow peas with potato, egg, onion, chilli and tamarind sauce.", "street food", None, None),
 ("jhalmuri", "Jhalmuri", "ঝালমুড়ি", ["jhal muri", "masala muri"], "Jhalmuri is puffed rice tossed with mustard oil, onion, chilli, chanachur and spices.", "street food", None, None),
 ("porota", "Porota", "পরোটা", ["paratha", "parata", "porata"], "Porota is a layered, pan-fried flatbread made from wheat flour with oil or ghee.", "bread", "ruti", "proxy"),
 ("ruti", "Ruti", "রুটি", ["roti", "chapati", "atar ruti"], "Ruti is a thin unleavened whole-wheat flatbread cooked on a dry griddle.", "bread", "ruti", "exact"),
 ("bakorkhani", "Bakorkhani", "বাকরখানি", ["bakarkhani", "bakharkhani"], "Bakorkhani is a thick, crisp, layered flatbread associated with Old Dhaka, usually eaten with tea.", "bread", None, None),
 ("roshgolla", "Roshgolla", "রসগোল্লা", ["rasgulla", "rasogolla", "roshogolla", "rosogolla"], "Roshgolla is a spongy ball of chhana (fresh curd cheese) cooked in light sugar syrup.", "sweet", "sweets", "proxy"),
 ("roshmalai", "Roshmalai", "রসমালাই", ["rasmalai", "rosmalai"], "Roshmalai is small chhana balls soaked in thickened, sweetened milk. Comilla is famous for it.", "sweet", "sweets", "proxy"),
 ("chomchom", "Chomchom", "চমচম", ["chamcham", "cham cham"], "Chomchom is an oval chhana sweet in syrup. The Porabari chomchom of Tangail is well known.", "sweet", "sweets", "proxy"),
 ("kacha golla", "Kacha golla", "কাঁচাগোল্লা", ["kachagolla", "kacha gulla", "কাচা গোল্লা"], "Kacha golla is a soft sweet made from fresh chhana and sugar, famous from Natore.", "sweet", "sweets", "proxy"),
 ("jilapi", "Jilapi", "জিলাপি", ["jalebi", "jilebi"], "Jilapi is deep-fried spirals of fermented batter soaked in sugar syrup.", "sweet", "sweets", "proxy"),
 ("yogurt", "Doi / Mishti doi", "দই", ["doi", "dahi", "curd", "mishti doi", "মিষ্টি দই"], "Mishti doi is sweetened yogurt set with caramelised milk in clay pots. Bogura is famous for it.", "sweet", None, None),
 ("payesh", "Payesh", "পায়েস", ["payes", "kheer", "rice pudding"], "Payesh is rice pudding made with aromatic rice, milk and sugar or date-palm jaggery.", "sweet", None, None),
 ("shemai", "Shemai", "সেমাই", ["semai", "vermicelli"], "Shemai is a sweet dish of fine vermicelli cooked in milk and sugar, traditional at Eid.", "sweet", None, None),
 ("bhapa pitha", "Bhapa pitha", "ভাপা পিঠা", ["vapa pitha", "pitha"], "Bhapa pitha is a steamed rice-flour cake filled with date-palm jaggery and grated coconut, a winter favourite.", "sweet", None, None),
 ("borhani", "Borhani", "বোরহানি", ["burhani"], "Borhani is a spiced yogurt drink with mint and black salt, served with biriyani at weddings.", "drink", None, None),
 ("salad", "Salad", "সালাদ", ["shosha salad"], "In Bangladesh a side salad is usually sliced cucumber, onion, tomato and green chilli with lemon.", "side", "salad", "exact"),
]
DESHI_BN = {"বাকড়খানি": "bakorkhani", "ডিম ভাজি": "egg omlete", "রসগোল্লা": "roshgolla", "ছোলা ভুনা": "chickpeas", "নেহারি": "nehari",
            "খিচুড়ি": "khichuri", "কাচা গোল্লা": "kacha golla", "বেগুনি": "beguni", "আলু ভর্তা": "mashed potato", "রসমালাই": "roshmalai",
            "কালা ভুনা": "kala bhuna", "ইলিশ মাছ": "hilsha fish", "হালিম": "haleem", "দই": "yogurt", "পরোটা": "porota",
            "ফুচকা": "fuchka", "মোরগ পোলাও": "morog polao", "কাবাব": "kabab", "বিরিয়ানি": "biriyani"}
EXTRA_KB_CSV = find_first("**/extra_kb*.csv")      # optional: columns name, name_bn, description

def norm(s):
    s = unicodedata.normalize("NFC", str(s)).lower().strip()
    s = re.sub(r"[_\-]+", " ", s)
    s = re.sub(r"[^\w\sঀ-৿]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

DOCS, ALIASES, FOODBD_MAP = {}, collections.defaultdict(set), {}
DOC_FACTS = collections.defaultdict(list)   # dataset facts (FoodBD co-occurrence, FoodNetBD per-100 g / diet tags): evidence only for questions that ask
def add_doc(canon, name, bn, text, source, category=""):
    DOCS[canon] = {"canon": canon, "name": name, "bn": bn or "", "text": text.strip(), "source": source, "category": category}
for canon, name, bn, al, desc, cat, item, exact in SEED_KB:
    add_doc(canon, name, bn, desc, "seed", cat)
    for a in [canon, name, re.sub(r"\s*\(.*?\)", "", name), bn] + al: ALIASES[canon].add(norm(a))
    if item: FOODBD_MAP[canon] = (item, exact)
for bn, en in DESHI_BN.items(): ALIASES[en].add(norm(bn))
for n in NAMES:                       # FoodBD plate items are dishes too ("rice", "egg-boiled" ...)
    c = norm(n)
    if c not in DOCS and not any(c in v for v in ALIASES.values()):
        FOODBD_MAP[c] = (n, "exact"); ALIASES[c].add(c)
    elif c in DOCS and c not in FOODBD_MAP: FOODBD_MAP[c] = (n, "exact")
FOODBD_MAP = {k: v for k, v in FOODBD_MAP.items() if v[0] in NAMES}

# FoodNetBD diet-tag conditions: (display name, regex for the tag key / column, regex for the user's question in English / Banglish / Bangla)
CONDITIONS = {
    "t2d": ("type 2 diabetes", r"t2d|diabet", r"diabet|\bsugar\b|blood sugar|ডায়াবেটিস|ডায়াবেটিক|সুগার"),
    "pcos": ("PCOS", r"pcos|pcod", r"\bpco[sd]\b|পিসিওএস"),
    "hypertension": ("hypertension", r"hypertens|\bhtn\b|high.?blood.?pressure", r"hypertension|high (blood )?pressure|\bhigh bp\b|উচ্চ রক্তচাপ|হাই প্রেশার"),
    "hypotension": ("hypotension", r"hypotens|low.?blood.?pressure", r"hypotension|low (blood )?pressure|\blow bp\b|নিম্ন রক্তচাপ|লো প্রেশার"),
    "ckd": ("chronic kidney disease", r"\bckd\b|kidney|renal", r"kidney|\bckd\b|renal|কিডনি"),
    "anemia": ("anemia", r"an(a)?emi", r"an(a)?emi|রক্তস্বল্পতা|রক্তশূন্যতা"),
    "gout": ("gout", r"gout|uric", r"gout|uric acid|গেঁটে বাত|ইউরিক"),
    "ibs": ("irritable bowel syndrome", r"\bibs\b|irritable", r"\bibs\b|irritable bowel"),
    "asthma": ("asthma", r"asthma", r"asthma|হাঁপানি|অ্যাজমা|এজমা"),
    "hypothyroidism": ("hypothyroidism", r"hypothyr", r"hypothyroid|\bthyroid\b|থাইরয়েড"),
    "hyperthyroidism": ("hyperthyroidism", r"hyperthyr", r"hyperthyroid|\bthyroid\b|থাইরয়েড"),
    "celiac": ("celiac disease", r"c(o)?eliac|gluten", r"c(o)?eliac|gluten|গ্লুটেন|সিলিয়াক"),
}
COND_NAME = {k: v[0] for k, v in CONDITIONS.items()}
def cond_of(key):
    key = str(key).lower()
    if "hypo" in key and "thyr" in key: return "hypothyroidism"
    if "hyper" in key and "thyr" in key: return "hyperthyroidism"
    return next((k for k, (_, kp, _) in CONDITIONS.items() if re.search(kp, key)), None)
def conditions_in(q):
    """Health conditions mentioned in a question ('is biriyani ok for diabetes?', 'ডায়াবেটিস রোগী কি খিচুড়ি খেতে পারে?')."""
    return [k for k, (_, _, qp) in CONDITIONS.items() if re.search(qp, q, re.I)]

META_COLS = ["dish_name", "dish_name_bn", "category", "region", "description", "ingredients", "cooking_method"]
def _load_tz():
    """Ranged reads of only the text columns (the parquet files also hold ~6 GB of images)."""
    rows = []
    try:
        from huggingface_hub import HfFileSystem
        import pyarrow.parquet as pq
        files = sorted(HfFileSystem().glob("datasets/tanziro/bangladeshi-food/data/train-*.parquet"))
        def one(f):                                    # v5 read the 13 files one after another (~300 s); now in parallel
            with HfFileSystem().open(f, "rb", block_size=1 << 20) as fh:
                pf = pq.ParquetFile(fh); cols = [c for c in META_COLS if c in pf.schema_arrow.names]
                return pf.read(columns=cols).to_pylist()
        from concurrent.futures import ThreadPoolExecutor
        with ThreadPoolExecutor(min(8, len(files))) as ex:
            for part in ex.map(one, files): rows += part
        print(f"tanziro: {len(rows)} rows from {len(files)} files (text columns only, parallel ranged reads)")
    except Exception as e:
        print("ranged read failed (" + repr(e)[:120] + ") -> streaming fallback")
        from datasets import load_dataset
        ds = load_dataset("tanziro/bangladeshi-food", split="train", streaming=True)
        ds = ds.select_columns([c for c in META_COLS if c in (ds.column_names or META_COLS)])
        rows = list(ds)
    seen, uniq = set(), []
    for r in rows:                       # dataset is one row per image; keep one row per dish
        if r.get("dish_name") and r["dish_name"] not in seen: seen.add(r["dish_name"]); uniq.append(r)
    return uniq

TZ_N = 0
def _clean(v):
    s = "" if v is None or (isinstance(v, float) and v != v) else str(v).strip()
    return "" if s in ("-1", "nan", "None", "") else s
def _wiki(name):
    """Wikipedia REST summary (Internet on); cached on disk with the KB."""
    if SMOKE: return None
    try:
        import requests
        r = requests.get("https://en.wikipedia.org/api/rest_v1/page/summary/" + name.replace(" ", "_"),
                         headers={"User-Agent": "bd-food-selfheal-rag/0.8 (student project)"}, timeout=8)
        j = r.json() if r.status_code == 200 else {}
        return j.get("extract") if j.get("type") == "standard" else None
    except Exception: return None
def build_domain_kb():
    """Other-domain KB -> [(canon, name, aliases, text, category)].
    * CSV (Indian Food 101 layout or any table): name column + a description column if there is one, otherwise every other text column
      is written as 'Column: value.' (missing values '-1' / NaN are skipped); category = course / category / type column.
    * no CSV (Food-101): the image class names, described by their Wikipedia summary ('X is a dish.' when offline)."""
    rows = []
    if DOMAIN_KB_PATH:
        df = read_csv_any(DOMAIN_KB_PATH); cols = list(df.columns)
        ncol = next((c for c in cols if str(c).lower() in ("name", "dish", "dish_name", "food", "title")), cols[0])
        dcol = next((c for c in cols if re.search(r"desc|summary|about", str(c), re.I)), None)
        ccol = next((c for c in cols if str(c).lower() in ("course", "category", "type", "food_type")), None)
        for _, r in df.iterrows():
            name = _clean(r[ncol])
            if not name: continue
            if dcol and _clean(r[dcol]): text = _clean(r[dcol])
            else:
                parts = [f"{str(c).replace('_', ' ').capitalize()}: {_clean(r[c])}." for c in cols if c not in (ncol,) and _clean(r[c])]
                text = f"{name} is a dish of {DOMAIN_LABEL}. " + " ".join(parts)
            rows.append((norm(re.sub(r"\s*\(.*?\)", "", name)), name, [name], text, _clean(r[ccol]) if ccol else ""))
    else:
        classes = sorted(d for d in os.listdir(DOMAIN_IMG_ROOT) if os.path.isdir(os.path.join(DOMAIN_IMG_ROOT, d)))
        wk = cached("domain_wiki", lambda: {c: _wiki(c.replace("_", " ")) for c in classes}, (DOMAIN, tuple(classes), SMOKE))
        for c in classes:
            name = c.replace("_", " ").capitalize()
            rows.append((norm(c.replace("_", " ")), name, [c.replace("_", " ")], wk.get(c) or f"{name} is a dish.", ""))
    return rows

with section("knowledge base"):
    tz = [] if (SMOKE or DOMAIN != "bd") else (try_load("kb:tanziro/bangladeshi-food", lambda: cached("tanziro_meta", _load_tz, "v4")) or [])
    alias_lookup = {a: c for c, al in ALIASES.items() for a in al}; _merged = set()
    for row in tz:
        name = row.get("dish_name")
        if not name: continue
        base = norm(re.sub(r"\s*\(.*?\)", "", name))
        canon = alias_lookup.get(base) or alias_lookup.get(norm(row.get("dish_name_bn") or "")) or base
        if canon in _merged: continue
        _merged.add(canon)
        t = (row.get("description") or "") + " "
        if row.get("ingredients"): t += "Ingredients: " + ", ".join(row["ingredients"]) + ". "
        if row.get("cooking_method"): t += f"Cooking method: {row['cooking_method']}. "
        if row.get("region") and row["region"] != "Nationwide": t += f"Region: {row['region']}."
        old = DOCS.get(canon)
        add_doc(canon, old["name"] if old else name, row.get("dish_name_bn") or (old["bn"] if old else ""),
                (old["text"] + " " + t) if old else t, "seed+tanziro" if old else "tanziro", row.get("category") or "")
        ALIASES[canon] |= {norm(name), base} | ({norm(row["dish_name_bn"])} if row.get("dish_name_bn") else set())
        TZ_N += 1
    if EXTRA_KB_CSV:
        ek = pd.read_csv(EXTRA_KB_CSV)
        for r in ek.itertuples(index=False):
            c = norm(r.name)
            if c and c not in DOCS and isinstance(getattr(r, "description", None), str):
                add_doc(c, r.name, getattr(r, "name_bn", ""), r.description, "extra-csv"); ALIASES[c] |= {c, norm(getattr(r, "name_bn", "") or "")}
        print("extra KB rows from", EXTRA_KB_CSV)
    # ---- FoodBD metadata: Bangla names for the 67 FoodBD classes ----
    item2canon = collections.defaultdict(list)
    for c, (it, how) in FOODBD_MAP.items():
        if how == "exact": item2canon[it].append(c)
    for it, bn_name in META_BN.items():
        for c in item2canon.get(it, []) or [norm(it)]: ALIASES[c].add(norm(bn_name))
    # ---- FoodBD facts: every annotated image (3,523), co-occurrence + metadata medians -> citable KB sentences ----
    FOODBD_FACTS = {}
    if len(INST):
        img_items = INST.groupby("stem")["cls"].agg(lambda s: frozenset(s))
        disp = lambda it: DOCS[item2canon[it][0]]["name"] if item2canon.get(it) and item2canon[it][0] in DOCS else it.replace("-", " ")
        for it in NAMES:
            with_it = [s for s in img_items if it in s]
            if not with_it: continue
            co = collections.Counter(o for s in with_it for o in s if o != it).most_common(3)
            t = f"In the FoodBD meal-photo dataset, {disp(it)} appears in {len(with_it)} annotated meals"
            t += (", most often served with " + ", ".join(disp(o) for o, _ in co) + "." if co else ".")
            if META_CLASS_NUM is not None and it in META_CLASS_NUM.index:
                t += " " + " ".join(f"FoodBD metadata, median {k}: {v:.0f}." for k, v in META_CLASS_NUM.loc[it].dropna().items())
            FOODBD_FACTS[it] = t
            targets = item2canon.get(it) or [norm(it)]
            for c in targets:
                if c in DOCS: DOC_FACTS[c].append(t)          # v7: kept out of the description (shown only when asked)
                else:                                    # FoodBD-only class (e.g. "egg-boiled"): give it a document so it can be grounded
                    add_doc(c, it.replace("-", " ").capitalize(), META_BN.get(it, ""), t, "foodbd"); ALIASES[c] |= {c, norm(it)}
        print(f"FoodBD facts added for {len(FOODBD_FACTS)} classes (co-occurrence over {len(img_items)} annotated images)")
    # ---- FoodNetBD (Mendeley 23f7yxnvy7, optional): per-100 g nutrition + Bangla names, linked to KB dishes ----
    NUTR100, FNB_INFO, HEALTH, FNB_PAIRS = {}, {}, {}, []
    FNB_PATH = None if SMOKE else (find_first("**/FoodNetBD*.csv") or find_first("**/*[Ff]ood[Nn]et*BD*.csv"))
    if SMOKE:                                            # tiny synthetic FoodNetBD-like table, only to exercise the code path
        FNB_PATH = os.path.join(WORK, "synthetic_FoodNetBD.csv")
        pd.DataFrame([("Khichuri, cooked", "খিচুড়ি", 170, 5.1, 4.2, 28.0, 2.9, "T2D: restrict; Hypertension: safe; CKD: restrict; Gout: safe; Celiac: safe"),
                      ("Biriyani (mutton)", "বিরিয়ানি", 240, 9.8, 11.5, 25.3, 1.1, "Safe: Celiac | Restrict: Hypertension, Gout | Avoid: Type 2 Diabetes, CKD"),
                      ("Hilsa (Smoked/Dried Slice)", "শুঁটকি ইলিশ", 415, 58.0, 19.0, 0.0, 0.0, "Hypertension: avoid"),
                      ("Hilsa curry", "ইলিশ মাছ", 250, 19.0, 18.0, 3.0, 0.4, "Hypertension: restrict; Gout: restrict; T2D: safe"),
                      ("Daal (masoor), cooked", "ডাল", 115, 7.6, 2.9, 16.0, 4.1, '{"safe": ["T2D", "HTN", "Anemia"], "restrict": ["CKD", "Gout"]}'),
                      ("Ruti (atta)", "রুটি", 264, 8.5, 3.7, 49.0, 6.2, "T2D: restrict; Celiac: avoid; Hypertension: safe"),
                      ("Rice, parboiled, raw", "চাল", 350, 7.0, 0.5, 78.0, 1.0, "T2D: restrict")],
                     columns=["Food_Name_English", "Food_Name_Bengali", "Energy_kcal", "Protein_g", "Total_Fat_g", "Carbohydrate_g", "Dietary_Fibre_g",
                              "Disease_Tags"]).to_csv(FNB_PATH, index=False)
    if FNB_PATH:
        fnb = read_csv_any(FNB_PATH); cols = list(fnb.columns)
        bn_col = next((c for c in cols if fnb[c].dropna().astype(str).head(500).map(lambda s: bool(BN_RE.search(s))).mean() > 0.5), None)
        txt = [c for c in cols if c != bn_col and not pd.api.types.is_numeric_dtype(fnb[c]) and fnb[c].nunique() > 0.6 * len(fnb)]
        en_col = next((c for c in txt if re.search(r"name|food|item", str(c), re.I)), txt[0] if txt else None)
        def pick(pat, bad=r"satur|trans|mono|poly|kj\b"):
            for c in cols:
                if re.search(pat, str(c), re.I) and not re.search(bad, str(c), re.I) and pd.to_numeric(fnb[c], errors="coerce").notna().mean() > 0.5: return c
        NC100 = {"energy": (pick(r"kcal") or pick(r"energy|calor"), "kcal"), "protein": (pick(r"protein"), "g"), "fat": (pick(r"total.?fat|lipid") or pick(r"\bfat"), "g"),
                 "carbohydrate": (pick(r"carbo|\bcho\b"), "g"), "fibre": (pick(r"fib(re|er)"), "g")}
        NC100 = {k: v for k, v in NC100.items() if v[0] is not None}
        FNB_INFO = {"path": FNB_PATH, "rows": len(fnb), "name_col": en_col, "bangla_col": bn_col, "nutrients": {k: v[0] for k, v in NC100.items()}}
        # diet tags: a structured Disease_Tags field (any "condition: safe/restrict/avoid" text or JSON), or one column per condition
        VERD = r"(safe|restrict(?:ed|ion)?|limit(?:ed)?|moderate|caution|avoid)"
        _hasv = {c: fnb[c].astype(str).str.contains(VERD, case=False, regex=True).mean() for c in cols}
        tag_col = (next((c for c in cols if re.search(r"disease|tag|condition|health", str(c), re.I) and _hasv[c] > 0.3), None)
                   or next((c for c in cols if _hasv[c] > 0.5 and fnb[c].astype(str).str.len().mean() > 12), None))
        per_cond_cols = [c for c in cols if c != tag_col and fnb[c].dropna().astype(str).str.fullmatch(VERD, case=False).mean() > 0.8 and cond_of(c)]
        _norm_v = lambda v: "avoid" if v.lower().startswith("avoid") else ("safe" if v.lower().startswith("safe") else "restrict")
        def parse_tags(r):
            """Format-agnostic (FoodNetBD documents tiers safe / restrict / avoid but not the layout): every segment between ; | newline
            (then commas, if a segment holds several tiers) assigns its single tier to every condition named in it. Handles
            'T2D: avoid; HTN: safe', 'Safe: T2D, HTN | Avoid: CKD', JSON {"safe": [...]} and 'diabetes(restrict)'."""
            out = {}
            if tag_col and isinstance(r.get(tag_col), str):
                txt = r[tag_col]
                segs = [x for x in re.split(r"[;|\n]+|\}\s*,|\]\s*,", txt) if x.strip()]
                for seg in segs:
                    parts = [seg] if len(re.findall(VERD, seg, re.I)) <= 1 else re.split(r",", seg)
                    carry = None                                  # 'Safe: T2D, HTN' split by commas -> carry the tier forward
                    for p in parts:
                        vs = re.findall(VERD, p, re.I); carry = _norm_v(vs[0]) if vs else carry
                        if carry is None: continue
                        for k, (_, kp, _) in CONDITIONS.items():
                            if k in ("hypothyroidism", "hyperthyroidism"):
                                if re.search(("hypo" if k == "hypothyroidism" else "hyper") + r"\s*-?\s*thyr", p, re.I): out[k] = carry
                            elif re.search(kp, p, re.I) and not (k == "hypertension" and re.search(r"hypotens|low.?blood", p, re.I)): out[k] = carry
            for c in per_cond_cols:
                if isinstance(r.get(c), str): out[cond_of(c)] = _norm_v(r[c].strip())
            return out
        FNB_INFO["diet_tags"] = {"column": tag_col, "per_condition_columns": per_cond_cols,
                                 "examples": fnb[tag_col].dropna().astype(str).head(3).tolist() if tag_col else []}
        print("FoodNetBD columns:", cols[:60])
        print(f"diet-tag column: {tag_col!r} | per-condition columns: {per_cond_cols} | raw examples: {FNB_INFO['diet_tags']['examples']}")
        lookup = {a: c for c, al in ALIASES.items() for a in al}; RAW = re.compile(r"\b(raw|dry|dried|uncooked|flour|powder|seed|seeds)\b", re.I)
        QUAL = re.compile(r"smoked|dried|\bdry\b|canned|pickled|salted|powder|\braw\b|uncooked|flour|frozen|concentrate", re.I)
        best_row = {}                    # v7: several FoodNetBD rows can match one dish; pick the best (v6 took the first: dried hilsa, 415 kcal)
        for _, r in fnb.iterrows():
            en = str(r.get(en_col, "") or ""); bn_name = str(r.get(bn_col, "") or "") if bn_col else ""
            full = [norm(en), norm(re.sub(r"\(.*?\)", "", en)), norm(bn_name)]
            canon = next((lookup[k] for k in full if k and k in lookup), None); how = "exact"
            if canon is None and not RAW.search(en):    # "Khichuri, cooked" -> khichuri; never map raw ingredients to a cooked dish
                head = norm(en.split(",")[0]); canon = lookup.get(head); how = "head"
            if canon is None or canon not in DOCS: continue
            score = (how == "exact", not QUAL.search(en), fratio(norm(re.sub(r"\(.*?\)", "", en)), norm(DOCS[canon]["name"].split("(")[0])))
            if canon not in best_row or score > best_row[canon][0]: best_row[canon] = (score, r, en, bn_name, how)
        FNB_PAIRS = []                   # bilingual bridge: English name -> one KB dish, Bengali name -> another => the same food
        for _, r in fnb.iterrows():
            e_, b_ = lookup.get(norm(re.sub(r"\(.*?\)", "", str(r.get(en_col, "") or "")))), lookup.get(norm(str(r.get(bn_col, "") or ""))) if bn_col else None
            if e_ and b_ and e_ != b_ and e_ in DOCS and b_ in DOCS: FNB_PAIRS.append((e_, b_))
        for canon, (_, r, en, bn_name, how) in best_row.items():
            tags = parse_tags(r)
            if tags: HEALTH[canon] = {"name": en, "how": how, "tags": tags}
            vals = {k: float(pd.to_numeric(r[c], errors="coerce")) for k, (c, _) in NC100.items()}
            vals = {k: v for k, v in vals.items() if v == v}
            if vals: NUTR100[canon] = {"name": en, "how": how, **vals}
            if how == "exact" and bn_name: ALIASES[canon].add(norm(bn_name))
        for c, d in NUTR100.items():
            DOC_FACTS[c].append(f"FoodNetBD lists per 100 g of '{d['name']}': " + ", ".join(f"{k} {d[k]:.1f} {NC100[k][1]}" for k in NC100 if k in d) + ".")
        for c, d in HEALTH.items():
            DOC_FACTS[c].append(f"FoodNetBD diet tags for '{d['name']}': " + "; ".join(f"{COND_NAME[k]}: {v}" for k, v in d["tags"].items()) + ".")
        FNB_INFO["linked_dishes"] = len(NUTR100); FNB_INFO["dishes_with_diet_tags"] = len(HEALTH)
        if HEALTH: print("diet tags parsed for", len(HEALTH), "dishes, e.g.", next(iter(HEALTH.items())))
        print(f"FoodNetBD: {len(fnb)} rows from {FNB_PATH} | name={en_col!r} bangla={bn_col!r} nutrients={FNB_INFO['nutrients']} | linked to {len(NUTR100)} KB dishes")
    else:
        print("FoodNetBD.csv not attached (optional: Mendeley 23f7yxnvy7) - per-100 g nutrition answers disabled.")
    HELDOUT = []
    if DOMAIN != "bd":
        # ---- other domain: the Bangladeshi KB is replaced; every later section (linker, retrieval, evaluator, benchmark, judges) is unchanged ----
        for store in (DOCS, ALIASES, FOODBD_MAP, DOC_FACTS, NUTR100, HEALTH): store.clear()
        FNB_PAIRS = []
        drows = build_domain_kb()
        # held-out dishes: ~10 % of the dishes that have photos are REMOVED from the KB, so their questions and photos must be declined
        # (an in-domain "near-miss" test that the Bangladeshi benchmark cannot give)
        img_cls = {norm(d.replace("_", " ")) for d in os.listdir(DOMAIN_IMG_ROOT)} if DOMAIN_IMG_ROOT and os.path.isdir(DOMAIN_IMG_ROOT) else set()
        with_img = sorted(c for c, *_ in drows if c in img_cls)
        HELDOUT = sorted(with_img, key=lambda c: hashlib.md5(c.encode()).hexdigest())[:max(1, len(with_img) // 10)] if len(with_img) >= 5 else []
        for canon, name, al, text, cat in drows:
            if canon in HELDOUT or canon in DOCS: continue
            add_doc(canon, name, "", text, DOMAIN, cat); ALIASES[canon] |= {norm(a) for a in [canon, name] + al}
        print(f"DOMAIN {DOMAIN}: {len(DOCS)} dishes from {DOMAIN_KB_PATH or 'image class names + Wikipedia'} | {len(with_img)} have photos | "
              f"held out (must be declined): {HELDOUT}")
    for c in list(ALIASES):
        ALIASES[c].discard("")
        if c not in DOCS and c not in FOODBD_MAP: ALIASES.pop(c)
    ALIAS2CANON = {}
    for c, al in ALIASES.items():
        for a in al:
            if a and (a not in ALIAS2CANON or len(c) < len(ALIAS2CANON[a])): ALIAS2CANON[a] = c
print(f"KB: {len(DOCS)} dishes with text ({TZ_N} merged from tanziro) | {len(ALIAS2CANON)} aliases | {len(FOODBD_MAP)} dishes linked to FoodBD items")
print("Sources:", dict(collections.Counter(d["source"] for d in DOCS.values())))

## 5. Bangla / Banglish normalisation and the entity linker

* `bn2latin` — rule-based Bangla→Latin transliteration (consonants, vowel signs, hasant, inherent vowel).
* `phonetic_key` — collapses Banglish spelling variation (bh/v→b, sh/s, kh→k, o/u→a, ee/y→i, doubled letters).
* **Greedy longest-span linker**: all spans of ≤ 4 content tokens are scored — exact alias, else fuzzy alias (edit ratio), else phonetic repair; spans are then chosen longest-first (exact > fuzzy > phonetic for equal length). Fuzzy/phonetic spans never cross a conjunction ("and", "ar", "আর"), so "rice and daal" stays two dishes.
* Fuzzy and phonetic links are **candidates**: the controller (section 8) accepts them only if the evaluator agrees the dish is relevant to the question.

In [ ]:
_BN_CONS = {"ক": "k", "খ": "kh", "গ": "g", "ঘ": "gh", "ঙ": "ng", "চ": "ch", "ছ": "chh", "জ": "j", "ঝ": "jh", "ঞ": "n", "ট": "t", "ঠ": "th",
            "ড": "d", "ঢ": "dh", "ণ": "n", "ত": "t", "থ": "th", "দ": "d", "ধ": "dh", "ন": "n", "প": "p", "ফ": "ph", "ব": "b", "ভ": "bh",
            "ম": "m", "য": "j", "র": "r", "ল": "l", "শ": "sh", "ষ": "sh", "স": "s", "হ": "h", "ড়": "r", "ঢ়": "rh", "য়": "y", "ৎ": "t"}
_BN_VSIGN = {"া": "a", "ি": "i", "ী": "i", "ু": "u", "ূ": "u", "ৃ": "ri", "ে": "e", "ৈ": "oi", "ো": "o", "ৌ": "ou"}
_BN_VOW = {"অ": "o", "আ": "a", "ই": "i", "ঈ": "i", "উ": "u", "ঊ": "u", "ঋ": "ri", "এ": "e", "ঐ": "oi", "ও": "o", "ঔ": "ou"}
_BN_MISC = {"ং": "ng", "ঃ": "h", "ঁ": "", "্": "", "়": ""}
def bn2latin(s):
    s = unicodedata.normalize("NFC", s).replace("ড়", "ড়").replace("য়", "য়"); out = []
    for i, ch in enumerate(s):
        if ch in _BN_CONS:
            out.append(_BN_CONS[ch]); nxt = s[i + 1] if i + 1 < len(s) else ""
            if nxt and nxt not in _BN_VSIGN and nxt not in ("্", "়") and "ঀ" <= nxt <= "৿": out.append("o")
        elif ch in _BN_VSIGN: out.append(_BN_VSIGN[ch])
        elif ch in _BN_VOW: out.append(_BN_VOW[ch])
        elif ch in _BN_MISC: out.append(_BN_MISC[ch])
        else: out.append(ch)
    return "".join(out)

_PH_RULES = [("chh", "c"), ("bh", "b"), ("v", "b"), ("kh", "k"), ("gh", "g"), ("ch", "c"), ("jh", "j"), ("th", "t"), ("dh", "d"),
             ("ph", "f"), ("sh", "s"), ("z", "j"), ("w", "o"), ("q", "k"), ("x", "ks"), ("ee", "i"), ("oo", "u"), ("y", "i"),
             ("o", "a"), ("u", "a"), ("e", "i")]
def phonetic_key(s):
    s = norm(s)
    if re.search(r"[ঀ-৿]", s): s = bn2latin(s)
    s = re.sub(r"[^a-z]", "", s)
    for a, b in _PH_RULES: s = s.replace(a, b)
    s = re.sub(r"(.)\1+", r"\1", s)
    return s[:-1] if len(s) > 4 and s.endswith("h") else s

STOP = set("""how many much what whats is are the of in a an and with meal calories calorie kcal protein carb carbs carbohydrate
carbohydrates fat fats fiber fibre glycemic gl nutrition nutritional nutrients content value values info information tell me about
give show contain contains does do have has for to on it this that dish food foods picture pictures photo photos image images pics
please some more similar like find look looks er ar ki koto kemon kivabe dekhao dekhan chobi niye bolo amake ektu ache hoy describe
কি কী কত এর আর ছবি দেখাও সম্পর্কে বলো খাবার পুষ্টি ক্যালরি প্রোটিন কার্ব ফ্যাট in per one serving plate""".split())
# v7: question words around diet / recipe questions were linked as dishes in the v6 run ("পারে" -> pari, "kibhabe" -> kibab, "curry" -> churi)
STOP |= set("""ok okay safe good bad healthy unhealthy eat eating eaten can could should would i my we patient patients person people
diabetes diabetic sugar pressure bp blood high low kidney gout uric acid pcos pcod asthma thyroid celiac coeliac gluten anemia anaemia ibs
recipe recipes make making cook cooking cooked prepare preparation way ways steps how to usually typically eaten served
kore korbo korte kibhabe kivabe ranna rannar ranner banay banano banabo khete khabo khawa khaoa pare parbe parben rogi rogir jonno bhalo
kharap khabar thik ache naki
পারে পারবে খেতে খাওয়া খাবো রোগী রোগীর জন্য ভালো খারাপ কিভাবে কীভাবে রান্না রান্নার তৈরি দিয়ে করে করতে যায় হয় কেমন নাকি ডায়াবেটিস সুগার প্রেশার
কিডনি রেসিপি""".split())
# generic cooking words: never linked ALONE by fuzzy / phonetic repair (they are fine inside a full dish name such as "beef curry")
GENERIC = set("curry torkari tarkari bhaji vaji bhaja bhuna jhol bhorta vorta bharta fry roast kalia set plate dish item gravy masala".split())
CONJ = {"and", "ar", "with", "plus", "o", "আর", "ও", "এবং"}

_FOLD = [("\u200c", ""), ("\u200d", ""), ("\u0981", ""), ("\u09c0", "\u09bf"), ("\u09c2", "\u09c1"), ("\u0988", "\u0987"), ("\u098a", "\u0989"),
         ("\u09a3", "\u09a8"), ("\u09b7", "\u09b6"), ("\u09b8", "\u09b6"), ("\u09dc", "\u09b0"), ("\u09a1\u09bc", "\u09b0"), ("\u09dd", "\u09b0"),
         ("\u09a2\u09bc", "\u09b0"), ("\u09ce", "\u09a4"), ("\u0999", "\u0982")]
def bn_fold(s):
    """Fold Bangla spelling variants: long/short i and u signs, ণ/ন, শ/ষ/স, ড়/ঢ়/র, chandrabindu, ZWJ/ZWNJ  (বেগুনী == বেগুনি, খিচুড়ি == খিচুরি)."""
    s = unicodedata.normalize("NFC", s)
    for a, b in _FOLD: s = s.replace(a, b)
    return s

def _merge_into(keep, c):
    if DOCS[c]["text"] and DOCS[c]["text"] not in DOCS[keep]["text"]: DOCS[keep]["text"] += " " + DOCS[c]["text"]
    DOCS[keep]["bn"] = DOCS[keep]["bn"] or DOCS[c]["bn"]
    ALIASES[keep] |= ALIASES.pop(c, set()) | {c}
    if c in FOODBD_MAP and (keep not in FOODBD_MAP or (FOODBD_MAP[keep][1] == "proxy" and FOODBD_MAP[c][1] == "exact")): FOODBD_MAP[keep] = FOODBD_MAP[c]
    for store in (NUTR100, HEALTH):
        if c in store and keep not in store: store[keep] = store[c]
        store.pop(c, None)
    DOC_FACTS[keep] += [x for x in DOC_FACTS.pop(c, []) if x not in DOC_FACTS[keep]]
    FOODBD_MAP.pop(c, None); DOCS.pop(c)

def merge_duplicate_dishes():
    """Merge entries that are the same food, so gold labels and evidence are not split:
    (1) near-duplicate spellings (tanziro 'dundol' / 'dhundul'): same phonetic key AND close strings or the same folded Bangla name;
    (2) v7: identical folded Bangla names (two English spellings of one dish);
    (3) v7: FoodNetBD's bilingual rows as a bridge — English name -> dish A, Bengali name -> dish B  => A == B ('cabbage' / 'bandhakopi',
        which the v6 run scored as an error for 'বাঁধাকপি কি?')."""
    rank = lambda c: ("seed" not in DOCS[c]["source"], not (c in FOODBD_MAP and FOODBD_MAP[c][1] == "exact"), -len(DOCS[c]["text"]))
    merges = []
    groups = collections.defaultdict(list)
    for c, d in DOCS.items():
        k = phonetic_key(d["name"])
        if len(k) >= 4: groups[k].append(c)
    for cs in groups.values():
        if len(cs) < 2: continue
        cs = sorted(cs, key=rank); keep = cs[0]
        for c in cs[1:]:
            same_bn = bool(DOCS[keep]["bn"]) and bn_fold(norm(DOCS[keep]["bn"])) == bn_fold(norm(DOCS[c]["bn"] or "-"))
            if not (fratio(norm(DOCS[keep]["name"]), norm(DOCS[c]["name"])) >= 75 or same_bn): continue
            _merge_into(keep, c); merges.append((c, keep, "spelling"))
    groups = collections.defaultdict(list)
    for c, d in DOCS.items():
        if d["bn"] and len(bn_fold(norm(d["bn"]))) >= 3: groups[bn_fold(norm(d["bn"]))].append(c)
    for cs in groups.values():
        if len(cs) < 2: continue
        cs = sorted(cs, key=rank)
        for c in cs[1:]: _merge_into(cs[0], c); merges.append((c, cs[0], "same Bangla name"))
    for a, b in FNB_PAIRS:
        a = next((k for k in DOCS if a == k or a in ALIASES.get(k, ())), None); b = next((k for k in DOCS if b == k or b in ALIASES.get(k, ())), None)
        if a and b and a != b:
            keep, c = sorted([a, b], key=rank); _merge_into(keep, c); merges.append((c, keep, "FoodNetBD bilingual row"))
    return merges
KB_MERGES = merge_duplicate_dishes()
ALIAS2CANON = {}
for c, al in ALIASES.items():
    for a in al:
        if a and (a not in ALIAS2CANON or len(c) < len(ALIAS2CANON[a])): ALIAS2CANON[a] = c
ALIAS_FOLD = {}
for a, c in ALIAS2CANON.items():
    if not a.isascii(): ALIAS_FOLD.setdefault(bn_fold(a), c)
print(f"merged {len(KB_MERGES)} near-duplicate dishes, e.g. {KB_MERGES[:6]} | KB now {len(DOCS)} dishes")
ALIAS_LIST = list(ALIAS2CANON)
ALIAS_KEYS, ALIAS_KEY_NW = collections.defaultdict(set), {}
for a, c in ALIAS2CANON.items():
    k = phonetic_key(a)
    if len(k) >= 3: ALIAS_KEYS[k].add(c); ALIAS_KEY_NW[k] = max(ALIAS_KEY_NW.get(k, 0), len(a.split()))
ALIAS_KEY_LIST = list(ALIAS_KEYS)

_LINK_CACHE = {}
def link_entities(query, fuzzy_cutoff=84, phonetic=True, phon_cutoff=86, tr=None):
    key = (query, fuzzy_cutoff, phonetic, phon_cutoff)
    if key not in _LINK_CACHE:
        ev = []; _LINK_CACHE[key] = (_link(query, fuzzy_cutoff, phonetic, phon_cutoff, ev), ev)
    found, ev = _LINK_CACHE[key]
    if tr is not None: tr.extend(ev)
    return dict(found)

_BN_SUFFIX = tuple(unicodedata.normalize("NFC", x) for x in ("য়ের", "য়ে", "ের", "এর", "র", "টা", "টি", "গুলো", "তে", "কে"))
_BL_SUFFIX = ("er", "r", "ta", "ti", "gulo")
def strip_case(span):
    """Strip one Bangla/Banglish case or classifier suffix from the last word (genitive -er/-এর, -টা/-টি ...)."""
    *head, last = span.split(" ")
    for suf in (_BN_SUFFIX if not last.isascii() else _BL_SUFFIX):
        if last.endswith(suf) and len(last) - len(suf) >= (2 if not last.isascii() else 4):
            return " ".join(head + [last[: -len(suf)]])
    return None

def _link(query, fuzzy_cutoff, phonetic, phon_cutoff, ev):
    toks = norm(query).split(); cands = []
    for n in range(min(4, len(toks)), 0, -1):
        for i in range(len(toks) - n + 1):
            st = toks[i:i + n]; span = " ".join(st)
            if n >= 2 and span in ALIAS2CANON:                       # a full dish name may end in a stop word ("beef curry plate")
                cands.append((n, 3, 100.0, i, ALIAS2CANON[span], "alias:" + span)); continue
            if st[0] in STOP or st[-1] in STOP: continue
            if span in ALIAS2CANON and (len(span) >= 3 or not span.isascii()):
                cands.append((n, 3, 100.0, i, ALIAS2CANON[span], "alias:" + span)); continue
            st_ = strip_case(span)                                   # ইলিশ মাছের -> ইলিশ মাছ, biriyanir -> biriyani
            if st_ and st_ in ALIAS2CANON:
                cands.append((n, 3, 99.0, i, ALIAS2CANON[st_], f"alias:{st_} (suffix stripped from '{span}')")); continue
            if not span.isascii():                                   # Bangla spelling variants: বেগুনী -> বেগুনি
                fs = bn_fold(span); hit = next((x for x in (fs, strip_case(fs)) if x and x in ALIAS_FOLD), None)
                if hit: cands.append((n, 3, 98.0, i, ALIAS_FOLD[hit], f"alias:{hit} (Bangla spelling folded from '{span}')")); continue
            if any(t in CONJ for t in st) or (span.isascii() and len(span.replace(" ", "")) < 4): continue
            if n == 1 and span in GENERIC: continue
            if n >= 2 and (len(st[0]) <= 2 or len(st[-1]) <= 2): continue        # 'biriyani te', 'singara r', 'e ilish': particles are not dish words
            m = best_match(span, ALIAS_LIST, fuzzy_cutoff)
            # v7: a fuzzy match must explain every word of the span ("khichuri ok" -> "khichuri" absorbed "ok" and then lost to link
            # verification, so "is khichuri ok for diabetes?" abstained; "make bottle gourd" -> "lau bottle gourd")
            if m and n >= 2:
                at = m[0].split()
                if len(at) < n or fratio(st[0], at[0]) < 60 or fratio(st[-1], at[-1]) < 60: m = None
            if m: cands.append((n, 2, m[1], i, ALIAS2CANON[m[0]], f"fuzzy:{span}->{m[0]}")); continue
            if phonetic:
                k = phonetic_key(span)
                if len(k) >= 4:
                    m = best_match(k, ALIAS_KEY_LIST, phon_cutoff)
                    if m and len(ALIAS_KEYS[m[0]]) == 1 and ALIAS_KEY_NW.get(m[0], 1) >= n:
                        cands.append((n, 1, m[1], i, next(iter(ALIAS_KEYS[m[0]])), f"phonetic:{span}->{m[0]}"))
    cands.sort(key=lambda c: (-c[0], -c[1], -c[2], c[3]))
    used, found = set(), {}
    for n, meth, sc, i, canon, how in cands:
        if any(j in used for j in range(i, i + n)): continue
        used.update(range(i, i + n))
        if canon not in found:
            found[canon] = (sc, how)
            if meth < 3: ev.append(f"QUERY_REPAIR {how}")
    return found

NUTR_MAP = {"calorie": ["Calorie"], "calories": ["Calorie"], "kcal": ["Calorie"], "ক্যালরি": ["Calorie"], "ক্যালোরি": ["Calorie"],
            "protein": ["Protein"], "প্রোটিন": ["Protein"], "carb": ["CHO"], "carbs": ["CHO"], "carbohydrate": ["CHO"],
            "carbohydrates": ["CHO"], "শর্করা": ["CHO"], "fat": ["Fat"], "fats": ["Fat"], "ফ্যাট": ["Fat"], "চর্বি": ["Fat"],
            "fiber": ["Fiber"], "fibre": ["Fiber"], "glycemic": ["GL"], "nutrition": NUTR, "nutritional": NUTR, "nutrients": NUTR, "পুষ্টি": NUTR}
IMG_WORDS = {"picture", "pictures", "photo", "photos", "image", "images", "pics", "chobi", "ছবি", "dekhao", "দেখাও", "similar"}
def detect_intent(q):
    toks = set(norm(q).split()); cols = set()
    for t in toks: cols.update(NUTR_MAP.get(t, []))
    cols = [c for c in NUTR if c in cols]; want_imgs = bool(toks & IMG_WORDS)
    return ("nutrition" if cols else ("images" if want_imgs else "describe")), cols, want_imgs

for t in ["calories in kalavuna", "what is roshogolla?", "rice and daal calories", "বিরিয়ানি এর ক্যালরি কত", "খিচুরি কি",
          "biriani er chobi dekhao", "tell me about shorshe ilsh", "what is the capital of France", "fiber in zxqowk"]:
    print(f"{t!r:34} -> {detect_intent(t)[0]:9} {{{', '.join(f'{k}: {v[1]}' for k, v in link_entities(t).items())}}}")

## 6. Hybrid retrieval and the CRAG-style retrieval evaluator (batched + cached)

* Dense `BAAI/bge-m3` (multilingual) — baselines use the English `bge-small-en-v1.5` of v1/v2.
* Lexical char 2–4-gram TF-IDF over name + Bangla name + transliteration + aliases + description.
* Reciprocal-rank fusion (k = 60).
* Evaluator `bge-reranker-v2-m3` → CORRECT / AMBIGUOUS / INCORRECT, thresholds tuned on dev.
* `prefetch_*` functions encode all benchmark queries and score all (query, dish) pairs in large GPU batches; results are memoised.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
DOC_IDS = sorted(DOCS); DOC_POS = {c: i for i, c in enumerate(DOC_IDS)}
def name_block(c):
    d = DOCS[c]; return " ".join([d["name"], d["bn"], bn2latin(d["bn"])] + sorted(ALIASES.get(c, [])))
DOC_TEXT = {c: f"{DOCS[c]['name']}{' (' + DOCS[c]['bn'] + ')' if DOCS[c]['bn'] else ''}. {DOCS[c]['text']}" for c in DOC_IDS}
INDEX_TEXT = [name_block(c) + ". " + DOC_TEXT[c] for c in DOC_IDS]
KB_KEY = hashlib.md5("\n".join(INDEX_TEXT).encode()).hexdigest()[:10]

def content_query(q):
    toks = [t for t in norm(q).split() if t not in STOP]
    return " ".join(toks) if toks else norm(q)
def expand_query(q):
    bn = re.findall(r"[ঀ-৿]+", q)
    return q + (" " + " ".join(bn2latin(t) for t in bn) if bn else "")

class DenseEncoder:
    def __init__(self, name, tag):
        self.model = None
        if HAS_ST and not SMOKE:
            from sentence_transformers import SentenceTransformer
            self.model = try_load(tag, lambda: SentenceTransformer(model_src(name), device=DEVICE))
            if self.model is not None and GPU: self.model.half()
        else: MODEL_STATUS[tag] = "FALLBACK - word TF-IDF" + (" (SMOKE)" if SMOKE else " (no sentence-transformers)")
        self.kind = name if self.model is not None else "word-tfidf fallback"; self._q = {}
        if self.model is None:
            self.vec = TfidfVectorizer(analyzer="word", ngram_range=(1, 2), sublinear_tf=True, token_pattern=r"(?u)\b\w+\b")
    def fit(self, texts):
        if self.model is None: self.vec.fit(texts)
        return self
    def encode(self, texts, bs=32):
        texts = list(texts)
        if self.model is not None:
            return np.asarray(self.model.encode(texts, normalize_embeddings=True, batch_size=bs, show_progress_bar=False), dtype=np.float32)
        X = self.vec.transform(texts).toarray(); return X / np.maximum(np.linalg.norm(X, axis=1, keepdims=True), 1e-9)
    def encode_queries(self, qs):          # memoised, batched
        new = [q for q in dict.fromkeys(qs) if q not in self._q]
        if new:
            for q, v in zip(new, self.encode(new, bs=64)): self._q[q] = v
        return np.stack([self._q[q] for q in qs])

class HybridRetriever:
    def __init__(self, enc, base_enc):
        self.enc, self.base = enc.fit(INDEX_TEXT), base_enc.fit(INDEX_TEXT)
        self.E = cached("doc_emb", lambda: self.enc.encode(INDEX_TEXT), (self.enc.kind, KB_KEY)) if self.enc.model is not None else self.enc.encode(INDEX_TEXT)
        self.EB = cached("doc_emb", lambda: self.base.encode(INDEX_TEXT), (self.base.kind, KB_KEY)) if self.base.model is not None else self.base.encode(INDEX_TEXT)
        self.lex = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), sublinear_tf=True).fit(INDEX_TEXT)
        self.L = self.lex.transform(INDEX_TEXT); self.NB = self.lex.transform([name_block(c) for c in DOC_IDS]); self._cache = {}
    def prefetch(self, queries):
        qs = list(dict.fromkeys(expand_query(q) for q in queries))
        self.enc.encode_queries(qs); self.base.encode_queries(qs)
    def scores(self, qx, mode):
        if mode == "base": return self.EB @ self.base.encode_queries([qx])[0]
        if mode == "dense": return self.E @ self.enc.encode_queries([qx])[0]
        if mode == "lexical": return (self.L @ self.lex.transform([qx]).T).toarray().ravel()
        d, l = self.scores(qx, "dense"), self.scores(qx, "lexical")
        rd, rl = np.argsort(np.argsort(-d)), np.argsort(np.argsort(-l))
        return 1 / (60 + rd) + 1 / (60 + rl)
    def search(self, q, k=8, mode="hybrid"):
        key = (q, k, mode)
        if key not in self._cache:
            qx = expand_query(q); s = self.scores(qx, mode); idx = np.argsort(-s)[:k]
            conf = float(self.scores(qx, "dense" if mode == "hybrid" else mode)[idx[0]])
            self._cache[key] = ([DOC_IDS[i] for i in idx], conf)
        r, c = self._cache[key]; return list(r), c
    def name_match(self, q, canons):
        v = self.lex.transform([expand_query(content_query(q))])
        return (self.NB[[DOC_POS[c] for c in canons]] @ v.T).toarray().ravel()

class RetrievalEvaluator:
    def __init__(self):
        self.ce = None; tag = "evaluator:" + CFG["RERANK_MODEL"]
        if HAS_ST and not SMOKE:
            from sentence_transformers import CrossEncoder
            self.ce = try_load(tag, lambda: CrossEncoder(model_src(CFG["RERANK_MODEL"]), max_length=384, device=DEVICE))
            if self.ce is not None and GPU:
                try: self.ce.model.half()
                except Exception: pass
        else: MODEL_STATUS[tag] = "FALLBACK - name-match char n-gram score" + (" (SMOKE)" if SMOKE else "")
        self.kind = CFG["RERANK_MODEL"] if self.ce is not None else "name-match fallback"
        self._pair = cached("grader_pairs", dict, (self.kind, KB_KEY)) if self.ce is not None else {}
    def prefetch(self, pairs, bs=64):
        new = list(dict.fromkeys(p for p in pairs if p not in self._pair and p[1] in DOCS))
        if not new: return
        if self.ce is not None:
            s = np.asarray(self.ce.predict([(q, DOC_TEXT[c][:1200]) for q, c in new], batch_size=bs, show_progress_bar=False), dtype=float)
            s = 1 / (1 + np.exp(-s)) if (s.min() < 0 or s.max() > 1) else s
        else:
            s = np.concatenate([self._fallback(q, [c]) for q, c in new])
        for p, v in zip(new, s): self._pair[p] = float(v)
    def _fallback(self, q, canons): return RET.name_match(q, canons)
    def score(self, q, canons):
        canons = [c for c in canons if c in DOCS]
        self.prefetch([(q, c) for c in canons]); return np.array([self._pair[(q, c)] for c in canons])
    def _fname(self): return f"grader_pairs_{hashlib.md5(str((self.kind, KB_KEY)).encode()).hexdigest()[:10]}.pkl"
    def load(self):
        """v7: v6 saved the evaluator's pair scores but never read them back."""
        src = cache_file(self._fname())
        if self.ce is not None and src:
            with open(src, "rb") as f: self._pair.update(pickle.load(f))
            print(f"evaluator cache: {len(self._pair)} scored pairs reused from {src}")
    def save(self):
        if self.ce is not None:
            with open(os.path.join(CACHE, self._fname()), "wb") as f: pickle.dump(self._pair, f)

with section("load retriever + evaluator"):
    ENC = DenseEncoder(CFG["EMBED_MODEL"], "embedder:" + CFG["EMBED_MODEL"])
    BASE_ENC = DenseEncoder(CFG["BASELINE_EMBED"], "baseline-embedder:" + CFG["BASELINE_EMBED"])
    RET = HybridRetriever(ENC, BASE_ENC)
    GRADER = RetrievalEvaluator(); GRADER.load()
THR = {"base": 0.5, "dense": 0.5, "grader_lo": 0.3, "grader_hi": 0.6, "link_min": 0.05, "link_rank": 0, "link_floor": 0.0, "link_sim": 101, "link_rep": 1.01, "cover_min": 0.0}   # replaced by dev-tuned values in section 11

ASPECTS = [  # (question cue, sentence cue): aspect questions must see the sentence that answers them, whatever its lexical overlap
    (r"\b(with|sathe|shathe|diye|served|eaten|accompan)\b|সাথে|সঙ্গে|দিয়ে খায়", r"served with|appears in \d+ annotated meals"),
    (r"ingredient|made (of|from|with)|kivabe|kibhabe|kibabe|ranna|recipe|how (to|do you|can i) (make|cook|prepare)|banano|banabo|তৈরি|উপকরণ|রান্না|কিভাবে|কীভাবে", r"^Ingredients:|Cooking method:"),
    (r"per 100|100 ?g|foodnetbd", r"FoodNetBD lists"),
    (r"region|where|kothay|কোথায়|অঞ্চল", r"^Region:"),
    (r"weight|gram|portion|serving", r"FoodBD metadata, median")]
def refine_evidence(canon, q, max_sents=3):
    """CRAG decompose-then-recompose: keep sentences most related to the question, always keep the identity sentence,
    and always keep a sentence that answers the question's aspect (served-with, ingredients, per-100 g, region, portion)."""
    sents = [s.strip() for s in re.split(r"(?<=[.!?।])\s+", DOC_TEXT[canon]) if len(s.strip()) > 3]
    asked = [sa for qa, sa in ASPECTS if re.search(qa, q, re.I)]
    facts = [s for s in DOC_FACTS.get(canon, []) if any(re.search(sa, s) for sa in asked)][:2]
    forced = {i for sa in asked for i, s in enumerate(sents) if re.search(sa, s)}
    if len(sents) <= max_sents: return sents + facts
    sim = (RET.lex.transform(sents) @ RET.lex.transform([expand_query(q)]).T).toarray().ravel()
    keep = sorted({0} | set(list(forced)[:2]) | set(np.argsort(-sim)[:max(1, max_sents - 1 - len(forced))].tolist()))
    return [sents[i] for i in keep] + facts

print("dense:", ENC.kind, "| baseline dense:", BASE_ENC.kind, "| evaluator:", GRADER.kind)
for q in ["what is kalavuna", "ইলিশ মাছ কি", "tell me about the capital of France"]:
    rk, conf = RET.search(q); g = GRADER.score(q, rk[:3])
    print(f"{q!r:38} top3={rk[:3]} dense_conf={conf:.2f} evaluator={np.round(g, 2).tolist()}")

## 7. Generation and claim verification (generation-side heal loop)

1. **Generate** from refined evidence with `Qwen2.5-1.5B-Instruct` (4-bit, ≈ 1.2 GB VRAM). SMOKE mode has no LLM, so a *simulated noisy generator* copies evidence and, for ~35 % of questions, appends one fabricated claim — a controlled test of the verifier, never used for real answers.
2. **Verify** each sentence: numbers must appear in the evidence, the dish must be named, and each sentence must be entailed by an evidence sentence or by the whole evidence (multilingual NLI `mDeBERTa-v3-base-xnli`; fallback = content-word coverage). Sentences copied verbatim from evidence skip NLI.
3. **Heal**: drop unsupported sentences → stricter regeneration → extractive evidence. Every step is logged.

In [ ]:
class NLIChecker:
    def __init__(self):
        self.model = None; tag = "nli:" + CFG["NLI_MODEL"]
        if HAS_TF and HAS_TORCH and not SMOKE:
            from transformers import AutoTokenizer, AutoModelForSequenceClassification
            def _l():
                tok = AutoTokenizer.from_pretrained(model_src(CFG["NLI_MODEL"]))
                m = AutoModelForSequenceClassification.from_pretrained(model_src(CFG["NLI_MODEL"])).to(DEVICE).eval()
                return tok, m
            got = try_load(tag, _l)
            if got:
                self.tok, self.model = got
                self.ent = [i for i, l in self.model.config.id2label.items() if "entail" in l.lower()][0]
        else: MODEL_STATUS[tag] = "FALLBACK - content-word coverage" + (" (SMOKE)" if SMOKE else "")
        self.kind = CFG["NLI_MODEL"] if self.model is not None else "word-coverage fallback"
        self.min_support = 0.5 if self.model is not None else 0.6
    def support_many(self, premises, hyps):
        """max entailment of each hypothesis over premises (+ the concatenated premise), one batched forward pass"""
        if not premises or not hyps: return [0.0] * len(hyps)
        prem = premises + ([" ".join(premises)] if len(premises) > 1 else [])
        if self.model is not None:
            P = [p for h in hyps for p in prem]; H = [h for h in hyps for _ in prem]
            enc = self.tok(P, H, truncation=True, padding=True, max_length=320, return_tensors="pt").to(self.model.device)
            with torch.no_grad(), amp(): pr = torch.softmax(self.model(**enc).logits.float(), -1)[:, self.ent].cpu().numpy()
            return pr.reshape(len(hyps), len(prem)).max(1).tolist()
        ev = set(t for p in prem for t in norm(p).split()); out = []
        for h in hyps:
            toks = [t for t in norm(h).split() if len(t) >= 3 and t not in STOP]
            out.append(sum(t in ev for t in toks) / max(len(toks), 1))
        return out

with section("load NLI verifier"):
    NLI = NLIChecker()

def numbers_in(t): return set(re.findall(r"\d+(?:\.\d+)?", t))
META_TALK = re.compile(r"\b(the|this|provided|given) (evidence|context|passage|text|information|source)s?\b|\bas an ai\b|\bsearch (engine|online)\b", re.I)
def split_sents(t): return [s.strip() for s in re.split(r"(?<=[.!?।])\s+", t) if len(s.strip()) > 3]
def mentions_dish(text, canon):
    t = " " + norm(text) + " "
    return any(f" {a} " in t for a in (ALIASES.get(canon, set()) | {norm(DOCS[canon]["name"])}) if a)

HEALTH_CLAIM = re.compile(r"diabet|blood sugar|pressure|kidney|\bgout\b|pcos|thyroid|gluten|celiac|asthma|anemi|safe (for|to)|can (eat|have)|"
                          r"should(n't| not)? (eat|avoid)|suitable|healthy|unhealthy|good for|bad for|ডায়াবেটিস|প্রেশার|খেতে পারে", re.I)
def lexical_support(s, ctx_words):
    """Share of a sentence's content words (len >= 4) that occur in the evidence (5-letter stem match). v6's NLI-only check passed
    'Korola vaji is made by boiling small bamboo shoots' and 'lau is sliced and served cold' — neither 'bamboo' nor 'cold' is in the evidence."""
    w = [t for t in norm(s).split() if len(t) >= 4 and t not in STOP]
    if not w: return 1.0
    return sum(t[:5] in ctx_words for t in w) / len(w)

_VERIFY = {}
def verify(answer, evidence, canon):
    key = (answer, tuple(evidence), canon)
    if key in _VERIFY: return _VERIFY[key]
    ctx = " ".join(evidence); evset = set(s.strip() for s in evidence)
    out = {"num_ok": numbers_in(answer) <= numbers_in(ctx), "entity_ok": mentions_dish(answer, canon), "unsupported": []}
    sents = split_sents(answer); todo = [s for s in sents if s not in evset and numbers_in(s) <= numbers_in(ctx)]
    sup = dict(zip(todo, NLI.support_many(evidence, todo)))
    ctx_words = {t[:5] for t in norm(ctx).split() if len(t) >= 4}; out["lexical_fail"] = 0
    for s in sents:
        if s in evset: continue
        lex = lexical_support(s, ctx_words) >= CFG.get("LEX_MIN", 0.6)
        if not lex: out["lexical_fail"] += 1
        if (META_TALK.search(s) or HEALTH_CLAIM.search(s) or not numbers_in(s) <= numbers_in(ctx) or sup.get(s, 0) < NLI.min_support
                or not lex): out["unsupported"].append(s)
    out["ok"] = out["num_ok"] and out["entity_ok"] and not out["unsupported"]
    _VERIFY[key] = out; return out

def extractive_answer(q, canon, evidence, strict=False): return " ".join(evidence)

_LLM = {}
def llm_available():
    if not (CFG["USE_LLM"] and HAS_TF and HAS_TORCH) or SMOKE:
        MODEL_STATUS.setdefault("generator:" + CFG["GEN_MODEL"], "FALLBACK - " + ("simulated noisy generator (SMOKE)" if SMOKE else "extractive")); return False
    if "m" not in _LLM: _LLM["m"] = try_load("generator:" + CFG["GEN_MODEL"], lambda: load_causal_lm(CFG["GEN_MODEL"]))
    return _LLM["m"] is not None

_GEN_CACHE = {}
def llm_answer(q, canon, evidence, strict=False):
    key = (q, canon, strict)
    if key not in _GEN_CACHE:
        tok, model = _LLM["m"]; ev = "\n".join(f"[{i + 1}] {s}" for i, s in enumerate(evidence))
        rule = ("Copy facts exactly as written in the evidence. At most 2 sentences." if strict else
                "Use only facts from the evidence. At most 3 short sentences. Do not add numbers, places or ingredients that are not in the evidence.")
        msgs = [{"role": "system", "content": "You answer questions about Bangladeshi food. " + rule + f" Always name the dish ({DOCS[canon]['name']})."},
                {"role": "user", "content": f"Evidence:\n{ev}\n\nQuestion: {q}\nAnswer in English:"}]
        _GEN_CACHE[key] = chat_generate(tok, model, msgs, max_new_tokens=100 if strict else 140)
    return _GEN_CACHE[key]

_FAKE = ["It usually contains about {n} kcal per serving.", "It originated in {c} during the {k}th century.",
         "It is traditionally cooked with {i}.", "Each portion has roughly {n} grams of protein."]
def simulated_noisy_answer(q, canon, evidence, strict=False):
    rng = random.Random(int(hashlib.md5((q + canon).encode()).hexdigest()[:8], 16)); ans = " ".join(evidence[:2])
    if not strict and rng.random() < 0.35:
        ans += " " + rng.choice(_FAKE).format(n=rng.randint(120, 900), c=rng.choice(["Persia", "Portugal", "Japan"]), k=rng.randint(12, 18),
                                                i=rng.choice(["cheese", "olive oil", "soy sauce", "seaweed"]))
    return ans

def generate_verified(q, canon, evidence, tr, use_llm=True, do_verify=True):
    gen = extractive_answer
    if use_llm: gen = llm_answer if llm_available() else (simulated_noisy_answer if SMOKE else extractive_answer)
    raw = gen(q, canon, evidence); info = {"raw": raw, "generator": gen.__name__}
    v = verify(raw, evidence, canon); info["raw_ok"] = v["ok"]
    if not do_verify or v["ok"]: info["stage"] = "raw"; return raw, info
    kept = [s for s in split_sents(raw) if s not in v["unsupported"]]
    tr.append(f"UNSUPPORTED_CLAIMS {len(v['unsupported'])} (num_ok={v['num_ok']}, entity_ok={v['entity_ok']}) -> drop sentences")
    if kept and verify(" ".join(kept), evidence, canon)["ok"]: info["stage"] = "dropped_unsupported"; return " ".join(kept), info
    if gen is not extractive_answer:
        tr.append("REGENERATE strict prompt"); ans2 = gen(q, canon, evidence, strict=True)
        if verify(ans2, evidence, canon)["ok"]: info["stage"] = "strict_regenerate"; return ans2, info
    tr.append("EXTRACTIVE_FALLBACK"); info["stage"] = "extractive"
    return extractive_answer(q, canon, evidence), info

## 8. The self-healing controller — expensive features once, cheap decisions many times

```
query ─► intent router ─► linker (longest span: exact > fuzzy > phonetic repair)
                │            │ fuzzy/phonetic link? ─► evaluator agrees? no ─► LINK_REJECTED
                │            │ linked ─► entity guard: evidence scoped to linked dishes
                │            ▼ not linked
                │     hybrid retrieval ─► evaluator: CORRECT / AMBIGUOUS / INCORRECT
                │            │ not CORRECT ─► rewrite (content words + transliteration) ─► re-link / re-retrieve ─► re-grade
                │            │ INCORRECT ─► ABSTAIN   · weak AMBIGUOUS ─► ABSTAIN   · close call ─► CLARIFY
                ▼            ▼
  nutrition: FoodBD lookup tiers + wide-range / proxy warnings — numbers are never generated
  describe : refined evidence ─► generator ─► verifier ─► drop / regenerate / extractive
  images   : picture retrieval ─► label-consistency filter (section 9)
```
`query_features(q)` computes everything that needs a model (links, rankings, evaluator scores for the query, its rewrites and every candidate dish) — once per query, batched. `decide(features, system, thresholds)` is pure Python, so baselines, ablations and threshold tuning cost almost nothing.

In [ ]:
import requests
_WIKI = {}
def wiki_summary(name):
    if SMOKE: return None
    if name in _WIKI: return _WIKI[name]
    try:
        r = requests.get("https://en.wikipedia.org/api/rest_v1/page/summary/" + name.title().replace(" ", "_"),
                         headers={"User-Agent": "bd-food-selfheal-rag/0.4 (student project)"}, timeout=6)
        _WIKI[name] = r.json().get("extract") if r.status_code == 200 and r.json().get("type") == "standard" else None
    except Exception: _WIKI[name] = None
    return _WIKI[name]

_B = dict(linker=False, phonetic=False, retriever="base", grader=False, rewrite=False, verify=False, abstain=None, guard=False, link_verify=False, probe=False)
_F = dict(linker=True, phonetic=True, retriever="hybrid", grader=True, rewrite=True, verify=True, abstain="grader", guard=True, link_verify=True, probe=True)
SYSTEMS = {
    "B0 naive dense (v1)":            dict(_B),
    "B1 dense+threshold (v2-base)":   dict(_B, abstain="base"),
    "B2 CRAG-style":                  dict(_B, retriever="hybrid", grader=True, rewrite=True, abstain="grader"),
    "B3 v2 linker+threshold":         dict(_B, linker=True, abstain="base", guard=True),
    "FULL v8":                        dict(_F),
    "FULL - phonetic repair":         dict(_F, phonetic=False),
    "FULL - link verification":       dict(_F, link_verify=False),
    "FULL - intent-stripped grading": dict(_F, probe=False),
    "FULL - evaluator":               dict(_F, grader=False, rewrite=False, abstain="dense", link_verify=False, probe=False),
    "FULL - verifier":                dict(_F, verify=False),
}
FULL = SYSTEMS["FULL v8"]
for S in SYSTEMS.values(): S["thr"] = dict(THR)

def rewrites(q):
    out = []
    for r in (content_query(expand_query(q)), bn2latin(content_query(q))):
        if r and r != q and r not in out: out.append(r)
    return out or [content_query(q) or q]

def plan_features(q):
    """Model-free part: links and rankings (the dense query embeddings must be prefetched)."""
    f = {"q": q, "links": {}, "rw_links": {}, "rank": {}, "conf": {}, "rw_rank": []}
    for ph in (True, False):
        ev = []; f["links"][ph] = (link_entities(q, phonetic=ph, tr=ev), ev)
        f["rw_links"][ph] = []
        for q2 in rewrites(q):
            ev2 = []; f["rw_links"][ph].append((link_entities(q2, fuzzy_cutoff=80, phonetic=ph, phon_cutoff=82, tr=ev2), ev2))
    for mode in ("base", "dense", "hybrid"):
        f["rank"][mode], f["conf"][mode] = RET.search(q, k=8, mode=mode)
    f["rw_rank"] = [RET.search(q2, k=8, mode="hybrid")[0] for q2 in rewrites(q)]
    return f

def probe_of(q):
    """Intent-stripped probe: the dish mention only ("how many calories in X?" -> "x"). The KB holds descriptions, FoodBD holds the
    numbers, so the evaluator should judge whether the passage is about the *dish*, not whether it answers the nutrition request."""
    cq = content_query(q); return strip_case(cq) or cq

def link_cands(f):
    return list(dict.fromkeys(c for ph in (True, False) for c in list(f["links"][ph][0]) + [c for e, _ in f["rw_links"][ph] for c in e] if c in DOCS))

def grader_pairs(f):
    q = f["q"]; pq = probe_of(q); top = f["rank"]["hybrid"][:5]
    pairs = [(q, c) for c in top] + [(pq, c) for c in top]
    for q2, r in zip(rewrites(q), f["rw_rank"]): pairs += [(q2, c) for c in r[:5]]
    pairs += [(x, c) for c in link_cands(f) for x in (q, pq)]   # every candidate link: original question AND its dish mention
    pairs += [(rq, c) for c, rq in repaired_queries(f).items()]
    return pairs

def _span_of(how):
    m = re.match(r"(?:fuzzy|phonetic):(.+?)->", how); return m.group(1) if m else None
def repaired_queries(f):
    """v7: the question with the misspelled span replaced by the dish name ('what is addo?' -> 'what is adda?'). v6 scored only the
    misspelled question, which a cross-encoder cannot match (addo / dewo / bheef were rejected at evaluator 0.02)."""
    out, nq = {}, norm(f["q"])
    for ph in (True, False):
        for ents in [f["links"][ph][0]] + [e for e, _ in f["rw_links"][ph]]:
            for c, (sc, how) in ents.items():
                sp = _span_of(how)
                if sp and c in DOCS and sp in nq and c not in out: out[c] = nq.replace(sp, norm(DOCS[c]["name"].split("(")[0]))
    return out
def link_coverage(f, c, how):
    """Share of the question's content words explained by the repaired span (near-miss foreign dishes are only partly explained)."""
    sp = _span_of(how); ct = content_query(f["q"]).split()
    return len(sp.split()) / max(len(ct), 1) if sp else 1.0

def _finish(f):
    q = f["q"]; pq = probe_of(q); top = f["rank"]["hybrid"][:5]
    f["g0"] = GRADER.score(q, top); f["g0p"] = np.maximum(f["g0"], GRADER.score(pq, top))
    f["g_rw"] = [GRADER.score(q2, r[:5]) for q2, r in zip(rewrites(q), f["rw_rank"])]
    f["link_score"] = {c: float(max(GRADER.score(q, [c])[0], GRADER.score(pq, [c])[0])) for c in link_cands(f)}
    ranks = [f["rank"]["hybrid"]] + f["rw_rank"]                   # retrieval agreement: best rank of the dish for the question or a rewrite
    f["link_rank"] = {c: min([r.index(c) + 1 for r in ranks if c in r] or [99]) for c in link_cands(f)}
    f["link_rep"] = {c: float(GRADER.score(rq, [c])[0]) for c, rq in repaired_queries(f).items()}
    return f

def query_features(queries):
    """Batched: one dense-encoding pass and one cross-encoder pass for all queries."""
    allq = list(dict.fromkeys(queries)); RET.prefetch(allq + [q2 for q in allq for q2 in rewrites(q)])
    feats = {q: plan_features(q) for q in allq}
    GRADER.prefetch([p for f in feats.values() for p in grader_pairs(f)])
    if len(allq) > 20: GRADER.save()
    return {q: _finish(f) for q, f in feats.items()}

FEATS = {}
def get_features(q):
    if q not in FEATS: FEATS.update(query_features([q]))
    return FEATS[q]

def _verified_links(f, ents, S, TH, tr):
    """A repaired (fuzzy/phonetic) link survives if ANY independent signal agrees: the evaluator, retrieval rank, or a near-exact string match."""
    ents = dict(ents)
    if S["link_verify"]:
        for c, (sc, how) in list(ents.items()):
            if how.startswith("alias:") or c not in DOCS: continue
            ls, rk = f["link_score"].get(c, 0.0), f.get("link_rank", {}).get(c, 99)
            # retrieval rank is not fully independent of the misspelled string (lexical n-grams), so it also needs a small evaluator floor (v5:
            # rank alone let foreign / near-miss dishes through: neg:near-miss recall fell to 0.68)
            lr, cov = f.get("link_rep", {}).get(c, 0.0), link_coverage(f, c, how)
            if cov < TH.get("cover_min", 0.0):
                ents.pop(c); tr.append(f"LINK_REJECTED {how} (explains only {cov:.0%} of the question's content words < {TH['cover_min']:.0%})"); continue
            if (ls >= TH["link_min"] or (rk <= TH.get("link_rank", 0) and ls >= TH.get("link_floor", 0.0)) or sc >= TH.get("link_sim", 101)
                    or (lr >= TH.get("link_rep", 1.01) and sc >= 80)):
                if lr >= TH.get("link_rep", 1.01) and ls < TH["link_min"]: tr.append(f"LINK_ACCEPTED {how} (repaired question scores {lr:.2f})")
                continue
            ents.pop(c); tr.append(f"LINK_REJECTED {how} (evaluator {ls:.3f} < {TH['link_min']:.3f}, retrieval rank {rk} > {TH.get('link_rank', 0)} "
                                   f"or evaluator < floor {TH.get('link_floor', 0):.3f}, string sim {sc:.0f} < {TH.get('link_sim', 101)}, repaired question {lr:.2f})")
    return ents

def decide(f, S, TH, tr):
    """Pure decision rule over precomputed features -> (dishes, abstained, kind, ranking)."""
    ph = S["phonetic"]; ranking, conf = f["rank"][S["retriever"]], f["conf"][S["retriever"]]
    if S["linker"]:
        ents, ev = f["links"][ph]; tr.extend(ev); ents = _verified_links(f, ents, S, TH, tr)
        if ents:
            if S["guard"] and ranking[0] not in ents:
                tr.append(f"ENTITY_GUARD retrieval top '{ranking[0]}' disagrees with linked {list(ents)} -> evidence scoped to linked dishes")
            return list(ents), False, "linked", [e for e in ents if e in DOCS] + [c for c in ranking if c not in ents]
    if S["grader"]:
        rk, g = f["rank"]["hybrid"][:5], (f["g0p"] if S.get("probe") else f["g0"]); o = np.argsort(-g); best, gs = rk[o[0]], float(g[o[0]])
        act = lambda s: "CORRECT" if s >= TH["grader_hi"] else ("INCORRECT" if s < TH["grader_lo"] else "AMBIGUOUS")
        action = act(gs); tr.append(f"EVALUATOR {action} ({gs:.2f} for '{best}')")
        if S["rewrite"]:
            for step, q2 in enumerate(rewrites(f["q"])[:CFG["MAX_HEAL_STEPS"]]):
                if action == "CORRECT": break
                if S["linker"]:
                    e2, ev2 = f["rw_links"][ph][step]; tr.extend(ev2); e2 = _verified_links(f, e2, S, TH, tr)
                    if e2:
                        tr.append(f"REWRITE#{step + 1} '{q2}' -> linked {list(e2)}")
                        return list(e2), False, "linked", list(e2) + [c for c in rk if c not in e2]
                r2, g2 = f["rw_rank"][step][:5], f["g_rw"][step]; o2 = np.argsort(-g2)
                if len(g2) and float(g2[o2[0]]) > gs: best, gs, g, rk, o = r2[o2[0]], float(g2[o2[0]]), g2, r2, o2
                action = act(gs); tr.append(f"REWRITE#{step + 1} '{q2}' -> EVALUATOR {action} ({gs:.2f} for '{best}')")
        ranked = [rk[i] for i in o]
        if action == "INCORRECT": tr.append("ABSTAIN (no relevant evidence after healing)"); return [], True, "abstain", ranked
        if action == "AMBIGUOUS" and len(g) > 1:
            second, mid = float(g[o[1]]), (TH["grader_lo"] + TH["grader_hi"]) / 2
            if gs < mid: tr.append(f"ABSTAIN ambiguous evidence {gs:.2f} < {mid:.2f}"); return [], True, "abstain", ranked
            if gs - second < 0.10 and rk[o[1]] != best:
                tr.append(f"CLARIFY '{best}' vs '{rk[o[1]]}' (margin {gs - second:.2f})"); return [best, rk[o[1]]], True, "clarify", ranked
        return [best], False, "retrieved", ranked
    if S["abstain"] in ("base", "dense") and conf < TH[S["abstain"]]:
        tr.append(f"ABSTAIN similarity {conf:.2f} < {TH[S['abstain']]:.2f}"); return [], True, "abstain", ranking
    return [ranking[0]], False, "retrieved", ranking

LABEL = {"CHO": "Carbohydrates (g)", "Protein": "Protein (g)", "Fat": "Fat (g)", "Fiber": "Fiber (g)", "Calorie": "Calories (kcal)", "GL": "Glycemic load"}
TIER_TXT = {"exact": "FoodBD plates with exactly these items", "superset": "FoodBD plates containing these items plus others", "model": "a model over all FoodBD plates (no plate has exactly this combination; rough)"}
IMAGE_HOOK = {}

def answer_query(q, S=FULL, use_llm=True, pool=None, feats=None):
    t0 = time.perf_counter(); tr = []
    f = feats if feats is not None else get_features(q)
    intent, cols, want_imgs = detect_intent(q)
    pred, abstained, kind, ranking = decide(f, S, S.get("thr", THR), tr)
    res = {"query": q, "intent": intent, "entities": [] if abstained else pred, "abstained": abstained, "kind": kind,
           "ranking": ranking, "trace": tr, "nutrition": None, "gen": None, "images": None}
    out = []
    if abstained:
        out.append("I could not find this dish in my Bangladeshi-food knowledge base, so I won't guess." if kind == "abstain" else
                   f"Did you mean {DOCS[pred[0]]['name']} or {DOCS[pred[1]]['name']}? Please tell me which one.")
    elif intent == "nutrition":
        items, proxies, missing = [], [], []
        for c in pred:
            m = FOODBD_MAP.get(c)
            if m is None: missing.append(c); continue
            items.append(m[0]); proxies += [f"{c} ~ {m[0]}"] if m[1] == "proxy" else []
        r = nutrition_for_items(items, pool) if items else None; res["nutrition"] = r
        if r:
            cols = cols or ["Calorie"]
            out.append(f"Nutrition for {' + '.join(items)} per plate, from {TIER_TXT[r['tier']]}" + (f" (n={r['n']})" if r["n"] else "") +
                       f" [confidence: {r['conf']}]: " + "; ".join(f"{LABEL[c]} ~{r['stats'][c][0]:.0f} (range {r['stats'][c][1]:.0f}-{r['stats'][c][2]:.0f})" for c in cols) + ".")
            if r.get("spread", 0) > 2: tr.append(f"WIDE_RANGE spread {r['spread']}x -> confidence lowered"); out.append("The range is very wide; treat it as a rough guide.")
            if proxies: tr.append("PROXY_MAPPING " + ", ".join(proxies)); out.append("Approximate: FoodBD has no separate class for " + ", ".join(proxies) + ".")
        per100 = [c for c in pred if c in NUTR100 and (c in missing or not r or FOODBD_MAP.get(c, ("", ""))[1] == "proxy")]
        for c in per100:                                   # FoodNetBD: dish-specific per-100 g values (better than a FoodBD proxy class)
            d = NUTR100[c]; tr.append(f"FOODNETBD_PER100G {c} ('{d['name']}', match={d['how']})")
            out.append(f"Per 100 g of {DOCS[c]['name']} (FoodNetBD entry '{d['name']}'): " + ", ".join((f"{k} {d[k]:.0f} kcal" if k == "energy" else f"{k} {d[k]:.1f} g") for k in ("energy", "protein", "fat", "carbohydrate", "fibre") if k in d) + ".")
        rest = [c for c in (missing or ([] if r else pred)) if c not in per100]
        if rest:
            tr.append(f"NO_NUTRITION_DATA {rest} -> description instead of invented numbers")
            out.append("I have no nutrition data for " + ", ".join(DOCS[c]["name"] if c in DOCS else c for c in rest) + ", so I won't give numbers.")
            for c in rest[:2]:
                if c in DOCS: out.append(DOC_TEXT[c])
    elif intent == "images" and "search" in IMAGE_HOOK:
        res["images"] = {"pictures": [], "verified": 0, "per_dish": {}}
        k_each = 5 if len(pred) == 1 else 3                    # "show me ruti and murgir mangsho" -> pictures of both
        for c in pred[:3]:
            s = IMAGE_HOOK["search"](c, k=k_each, tr=tr); n = s["verified"]; nm = DOCS[c]["name"] if c in DOCS else c
            res["images"]["pictures"] += s["pictures"]; res["images"]["verified"] += n; res["images"]["per_dish"][c] = n
            ident = (split_sents(DOC_TEXT[c]) or [nm])[0] if c in DOCS else c
            out.append(ident + (f" Here are {n} verified pictures of {nm}." if n else
                                f" I have no verified pictures of {nm} in my image gallery, so I am not showing unrelated photos."))
    else:
        conds_q = conditions_in(q); recipe_q = bool(re.search(ASPECTS[1][0], q, re.I))
        for c in pred[:2]:
            if c in DOCS and recipe_q:
                # v7: the KB stores ingredients / method lines, not step-by-step recipes; the 3B generator invented steps in v6 (korola vaji
                # "made from bamboo shoots", lau "served cold"). Answer extractively and say what is missing.
                sents = split_sents(DOC_TEXT[c]); facts = [s for s in sents if re.search(r"^Ingredients:|Cooking method:", s)]
                tr.append(f"RECIPE_QUESTION -> extractive ({len(facts)} ingredient/method lines in KB, no generation)")
                out.append(" ".join(sents[:1] + facts) + (" " if facts else " ") +
                           f"My knowledge base has no step-by-step recipe for {DOCS[c]['name']}, so I won't invent one.")
                continue
            if c in DOCS:
                ev = refine_evidence(c, q)
                if conds_q: tr.append("DIET_QUESTION -> description is extractive; suitability comes only from FoodNetBD tags")
                ans, ginfo = generate_verified(q, c, ev, tr, use_llm=use_llm and not conds_q, do_verify=S["verify"])
                if res["gen"] is None: res["gen"] = ginfo | {"evidence": ev, "canon": c, "answer": ans}
                out.append(ans)
            else:
                w = wiki_summary(c)
                tr.append(f"NO_KB_DOC({c}) -> " + ("Wikipedia fallback" if w else "no source, abstain on description"))
                out.append(w if w else f"I know '{c}' only as a FoodBD plate item and have no reliable description.")
        if want_imgs and "search" in IMAGE_HOOK and pred:
            res["images"] = IMAGE_HOOK["search"](pred[0], k=5, tr=tr)
    conds = conditions_in(q) if not abstained else []
    if conds:                                            # diet questions: only FoodNetBD's tag is stated, never an LLM opinion
        res["health"] = {}
        for c in pred[:2]:
            nm = DOCS[c]["name"] if c in DOCS else c; tags = HEALTH.get(c, {}).get("tags", {})
            for k in conds:
                if k in tags:
                    res["health"][(c, k)] = tags[k]; tr.append(f"DIET_TAG {c}/{k}={tags[k]} (FoodNetBD '{HEALTH[c]['name']}', match={HEALTH[c]['how']})")
                    out.append(f"FoodNetBD tags {nm} as '{tags[k]}' for {COND_NAME[k]}.")
                else:
                    tr.append(f"NO_DIET_TAG {c}/{k} -> no claim"); out.append(f"I have no FoodNetBD diet tag for {nm} and {COND_NAME[k]}, so I won't say whether it is suitable.")
        out.append("(Diet tags are general guidance from the FoodNetBD dataset, not medical advice; ask your doctor or dietitian.)")
    res["answer"] = "\n".join(out); res["latency_ms"] = 1000 * (time.perf_counter() - t0)
    return res

def show(res):
    print("Q:", res["query"]); print(res["answer"]); print("  entities:", res["entities"], "| kind:", res["kind"])
    print("  healing trace:", res["trace"] or "none needed")

for q in ["How many calories are in khichuri?", "rice and daal calories", "what is halim", "calories in kalavuna", "ফুচকা কি",
          "tell me about shorshe ilsh", "fiber in zxqowk", "what is the capital of France", "is khichuri ok for diabetes?",
          "ডায়াবেটিস রোগী কি বিরিয়ানি খেতে পারে?"]:
    show(answer_query(q)); print()

## 9. Image branch: recognition with healing, calibrated abstention, verified picture retrieval

* **Recognisers** — real mode: `tanziro` EfficientNetV2-S (841 Bangladeshi food classes) + CLIP **few-shot prototypes + prompts** over the classes that have gallery photos (v4 used zero-shot over all ~600 KB names: 9 % top-1). SMOKE: two light recognisers on synthetic images.
* **Heal on disagreement / low confidence**: add a third, retrieval-based vote — kNN over a labelled gallery (DeshiFoodBD train + any extra folder-per-class datasets found via `EXTRA_IMAGE_GLOBS`).
* **Calibrated abstention**: the fused confidence is first **Platt-scaled** on the calibration split (v4's raw fused scores were badly under-confident, ECE 0.47); τ = smallest calibrated confidence whose precision on the calibration split reaches `TARGET_IMG_PRECISION`.
* **Picture retrieval**: CLIP text→image candidates are kept only if their gallery label is the requested dish (label-consistency filter); shortfalls are reported, never padded.
* **Whole-plate photo → nutrition**: CLIP embedding ridge regression with conformal interval (real FoodBD photos only).
* All recogniser outputs are computed **once per image, in batches**, and cached.

In [ ]:
from PIL import Image, ImageDraw

def to_canon(label):
    e = link_entities(str(label).replace("_", " "))
    return max(e, key=lambda k: e[k][0]) if e else norm(label)

_LABEL_MAP = {}
def map_label(label):
    """Classifier label -> KB dish (v7). tanziro's 841 labels mix Bangla folder names ('কাচা_গোল্লা') and English; v6 mapped them with the
    query linker only, and recogniser A scored 4.6 % on FoodBD crops. Order: DeshiFoodBD Bangla names, exact alias, Bangla spelling fold,
    linker, fuzzy alias on the transliteration. Unmapped labels return None (their probability is not credited to any dish)."""
    if label in _LABEL_MAP: return _LABEL_MAP[label]
    l = str(label).replace("_", " ").strip(); n = norm(l); out = None
    if l in DESHI_BN: out = DESHI_BN[l]
    elif n in ALIAS2CANON: out = ALIAS2CANON[n]
    elif not n.isascii() and bn_fold(n) in ALIAS_FOLD: out = ALIAS_FOLD[bn_fold(n)]
    else:
        e = link_entities(l, phonetic=False)
        if e: out = max(e, key=lambda k: e[k][0])
        else:
            m = best_match(n if n.isascii() else bn2latin(n), ALIAS_LIST, 88)
            out = ALIAS2CANON[m[0]] if m else None
    _LABEL_MAP[label] = out if (out is None or out in DOCS) else None
    return _LABEL_MAP[label]

IMG_EXTS = (".jpg", ".jpeg", ".png", ".webp")
def folder_dataset(root, per_class=40):
    out = []
    for d in sorted(glob.glob(os.path.join(root, "**", ""), recursive=True)):
        files = [f for f in sorted(os.listdir(d)) if f.lower().endswith(IMG_EXTS)] if os.path.isdir(d) else []
        if len(files) >= 5:
            c = to_canon(os.path.basename(os.path.normpath(d)))
            if c in DOCS: out += [(os.path.join(d, f), c) for f in files[:per_class]]
    return out

IMG_SPLITS = {"gallery": [], "calib": [], "test": []}
desh_train = None if (SMOKE or DOMAIN != "bd") else find_first("**/food_data_english/train")
desh_test = None if (SMOKE or DOMAIN != "bd") else find_first("**/food_data_english/test")
OOD = "__ood__"                                            # label of photos whose dish is NOT in the KB: the right answer is to decline
if DOMAIN != "bd":
    # other domain: one folder per class. Food-101's official meta/test.txt is respected when present (nateraw/food was trained on the
    # train list, so it is tested only on unseen images); otherwise each class is split 50/50 with a fixed seed. Classes that are not
    # in the KB (incl. the held-out dishes) give must-decline photos and never enter the gallery.
    IMG_SOURCE = DOMAIN_IMG_SOURCE or DOMAIN; _rs = random.Random(7)
    _test_ids = set(l.strip() for l in open(DOMAIN_TEST_LIST)) if DOMAIN_TEST_LIST else set()
    G_N, C_N, T_N, C_OOD, T_OOD = 30, 8, 12 if PROFILE == "fast" else 25, 4, 8 if PROFILE == "fast" else 15
    for d in sorted(os.listdir(DOMAIN_IMG_ROOT)):
        p = os.path.join(DOMAIN_IMG_ROOT, d)
        if not os.path.isdir(p): continue
        files = sorted(f for f in os.listdir(p) if f.lower().endswith(IMG_EXTS))
        if len(files) < 4: continue
        if _test_ids:
            te_f = [f for f in files if f"{d}/{os.path.splitext(f)[0]}" in _test_ids]; tr_f = [f for f in files if f not in set(te_f)]
        else:
            _rs.shuffle(files); te_f, tr_f = files[: len(files) // 2], files[len(files) // 2:]
        canon = norm(d.replace("_", " ")); canon = canon if canon in DOCS else (map_label(d) if map_label(d) in DOCS else None)
        J = lambda fs: [os.path.join(p, f) for f in fs]
        if canon:
            IMG_SPLITS["gallery"] += [(x, canon) for x in J(tr_f[:G_N])]
            IMG_SPLITS["calib"] += [(x, canon) for x in J(te_f[:C_N])]; IMG_SPLITS["test"] += [(x, canon) for x in J(te_f[C_N:C_N + T_N])]
        else:
            IMG_SPLITS["calib"] += [(x, OOD) for x in J(te_f[:C_OOD])]; IMG_SPLITS["test"] += [(x, OOD) for x in J(te_f[C_OOD:C_OOD + T_OOD])]
    print(f"{DOMAIN}: {len({l for _, l in IMG_SPLITS['gallery']})} known dishes with photos | must-decline photos: "
          f"calib {sum(l == OOD for _, l in IMG_SPLITS['calib'])}, test {sum(l == OOD for _, l in IMG_SPLITS['test'])}")
elif desh_train and desh_test:
    IMG_SPLITS["gallery"] = folder_dataset(desh_train, 40)
    tst = folder_dataset(desh_test, 10**6); random.Random(1).shuffle(tst); h = len(tst) // 2
    IMG_SPLITS["calib"], IMG_SPLITS["test"] = tst[:min(h, CFG["IMG_CALIB_N"])], tst[h:h + CFG["IMG_TEST_N"]]
    IMG_SOURCE = "DeshiFoodBD"
    for g in EXTRA_IMAGE_GLOBS:                       # extra galleries enrich kNN healing and picture retrieval
        for root in sorted(set(find_dirs(g))):
            extra = folder_dataset(root, 30)
            if extra: IMG_SPLITS["gallery"] += extra; print(f"extra gallery {root}: {len(extra)} images, {len({c for _, c in extra})} dishes")
else:
    IMG_SOURCE = "synthetic"; out_dir = os.path.join(WORK, "synth_imgs_v2"); rng = np.random.RandomState(3)
    classes = ["biriyani", "khichuri", "haleem", "fuchka", "roshgolla", "roshmalai", "beguni", "porota", "kabab", "hilsha fish", "yogurt", "kala bhuna"]
    for ci, c in enumerate(classes):
        base = rng.randint(40, 215, size=3); n_blobs = 3 + ci % 5; os.makedirs(os.path.join(out_dir, c), exist_ok=True)
        for i in range(40):
            p = os.path.join(out_dir, c, f"{i}.png")
            if not os.path.exists(p):
                col = np.clip(base + rng.randint(-30, 31, size=3), 0, 255)
                img = Image.new("RGB", (64, 64), tuple(int(v) for v in col)); d = ImageDraw.Draw(img)
                for _ in range(n_blobs + rng.randint(-2, 3)):
                    x, y, r = rng.randint(0, 64), rng.randint(0, 64), rng.randint(3, 6 + ci % 6)
                    d.ellipse([x - r, y - r, x + r, y + r], fill=tuple(int(v) for v in np.clip(255 - col + rng.randint(-60, 60, 3), 0, 255)))
                arr = np.clip(np.asarray(img, dtype=float) + rng.normal(0, 14, (64, 64, 3)), 0, 255).astype(np.uint8)
                Image.fromarray(arr).save(p)
            IMG_SPLITS["gallery" if i < 20 else ("calib" if i < 30 else "test")].append((p, c))
IMG_SRC = {p: IMG_SOURCE for v in IMG_SPLITS.values() for p, _ in v}

def foodbd_crops(split, per_class, min_side=0.12):
    """FoodBD polygon instances as single-dish photos: the largest instance of each class per image, cropped to its box (+4 % margin).
    v5's image branch only knew DeshiFoodBD's 19 dishes, so a photo of ruti / bhat / murgir mangsho could never be named and
    'show me pictures of ruti' found nothing. Splits follow FoodBD's own train / valid / test folders, so test crops are unseen plates."""
    if INST.empty or INST["path"].isna().all(): return []
    item2c = {}
    for c, (it, how) in FOODBD_MAP.items():
        if how == "exact" and c in DOCS: item2c.setdefault(it, c)
    d = INST[(INST["split"] == split) & INST["cls"].isin(item2c) & INST["path"].notna()]
    d = d[d["bbox"].map(lambda b: b is not None and b[2] - b[0] >= min_side and b[3] - b[1] >= min_side)]
    out = []
    for it, g in d.groupby("cls"):
        g = g.sort_values("area", ascending=False).drop_duplicates("stem").head(per_class)
        out += [(f"{r.path}#{r.bbox[0]:.4f},{r.bbox[1]:.4f},{r.bbox[2]:.4f},{r.bbox[3]:.4f}#{r.Index}", item2c[it]) for r in g.itertuples()]
    return out

CROPS = {"gallery": foodbd_crops("train", 30), "calib": foodbd_crops("valid", 12), "test": foodbd_crops("test", 20)}
for k, v in CROPS.items():
    IMG_SPLITS[k] = IMG_SPLITS[k] + v; IMG_SRC.update({p: "FoodBD crop" for p, _ in v})
print(IMG_SOURCE, "+ FoodBD crops" if DOMAIN == "bd" else "", {k: len(v) for k, v in IMG_SPLITS.items()}, "| FoodBD crops:", {k: len(v) for k, v in CROPS.items()},
      "| classes:", len({c for _, c in IMG_SPLITS["gallery"]}), f"(v5: {len({c for p, c in IMG_SPLITS['gallery'] if IMG_SRC.get(p) != 'FoodBD crop'})})")

CLF, CLIP = None, None
with section("load image models"):
    if not SMOKE and HAS_TF and HAS_TORCH:
        from transformers import pipeline, CLIPModel, CLIPProcessor
        CLF = try_load("image-classifier:" + CFG["CLF_MODEL"], lambda: pipeline("image-classification", model=model_src(CFG["CLF_MODEL"]), top_k=5, device=0 if GPU else -1, **({"torch_dtype": torch.float16} if GPU else {})))
        if CLF is not None:
            _labs = list(CLF.model.config.id2label.values()); _mp = {l: map_label(l) for l in _labs}
            _um = [l for l, c in _mp.items() if c is None]
            print(f"recogniser A: {len(_labs)} labels, {len(_labs) - len(_um)} mapped to KB dishes ({1 - len(_um) / max(len(_labs), 1):.0%}); "
                  f"unmapped e.g. {_um[:12]}")
        CLIP = try_load("clip:" + CFG["CLIP_MODEL"], lambda: (CLIPModel.from_pretrained(model_src(CFG["CLIP_MODEL"])).to(DEVICE).eval(), CLIPProcessor.from_pretrained(model_src(CFG["CLIP_MODEL"]))))
    else:
        MODEL_STATUS["image-classifier:" + CFG["CLF_MODEL"]] = "FALLBACK - logistic regression on 8x8 pixels (synthetic)"
        MODEL_STATUS["clip:" + CFG["CLIP_MODEL"]] = "FALLBACK - colour-histogram features (synthetic)"

def _feat(o):
    if torch.is_tensor(o): return o
    po = getattr(o, "pooler_output", None); return po if po is not None else o[0]
def _open(p, size=448):
    """Open an image; 'path#x0,y0,x1,y1' (normalised box) returns that crop (FoodBD polygon instances)."""
    crop, pid = None, None
    if "#" in p:
        parts = p.split("#"); p = parts[0]; crop = [float(v) for v in parts[1].split(",")]; pid = int(parts[2]) if len(parts) > 2 else None
    im = Image.open(p)
    if im.format == "JPEG": im.draft("RGB", (size * (2 if crop else 1), size * (2 if crop else 1)))   # reduced-size JPEG decoding: ~10x faster
    im = im.convert("RGB")
    if crop:
        W, H = im.size; x0, y0, x1, y1 = crop; m = 0.04
        box = (int(max(0, x0 - m) * W), int(max(0, y0 - m) * H), int(min(1, x1 + m) * W), int(min(1, y1 + m) * H))
        if pid is not None and CFG.get("MASK_CROPS", True) and "poly" in INST.columns:
            # v7: grey out everything outside the polygon (v6 box crops of rice / shak / vegetable also showed the neighbouring dishes)
            mask = Image.new("L", im.size, 0); ImageDraw.Draw(mask).polygon([(float(x) * W, float(y) * H) for x, y in INST.at[pid, "poly"]], fill=255)
            im = Image.composite(im, Image.new("RGB", im.size, (128, 128, 128)), mask)
        im = im.crop(box)
    im.thumbnail((size * 2, size * 2)); return im
from concurrent.futures import ThreadPoolExecutor
_POOL = ThreadPoolExecutor(8)
def open_many(items): return list(_POOL.map(lambda p: _open(p) if isinstance(p, str) else p, items))
def clip_img(items, bs=64):
    m, proc = CLIP; outs = []
    for i in range(0, len(items), bs):
        ims = open_many(items[i:i + bs])
        with torch.no_grad(), amp(): f = _feat(m.get_image_features(**proc(images=ims, return_tensors="pt").to(m.device)))
        outs.append(torch.nn.functional.normalize(f.float(), dim=-1).cpu().numpy())
    return np.concatenate(outs)
def clip_txt(texts, bs=256):
    m, proc = CLIP; outs = []
    for i in range(0, len(texts), bs):
        with torch.no_grad(), amp(): f = _feat(m.get_text_features(**proc(text=texts[i:i + bs], return_tensors="pt", padding=True, truncation=True, max_length=77).to(m.device)))
        outs.append(torch.nn.functional.normalize(f.float(), dim=-1).cpu().numpy())
    return np.concatenate(outs)

def _pix(p): return np.asarray(_open(p).resize((8, 8)), dtype=float).ravel() / 255
def _hist(p):
    hsv = np.asarray(_open(p).convert("HSV").resize((32, 32))).reshape(-1, 3)
    h, _ = np.histogramdd(hsv, bins=(8, 4, 4), range=((0, 256), (0, 256), (0, 256))); h = h.ravel(); return h / np.linalg.norm(h)
def img_embed(paths):
    if CLIP is not None: return clip_img(paths)
    X = np.array([np.r_[_hist(p), 0.5 * _pix(p)] for p in paths]); return X / np.linalg.norm(X, axis=1, keepdims=True)

with section("gallery embeddings"):
    G_PATHS = [p for p, _ in IMG_SPLITS["gallery"]]; G_LAB = np.array([c for _, c in IMG_SPLITS["gallery"]])
    G_EMB = cached("gallery_emb", lambda: img_embed(G_PATHS), (len(G_PATHS), G_PATHS[:3], G_PATHS[-3:], CFG["CLIP_MODEL"], CLIP is not None, "v7", CFG.get("MASK_CROPS", True)))
    IMG_CLASSES = sorted(set(G_LAB))
    if CLF is None:
        _rA = LogisticRegression(C=0.5, max_iter=2000).fit(np.array([_pix(p) for p in G_PATHS]), G_LAB)
        _protB = {c: np.mean([_hist(p) for p, l in IMG_SPLITS["gallery"] if l == c], 0) for c in IMG_CLASSES}
    # recogniser B (v7) = multinomial logistic-regression probe on the frozen image embedding, trained on the labelled gallery
    # (DeshiFoodBD train + extras + masked FoodBD train crops). v6 used class prototypes + text prompts: 0.65 top-1 on FoodBD crops.
    PROBE = LogisticRegression(C=CFG.get("PROBE_C", 10.0), max_iter=4000, class_weight="balanced").fit(G_EMB, G_LAB)
    PROTO = np.stack([G_EMB[G_LAB == c].mean(0) for c in IMG_CLASSES]); PROTO /= np.linalg.norm(PROTO, axis=1, keepdims=True)

P_PATHS, P_LAB, P_EMB = G_PATHS, G_LAB, G_EMB          # picture gallery = recognition gallery (DeshiFoodBD + extras + FoodBD train crops)
print(f"picture/recognition gallery: {len(P_PATHS)} images covering {len(set(P_LAB))} dishes")

_IMG = {}
def precompute_images(paths, bs=32):
    """Recogniser A, recogniser B, embedding, kNN — once per image, batched."""
    new = [p for p in dict.fromkeys(paths) if p not in _IMG]
    for i in range(0, len(new), 256):
        chunk = new[i:i + 256]; _t0 = time.perf_counter(); E = img_embed(chunk)
        if CLF is not None:
            outs = CLF(open_many(chunk), batch_size=bs)
            PA = []
            for o in outs:
                d = {}
                for r in o:
                    c = map_label(r["label"])
                    if c: d[c] = d.get(c, 0) + float(r["score"])
                d = d or {"_unmapped": 1.0}
                PA.append(d)
        else:
            PA = [dict(zip(_rA.classes_, pr)) for pr in _rA.predict_proba(np.array([_pix(p) for p in chunk]))]
        PB = PROBE.predict_proba(E)
        for p, e, pa, pr in zip(chunk, E, PA, PB):
            idx = np.argsort(-pr)[:5]; pb = {str(PROBE.classes_[j]): float(pr[j]) for j in idx}
            sims = G_EMB @ e; nn = np.argsort(-sims)
            nn = [j for j in nn[:11] if G_PATHS[j] != p][:10]          # never vote with the query image itself
            pk = collections.Counter()
            for j in nn: pk[G_LAB[j]] += max(float(sims[j]), 0)
            tot = sum(pk.values()) or 1.0
            _IMG[p] = {"emb": e, "A": pa, "B": pb, "K": {c: v / tot for c, v in pk.items()}}
        for p in chunk: _IMG[p]["ms"] = 1000 * (time.perf_counter() - _t0) / len(chunk)       # batched recogniser time per image

IMG_TAU, IMG_CAL = 0.5, (1.0, 0.0)                     # Platt scaling (a, b) of the fused confidence, fitted on the calibration split
W_FUSE, W_HEAL, IMG_HEAL = (0.5, 0.5), (1 / 3, 1 / 3, 1 / 3), True   # replaced by weights learned on the calibration split
def _logit(p): p = np.clip(p, 1e-4, 1 - 1e-4); return np.log(p / (1 - p))
def calibrated(conf): a, b = IMG_CAL; return float(1 / (1 + np.exp(-(a * _logit(conf) + b))))
def _combine(dicts, w):
    f = collections.Counter()
    for d, wi in zip(dicts, w):
        for c, v in d.items():
            if c != "_unmapped": f[c] += wi * v
    return f

CAT_TAU = None                                          # category back-off threshold, calibrated below (None = disabled)
def dish_category(c):
    cat = norm(DOCS[c].get("category", "")) if c in DOCS else ""
    return cat or None
def category_vote(fused):
    """Sum the fused dish probabilities by KB category ('meat curry', 'vegetable', 'sweet' ...)."""
    cv = collections.Counter(); members = collections.defaultdict(list)
    for c, v in fused.items():
        k = dish_category(c)
        if k: cv[k] += v; members[k].append((c, v))
    if not cv: return None
    k, s = cv.most_common(1)[0]; tot = sum(fused.values()) or 1.0
    return k, s / tot, [c for c, _ in sorted(members[k], key=lambda x: -x[1])[:3]]
def classify_image(path, heal=None, agree_conf=0.5, w_fuse=None, w_heal=None):
    heal = IMG_HEAL if heal is None else heal
    precompute_images([path]); d = _IMG[path]; tr = []; pa, pb = d["A"], d["B"]
    fused = _combine((pa, pb), w_fuse or W_FUSE)
    top_a, top_b = max(pa, key=pa.get), max(pb, key=pb.get); best, conf = fused.most_common(1)[0]
    if heal and (top_a != top_b or conf < agree_conf):
        pk = d["K"]; tr.append(f"RECOGNISERS_DISAGREE (A={top_a}, B={top_b}, fused={conf:.2f}) -> add gallery kNN vote")
        fused = _combine((pa, pb, pk), w_heal or W_HEAL)
        best, conf = fused.most_common(1)[0]; tr.append(f"HEALED -> {best} ({conf:.2f})")
    raw_conf = conf; conf = calibrated(conf)
    uncertain = conf < IMG_TAU; cat = None
    if uncertain:
        tr.append(f"IMAGE_UNCERTAIN {conf:.2f} < tau {IMG_TAU:.2f} -> do not name a dish")
        cv = category_vote(fused)
        if cv and CAT_TAU is not None and cv[1] >= CAT_TAU:
            cat = cv; tr.append(f"CATEGORY_BACKOFF '{cv[0]}' ({cv[1]:.2f} >= {CAT_TAU:.2f}) -> name the kind of dish, not the dish")
    return {"top": str(best), "conf": float(conf), "ranked": [(str(c), float(v)) for c, v in fused.most_common(5)], "A": top_a, "B": top_b,
            "uncertain": uncertain, "category": cat, "trace": tr, "emb": d["emb"], "raw_conf": float(raw_conf)}

def calibrate_tau(results, labels, target):
    conf = np.array([r["conf"] for r in results]); ok = np.array([r["top"] == l for r, l in zip(results, labels)])
    for t in np.unique(np.round(conf, 3)):
        m = conf >= t
        if m.sum() >= 10 and ok[m].mean() >= target: return float(t)
    return float(conf.max())

with section("recognise calibration + test images"):
    precompute_images([p for p, _ in IMG_SPLITS["calib"] + IMG_SPLITS["test"]])
    _cp, _cl = [p for p, _ in IMG_SPLITS["calib"]], [l for _, l in IMG_SPLITS["calib"]]
    def _cal_acc(**kw):
        rs = [classify_image(p, **kw) for p in _cp]; ok = np.array([r["top"] == l for r, l in zip(rs, _cl)])
        return ok.mean(), np.mean([r["raw_conf"] * o for r, o in zip(rs, ok)])        # accuracy, then confidence-on-correct as tie-break
    # v5 averaged A, B and kNN equally: with A at 97 % and B at 83 %, "healing" flipped correct A answers (0.983 fused -> 0.973 healed).
    grid = np.round(np.arange(0, 1.01, 0.1), 2)
    grid = [float(x) for x in grid]
    W_FUSE = max(((a, round(1 - a, 2)) for a in grid), key=lambda w: _cal_acc(heal=False, w_fuse=w))
    W_HEAL = max(((a, b, round(1 - a - b, 2)) for a in grid for b in grid if a + b <= 1.0001), key=lambda w: _cal_acc(heal=True, w_heal=w))
    acc_f, acc_h = _cal_acc(heal=False)[0], _cal_acc(heal=True)[0]; IMG_HEAL = acc_h > acc_f
    print(f"fusion weights A/B = {W_FUSE} (calib acc {acc_f:.3f}) | heal weights A/B/kNN = {W_HEAL} (calib acc {acc_h:.3f}) -> healing {'ON' if IMG_HEAL else 'OFF (does not help on calibration data)'}")
    calib = [classify_image(p) for p in _cp]
    _ok = np.array([r["top"] == l for r, l in zip(calib, _cl)])
    if 0 < _ok.sum() < len(_ok):
        _lr = LogisticRegression(C=10.0).fit(_logit(np.array([r["raw_conf"] for r in calib]))[:, None], _ok)
        IMG_CAL = (float(_lr.coef_[0, 0]), float(_lr.intercept_[0]))
    calib = [classify_image(p) for p in _cp]
    IMG_TAU = calibrate_tau(calib, _cl, CFG["TARGET_IMG_PRECISION"])
    # v7 category back-off: when the dish is uncertain, the KIND of dish is often clear (your v6 photo of a vegetable curry got a flat
    # refusal; a meat curry was called "chicken + beef"). Threshold = smallest category share whose precision on the UNCERTAIN calibration
    # images reaches CAT_PRECISION.
    _unc = [(r, l) for r, l in zip(calib, _cl) if r["conf"] < IMG_TAU and dish_category(l)]
    _cv = [(category_vote(collections.Counter(dict(r["ranked"]))), dish_category(l)) for r, l in _unc]
    _cv = [(v[1], v[0] == t) for v, t in _cv if v]
    for t in sorted({round(s, 3) for s, _ in _cv}):
        sel_ = [ok for s, ok in _cv if s >= t]
        if len(sel_) >= 10 and np.mean(sel_) >= CFG.get("CAT_PRECISION", 0.85): CAT_TAU = float(t); break
    print(f"category back-off: {len(_cv)} uncertain calibration images with a category | threshold "
          + (f"{CAT_TAU:.2f} (precision >= {CFG.get('CAT_PRECISION', 0.85):.0%})" if CAT_TAU is not None else "none reaches the target -> disabled"))
print(f"Platt calibration a={IMG_CAL[0]:.2f} b={IMG_CAL[1]:.2f} | calibration-split accuracy {_ok.mean():.3f}")
print(f"tau calibrated for {CFG['TARGET_IMG_PRECISION']:.0%} precision on {len(calib)} calibration images: {IMG_TAU:.3f}")

def image_search(canon, k=5, tr=None, verify_labels=True):
    if CLIP is not None:
        q = clip_txt([f"a photo of {DOCS[canon]['name'] if canon in DOCS else canon}, a Bangladeshi food"])[0]; order = np.argsort(-(P_EMB @ q))
    else:
        cen = P_EMB[P_LAB == canon].mean(0) if (P_LAB == canon).any() else P_EMB.mean(0); order = np.argsort(-(P_EMB @ cen))
    raw = [(P_PATHS[i], str(P_LAB[i])) for i in order[:k]]
    if not verify_labels: return {"pictures": raw, "raw": raw, "verified": None}
    ok = [(P_PATHS[i], str(P_LAB[i])) for i in order if P_LAB[i] == canon][:k]
    if tr is not None:
        bad = sum(l != canon for _, l in raw)
        if bad: tr.append(f"PICTURE_LABEL_MISMATCH {bad}/{k} raw results not '{canon}' -> filtered")
        if len(ok) < k: tr.append(f"ONLY_{len(ok)}_VERIFIED_PICTURES for '{canon}' (gallery has no more)")
    return {"pictures": ok, "raw": raw, "verified": len(ok)}
IMAGE_HOOK["search"] = image_search

# ---- Whole-plate item detector (multi-label). FoodBD photos are meals with several dishes; a single-dish recogniser cannot name
# "rice + daal + fish" (v5 answered "I'm not confident which dish this is" for FoodBD test plates). One logistic head per FoodBD class
# on the photo embedding, trained on ALL annotated FoodBD photos of the train folder (not only the 1,837 with nutrition), per-class
# thresholds tuned for F1 on the valid folder, reported on the test folder.
PLATE_DET, ALL_EMB_BY_STEM, ROUTER_T = None, {}, 0.5
if not INST.empty and INST["path"].notna().any():
    with section("plate item detector (multi-label, all annotated FoodBD photos)"):
        _st = INST.dropna(subset=["path"]).drop_duplicates("stem")[["stem", "split", "path"]].reset_index(drop=True)
        _AE = cached("foodbd_all_emb", lambda: img_embed(_st["path"].tolist()), (len(_st), _st["path"].iloc[0], CFG["CLIP_MODEL"], CLIP is not None, "v7", CFG.get("MASK_CROPS", True)))
        ALL_EMB_BY_STEM = dict(zip(_st["stem"], _AE))
        _items = INST.groupby("stem")["cls"].agg(set)
        _sp = {x: (_st["split"] == x).values for x in ("train", "valid", "test")}
        _Yall = np.array([[c in _items[s] for c in NAMES] for s in _st["stem"]], dtype=int)
        DET_CLASSES = [c for j, c in enumerate(NAMES) if _Yall[_sp["train"], j].sum() >= 8 and _Yall[_sp["train"], j].sum() < _sp["train"].sum()]
        _jj = [NAMES.index(c) for c in DET_CLASSES]; _Yd = _Yall[:, _jj]
        _heads = [LogisticRegression(C=2.0, max_iter=3000, class_weight="balanced").fit(_AE[_sp["train"]], _Yd[_sp["train"], j]) for j in range(len(_jj))]
        def _det_proba(E): return np.stack([h.predict_proba(E)[:, 1] for h in _heads], 1)
        _Pv, _Pt = _det_proba(_AE[_sp["valid"]]), _det_proba(_AE[_sp["test"]])
        DET_THR = np.full(len(_jj), 0.5)
        for j in range(len(_jj)):                                  # per-class threshold: best F1 on valid (needs >= 3 positives there)
            yv = _Yd[_sp["valid"], j]
            if yv.sum() < 3: continue
            f1 = lambda t: 2 * ((_Pv[:, j] >= t) & (yv == 1)).sum() / max((_Pv[:, j] >= t).sum() + yv.sum(), 1)
            DET_THR[j] = max(np.arange(0.2, 0.91, 0.05), key=f1)
        _pred, _yt = _Pt >= DET_THR, _Yd[_sp["test"]] == 1
        _tp, _fp, _fn = (_pred & _yt).sum(0), (_pred & ~_yt).sum(0), (~_pred & _yt).sum(0)
        _P, _R = _tp.sum() / max(_tp.sum() + _fp.sum(), 1), _tp.sum() / max(_tp.sum() + _fn.sum(), 1)
        per = pd.DataFrame({"item": DET_CLASSES, "n_test": _yt.sum(0), "precision": _tp / np.maximum(_tp + _fp, 1), "recall": _tp / np.maximum(_tp + _fn, 1)})
        per["f1"] = 2 * per.precision * per.recall / np.maximum(per.precision + per.recall, 1e-9)
        PLATE_DET = {"classes": len(DET_CLASSES), "images": {k: int(v.sum()) for k, v in _sp.items()}, "micro_precision": float(_P), "micro_recall": float(_R),
                     "micro_f1": float(2 * _P * _R / max(_P + _R, 1e-9)), "macro_f1": float(per.loc[per.n_test > 0, "f1"].mean()),
                     "exact_set_match": float((_pred == _yt).all(1).mean()), "per_item": per}
    print(f"plate item detector: {PLATE_DET['classes']} classes | images {PLATE_DET['images']} | TEST micro P/R/F1 "
          f"{PLATE_DET['micro_precision']:.3f}/{PLATE_DET['micro_recall']:.3f}/{PLATE_DET['micro_f1']:.3f} | macro F1 {PLATE_DET['macro_f1']:.3f} | "
          f"exact item set {PLATE_DET['exact_set_match']:.3f}")
    print(per.sort_values("n_test", ascending=False).head(15).round(3).to_string(index=False))
    # router: is this photo a multi-dish meal plate or a single dish? (the detector alone also fires on close-ups of one dish)
    _multi = np.array([len(_items[s]) >= 2 for s in _st["stem"]])
    _cal_emb = np.stack([_IMG[p]["emb"] for p, _ in IMG_SPLITS["calib"]]) if IMG_SPLITS["calib"] else np.zeros((0, _AE.shape[1]))
    _Xr = np.concatenate([_AE[_sp["train"] & _multi], G_EMB]); _yr = np.r_[np.ones((_sp["train"] & _multi).sum()), np.zeros(len(G_EMB))]
    ROUTER = LogisticRegression(C=1.0, max_iter=3000, class_weight="balanced").fit(_Xr, _yr)
    _Xv = np.concatenate([_AE[_sp["test"] & _multi], np.stack([_IMG[p]["emb"] for p, _ in IMG_SPLITS["test"]])])
    _yv = np.r_[np.ones((_sp["test"] & _multi).sum()), np.zeros(len(IMG_SPLITS["test"]))]
    # v7: threshold tuned on VALID multi-item photos vs calibration single-dish photos (v6 used 0.5: a single meat-curry photo at p=0.66
    # was answered as a "chicken + beef" plate)
    _Xc = np.concatenate([_AE[_sp["valid"] & _multi], _cal_emb]); _yc = np.r_[np.ones((_sp["valid"] & _multi).sum()), np.zeros(len(_cal_emb))]
    _pc = ROUTER.predict_proba(_Xc)[:, 1]
    ROUTER_T = max(np.arange(0.3, 0.96, 0.05), key=lambda t: 0.5 * ((_pc >= t)[_yc == 1].mean() + (_pc < t)[_yc == 0].mean()) if len(_yc) else 0)
    PLATE_DET["router_threshold"] = float(ROUTER_T)
    PLATE_DET["router_test_acc"] = float(((ROUTER.predict_proba(_Xv)[:, 1] >= ROUTER_T) == _yv).mean())
    print(f"plate-vs-single-dish router: threshold {ROUTER_T:.2f} | test accuracy {PLATE_DET['router_test_acc']:.3f} "
          f"({int(_yv.sum())} multi-item FoodBD test photos vs {int((1 - _yv).sum())} single-dish test photos)")

def is_plate_photo(path):
    if PLATE_DET is None: return 0.0
    precompute_images([path]); return float(ROUTER.predict_proba(_IMG[path]["emb"][None])[0, 1])

def detect_plate_items(path):
    """-> [(FoodBD item, KB dish, probability)] above the per-class thresholds, most likely first."""
    if PLATE_DET is None: return []
    e = img_embed([path]) if path not in _IMG else _IMG[path]["emb"][None]
    pr = _det_proba(e)[0]; it2c = {it: c for c, (it, how) in FOODBD_MAP.items() if how == "exact" and c in DOCS}
    return [(DET_CLASSES[j], it2c.get(DET_CLASSES[j], norm(DET_CLASSES[j])), float(pr[j])) for j in np.argsort(-pr) if pr[j] >= DET_THR[j]]

PLATE = None
if CLIP is not None and REAL_FOODBD:
    with section("plate photos -> nutrition regression"):
        _pp = [f"{FOODBD_ROOT}/{r.split}/images/{r.image}" for r in plates.itertuples()]; _ok = np.array([os.path.exists(p) for p in _pp])
        _stems = [os.path.splitext(r.image)[0] for r in plates.itertuples()]
        _ok = _ok & np.array([s in ALL_EMB_BY_STEM for s in _stems]) if ALL_EMB_BY_STEM else _ok          # reuse the detector's embeddings
        PLATE_EMB = (np.stack([ALL_EMB_BY_STEM[s] for s, o in zip(_stems, _ok) if o]) if ALL_EMB_BY_STEM else
                     cached("plate_emb", lambda: clip_img([p for p, o in zip(_pp, _ok) if o]), (int(_ok.sum()), CFG["CLIP_MODEL"])))
        _PE = np.zeros((len(plates), PLATE_EMB.shape[1]), dtype=np.float32); _PE[_ok] = PLATE_EMB
        _e2e = Ridge(alpha=10.0).fit(_PE[tr & _ok], Y[tr & _ok])
        _Qimg = conformal_q(np.abs(Y[va & _ok] - _e2e.predict(_PE[va & _ok])), 0.2)
        _m = te & _ok; kc = NUTR.index("Calorie")
        PLATE = {"mae_kcal": float(np.abs(Y[_m, kc] - _e2e.predict(_PE[_m])[:, kc]).mean()), "mean_baseline_kcal": float(np.abs(Y[_m, kc] - Y[tr, kc].mean()).mean()),
                 "coverage80": float((np.abs(Y[_m] - _e2e.predict(_PE[_m])) <= _Qimg)[:, kc].mean())}
    print("Photo -> plate nutrition (CLIP ridge, test plates):", {k: round(v, 3) for k, v in PLATE.items()})
else:
    print("Photo -> plate nutrition needs CLIP + real FoodBD photos (skipped in SMOKE mode).")

# Condition-aware (Mondrian) conformal intervals from FoodBD_Meta_data.csv: if the error depends on a photo condition (e.g. lighting),
# one global interval is too wide for easy photos and too narrow for hard ones. Per-condition intervals are calibrated on validation plates;
# for a new photo (no metadata) the condition is predicted from its CLIP embedding, used only if that predictor is reliable on validation.
PLATE_COND = None
if PLATE is not None and META_COND:
    _best = None
    for c in META_COND:
        g = plates[c].fillna("unknown").astype(str).values; mv = va & _ok
        res = np.abs(Y[mv, kc] - _e2e.predict(_PE[mv])[:, kc]); gv = g[mv]
        groups = [x for x in np.unique(gv) if (gv == x).sum() >= 15]
        if len(groups) < 2: continue
        means = [res[gv == x].mean() for x in groups]; spread = max(means) / max(min(means), 1e-6)
        if _best is None or spread > _best[0]: _best = (spread, c, groups)
    if _best:
        spread, c, groups = _best; g = plates[c].fillna("unknown").astype(str).values
        Qg = {x: conformal_q(np.abs(Y[va & _ok & (g == x)] - _e2e.predict(_PE[va & _ok & (g == x)])), 0.2) for x in groups}
        m_tr, m_va, m_te = [(s & _ok & np.isin(g, groups)) for s in (tr, va, te)]
        _cclf = LogisticRegression(C=1.0, max_iter=3000).fit(_PE[m_tr], g[m_tr]); cacc = float(_cclf.score(_PE[m_va], g[m_va]))
        err = np.abs(Y[m_te] - _e2e.predict(_PE[m_te])); gt = g[m_te]; gp = _cclf.predict(_PE[m_te])
        qt, qp = np.stack([Qg[x] for x in gt]), np.stack([Qg[x] for x in gp])
        tab = pd.DataFrame([{"group": x, "n_test": int((gt == x).sum()), "MAE_kcal": err[gt == x, kc].mean(), "halfwidth_kcal": Qg[x][kc],
                             "coverage_global": (err[gt == x] <= _Qimg)[:, kc].mean(), "coverage_mondrian": (err[gt == x] <= qt[gt == x])[:, kc].mean()} for x in groups]).round(3)
        PLATE_COND = {"column": c[5:], "groups": groups, "Q": Qg, "clf_acc_valid": cacc, "use_predicted": cacc >= 0.6, "table": tab,
                      "coverage_mondrian_predicted_group": float((err <= qp)[:, kc].mean()), "error_spread_ratio": round(float(spread), 2)}
        print(f"\nMondrian intervals by '{c[5:]}' (largest error spread on validation, x{spread:.2f}); photo->condition predictor valid acc {cacc:.2f}"
              f" -> {'used for new photos' if cacc >= 0.6 else 'not reliable: new photos keep the global interval'}")
        print(tab.to_string(index=False))
        PLATE["mondrian"] = {k: v for k, v in PLATE_COND.items() if k in ("column", "clf_acc_valid", "coverage_mondrian_predicted_group", "error_spread_ratio")}

def plate_estimate(path):
    if PLATE is None: return None
    e = clip_img([path]); p = _e2e.predict(e)[0]; q, cond = _Qimg, None
    if PLATE_COND and PLATE_COND["use_predicted"]:
        cond = str(_cclf.predict(e)[0]); q = PLATE_COND["Q"][cond]
    out = {c: (float(v), float(max(0, v - qq)), float(v + qq)) for c, v, qq in zip(NUTR, p, q)}
    out["_cond"] = (PLATE_COND["column"], cond) if cond else None
    return out

def show_images(pairs, title=""):
    import matplotlib.pyplot as plt
    if not pairs: print("(no pictures)"); return
    fig, axs = plt.subplots(1, len(pairs), figsize=(2.4 * len(pairs), 2.6))
    for ax, (p, l) in zip(np.atleast_1d(axs), pairs): ax.imshow(_open(p)); ax.set_title(l, fontsize=8); ax.axis("off")
    fig.suptitle(title, fontsize=9); plt.tight_layout(); plt.show()

def multimodal_answer(question=None, image=None, plate=False):
    rep = {"trace": []}; q = question or "what is this dish?"
    plate = plate or (PLATE is not None and detect_intent(q)[0] == "nutrition")   # nutrition question + photo -> whole-plate estimate too
    if image and plate:
        est = plate_estimate(image)
        if est:
            rep["plate_estimate"] = est; rep["trace"].append("PLATE_PHOTO -> CLIP regression estimate with conformal 80% interval")
            if est["_cond"]: rep["trace"].append(f"CONDITION {est['_cond'][0]}={est['_cond'][1]} (predicted from the photo) -> condition-specific interval")
    if image:
        c = classify_image(image); rep["trace"] += c["trace"]; rep["image_candidates"] = [(d, round(s, 2)) for d, s in c["ranked"][:3]]
        items = detect_plate_items(image)
        if items: rep["plate_items"] = [(it, round(p, 2)) for it, _, p in items]
        pp = is_plate_photo(image); rep["plate_prob"] = round(pp, 2)
        if not link_entities(q) and len(items) >= 2 and pp >= ROUTER_T:
            # a meal plate: name every detected item instead of forcing one dish label (or refusing)
            rep["trace"].append(f"MEAL_PLATE (router p={pp:.2f}) -> PLATE_ITEMS_DETECTED {rep['plate_items']} -> answer for the whole plate, "
                                f"not one dish label")
            dishes = list(dict.fromkeys(cn for _, cn, _ in items if cn in DOCS))
            names_ = ", ".join(f"{DOCS[cn]['name']} ({p:.2f})" if cn in DOCS else f"{it} ({p:.2f})" for it, cn, p in items)
            if detect_intent(q)[0] == "nutrition" and dishes:
                res = answer_query(q + " " + " and ".join(DOCS[d]["name"] for d in dishes[:4])); rep["trace"] += res["trace"]
                res["answer"] = f"This looks like a meal plate with: {names_}.\n" + res["answer"]
            else:
                res = {"answer": f"This looks like a meal plate with: {names_}. " + " ".join((split_sents(DOC_TEXT[d]) or [DOCS[d]['name']])[0] for d in dishes[:4]),
                       "entities": dishes, "trace": []}
            rep["text"] = res
            if rep.get("plate_estimate"):
                e = rep["plate_estimate"]; cols = detect_intent(q)[1] or ["Calorie"]
                res["answer"] += ("\nWhole-plate estimate from the photo (CLIP regression on FoodBD, 80% interval): " +
                                  "; ".join(f"{LABEL[c_]} ~{e[c_][0]:.0f} ({e[c_][1]:.0f}-{e[c_][2]:.0f})" for c_ in cols) + ".")
            sims = P_EMB @ c["emb"]; rep["similar"] = [(P_PATHS[i], P_LAB[i]) for i in np.argsort(-sims)[:5]]
            return rep
        if not link_entities(q):
            if c["uncertain"]:
                if c.get("category"):
                    k, s, mem = c["category"]
                    msg = (f"I'm not sure of the exact dish, but it looks like a {k} dish (category confidence {s:.2f}) - most likely "
                           + ", ".join(DOCS[m]["name"] for m in mem if m in DOCS) + ". Tell me the name and I'll look it up. Here are similar pictures.")
                else:
                    msg = (f"I'm not confident which dish this is (best {c['top']} at {c['conf']:.2f} < {IMG_TAU:.2f}), so I won't guess. "
                           f"Candidates: {rep['image_candidates']}. Here are the most similar pictures I have.")
                rep["text"] = {"answer": msg, "trace": []}
                if rep.get("plate_estimate"):         # the whole-plate estimate does not need the dish name, so it is still given
                    e = rep["plate_estimate"]; cols = detect_intent(q)[1] or ["Calorie"]
                    rep["text"]["answer"] += ("\nWhole-plate estimate from the photo (CLIP regression on FoodBD, 80% interval): " +
                                              "; ".join(f"{LABEL[c_]} ~{e[c_][0]:.0f} ({e[c_][1]:.0f}-{e[c_][2]:.0f})" for c_ in cols) + ".")
                sims = P_EMB @ c["emb"]; rep["similar"] = [(P_PATHS[i], P_LAB[i]) for i in np.argsort(-sims)[:5]]
                return rep
            q = f"{q} {c['top']}"; rep["trace"].append(f"Question had no dish -> using image prediction '{c['top']}'")
    res = answer_query(q); rep["text"] = res; rep["trace"] += res["trace"]
    if rep.get("plate_estimate"):
        e = rep["plate_estimate"]; cols = detect_intent(q)[1] or ["Calorie"]
        res["answer"] += ("\nWhole-plate estimate from the photo (CLIP regression on FoodBD, 80% interval): " +
                          "; ".join(f"{LABEL[c]} ~{e[c][0]:.0f} ({e[c][1]:.0f}-{e[c][2]:.0f})" for c in cols) + ".")
    if res.get("images"): rep["similar"] = res["images"]["pictures"]
    elif image: sims = P_EMB @ _IMG[image]["emb"]; rep["similar"] = [(P_PATHS[i], P_LAB[i]) for i in np.argsort(-sims)[:5]]
    return rep

## 10. Benchmark (text): generated queries with held-out surface forms

* **Forms**: `canonical`, `bangla` (in the alias table) and three **unseen** forms — `typo` (1–2 random edits), `phonetic` (Banglish respellings), `bangla_variant` (ি↔ী, ু↔ূ, শ↔স …). A generated form that collides with *another* dish's alias is dropped; one that hits its own dish's alias is marked `(seen)`.
* **Intents**: describe, nutrition (dishes linked to FoodBD), images, two-dish nutrition.
* **Must-abstain negatives** (≈ 230, ~20 % of the benchmark; v4 had 62): foreign dishes, **hard near-miss** South-Asian dishes that are not in the KB (dropped if they fuzzy-match a KB alias ≥ 88), non-food questions, invented names — asked with describe, nutrition and picture templates.
* **Size**: stratified sample capped at `BENCH_MAX` (all negatives and two-dish queries are kept). **Split** 30 % dev (tuning only) / 70 % test.

In [ ]:
BR = random.Random(42)
def surface(c):
    n = DOCS[c]["name"] if c in DOCS else c
    return norm(re.sub(r"\s*\(.*?\)", "", n).split("/")[0])
def typo(s, rng):
    for _ in range(20):
        w = list(s); pos = [i for i, ch in enumerate(w) if ch.isalpha()]
        for _ in range(1 if len(s) <= 6 else 2):
            i = rng.choice(pos); op = rng.choice(["del", "swap", "sub", "ins"])
            if op == "del" and len(w) > 4: w.pop(i)
            elif op == "swap" and i + 1 < len(w): w[i], w[i + 1] = w[i + 1], w[i]
            elif op == "sub": w[i] = rng.choice("aeioubdghklmnprst")
            else: w.insert(i, rng.choice("aeiouhy"))
            pos = [j for j, ch in enumerate(w) if ch.isalpha()]
        t = "".join(w)
        if t != s: return t
    return s + "a"
_PV = [("bh", "v"), ("b", "bh"), ("sh", "s"), ("s", "sh"), ("o", "a"), ("a", "o"), ("i", "ee"), ("u", "oo"), ("ch", "c"),
       ("k", "kh"), ("ll", "l"), ("ri", "ree"), ("ph", "f"), ("j", "z"), ("y", "i")]
def phon_variant(s, rng):
    out = s
    for _ in range(rng.choice([1, 2])):
        opts = [(a, b) for a, b in _PV if a in out]
        if not opts: break
        a, b = rng.choice(opts); i = rng.choice([m.start() for m in re.finditer(re.escape(a), out)])
        out = out[:i] + b + out[i + len(a):]
    return out if out != s else s + "h"
_BV = [("ি", "ী"), ("ী", "ি"), ("ু", "ূ"), ("ূ", "ু"), ("শ", "স"), ("স", "শ"), ("ড়", "র"), ("য়", "য"), ("ো", "ু")]
def bn_variant(s, rng):
    opts = [(a, b) for a, b in _BV if a in s]
    if not opts: return None
    a, b = rng.choice(opts); return s.replace(a, b, 1)

T = {"describe": {"en": ["what is {x}?", "tell me about {x}", "describe {x}"], "bl": ["{x} ki?", "{x} niye bolo"], "bn": ["{x} কি?", "{x} সম্পর্কে বলো"]},
     "nutrition": {"en": ["how many calories are in {x}?", "protein in {x}", "nutrition of {x}"], "bl": ["{x} er calorie koto?"], "bn": ["{x} এর ক্যালরি কত?"]},
     "images": {"en": ["show me pictures of {x}", "photos of {x}"], "bl": ["{x} er chobi dekhao"], "bn": ["{x} এর ছবি দেখাও"]}}
def q_of(intent, form, x, rng):
    lang = "bn" if form.startswith("bangla") else rng.choice(["en", "en", "bl"])
    return rng.choice(T[intent][lang]).format(x=x)

rows = []
def add(q, intent, form, gold, abstain=False):
    rows.append({"query": q, "intent": intent, "form": form, "gold": tuple(sorted(gold)), "should_abstain": abstain})
for c in sorted(DOCS):
    s = surface(c); bn = DOCS[c]["bn"]
    forms = {"canonical": s, "typo": typo(s, BR), "phonetic": phon_variant(s, BR)}
    if bn: forms["bangla"] = bn; bv = bn_variant(bn, BR); forms.update({"bangla_variant": bv} if bv else {})
    for f, x in forms.items():
        hit = ALIAS2CANON.get(norm(x))
        if f not in ("canonical", "bangla") and hit not in (None, c): continue      # collides with another dish -> ambiguous gold, drop
        f2 = f if (f in ("canonical", "bangla") or hit is None) else f + "(seen)"
        add(q_of("describe", f, x, BR), "describe", f2, [c])
        if c in FOODBD_MAP and f in ("canonical", "typo", "bangla"): add(q_of("nutrition", f, x, BR), "nutrition", f2, [c])
        if f in ("canonical", "phonetic"): add(q_of("images", f, x, BR), "images", f2, [c])
exact_sets = collections.Counter(plates["items"])
pairs = [(a, b) for a, b in itertools.combinations(sorted(FOODBD_MAP), 2)
         if FOODBD_MAP[a][1] == "exact" and FOODBD_MAP[b][1] == "exact" and exact_sets.get(frozenset([FOODBD_MAP[a][0], FOODBD_MAP[b][0]]), 0) >= 3]
for a, b in BR.sample(pairs, min(20, len(pairs))):
    add(BR.choice(["calories in {a} and {b}", "{a} ar {b} er calorie koto?", "how much protein in {a} with {b}"]).format(a=surface(a), b=surface(b)), "nutrition", "two-dish", [a, b])
# diet questions (only when FoodNetBD diet tags are attached): the dish must still be linked correctly, and the answer is the stored tag
COND_Q = {"t2d": ["diabetes", "diabetic", "ডায়াবেটিস"], "hypertension": ["high blood pressure", "high pressure"], "ckd": ["kidney disease", "কিডনি"],
          "gout": ["gout", "uric acid"], "celiac": ["gluten"], "anemia": ["anemia"], "pcos": ["PCOS"], "asthma": ["asthma"], "ibs": ["IBS"],
          "hypotension": ["low blood pressure"], "hypothyroidism": ["thyroid"], "hyperthyroidism": ["thyroid"]}
HEALTH_T = ["is {x} safe for {d}?", "can a {d} patient eat {x}?", "{x} ki {d} rogi khete pare?", "{x} and {d}"]
for c in sorted(HEALTH)[:80]:
    k = BR.choice(sorted(HEALTH[c]["tags"])); s = surface(c)
    add(BR.choice(HEALTH_T).format(x=s, d=BR.choice(COND_Q.get(k, [COND_NAME[k]]))), "describe", "health", [c])

NEG_T = ["what is {x}?", "how many calories are in {x}?", "{x} ki?", "show me pictures of {x}", "{x} er calorie koto?", "tell me about {x}", "protein in {x}"]
def add_neg(x, form):
    q = BR.choice(NEG_T).format(x=x); add(q, detect_intent(q)[0], form, [], True)
FOREIGN = ["sushi", "lasagna", "ramen", "tacos", "croissant", "paella", "pad thai", "tiramisu", "falafel", "kimchi", "pierogi", "goulash",
           "pizza margherita", "fish and chips", "bibimbap", "ratatouille", "poutine", "churros", "pho", "baklava", "hummus", "dim sum", "burrito",
           "schnitzel", "moussaka", "risotto", "gnocchi", "quiche", "crepe", "bratwurst", "shawarma pita", "tom yum", "laksa", "rendang", "adobo",
           "ceviche", "empanada", "enchilada", "nachos", "pretzel", "bagel", "pancake stack", "waffles", "cheesecake", "macaron", "souvlaki",
           "tagine", "couscous", "jollof rice", "injera", "borscht", "pelmeni", "tempura", "okonomiyaki", "takoyaki", "udon", "mochi", "bulgogi",
           "peking duck", "mapo tofu", "spring rolls", "satay", "nasi goreng", "lobster bisque", "clam chowder", "meatloaf", "mac and cheese"]
NEAR_MISS = ["masala dosa", "idli sambar", "vada pav", "pav bhaji", "dhokla", "rogan josh", "butter chicken", "palak paneer", "chole bhature",
             "pani puri", "dal makhani", "malai kofta", "rajma chawal", "upma", "poha", "thepla", "litti chokha", "momo", "thukpa", "nihari",
             "sajji", "chapli kebab", "appam", "puttu", "avial", "kothu parotta", "hyderabadi haleem", "gulab jamun", "rasmalai cake", "kulfi falooda",
             "dal baati", "sarson ka saag", "aloo tikki", "chicken 65", "goan fish curry", "vindaloo", "pongal", "bisi bele bath", "neer dosa", "sel roti"]
def SAFE(x):   # a real negative: no KB alias inside it (e.g. "hyderabadi haleem" contains "haleem" -> dropped) and no close fuzzy match
    return (norm(x) not in ALIAS2CANON and max(fratio(norm(x), a) for a in ALIAS_LIST) < 88
            and not any(h.startswith("alias:") for _, h in link_entities(x, phonetic=False).values()))
for f in FOREIGN:
    if SAFE(f): add_neg(f, "neg:foreign")
for f in NEAR_MISS:
    if SAFE(f): add_neg(f, "neg:near-miss")
if DOMAIN != "bd":
    # other domains: (1) held-out dishes of the SAME cuisine — removed from the KB, so they must be declined (hardest negatives);
    # (2) Bangladeshi dishes — another cuisine (only those that do not match a KB alias)
    for c in HELDOUT:
        if norm(c) not in ALIAS2CANON:
            for _ in range(3): add_neg(c, "neg:held-out")
    for canon_, name_, *_r in SEED_KB:
        x = norm(re.sub(r"\s*\(.*?\)", "", name_))
        if SAFE(x): add_neg(x, "neg:other-cuisine")
for q in ["what is the capital of France", "who won the world cup in 2022", "how do I fix my laptop battery", "explain quantum computing",
          "what is the weather in Dhaka today", "write a poem about rain", "convert 5 km to miles", "best smartphone under 20000 taka",
          "how to learn python fast", "what time is it in London", "tell me a joke", "who is the prime minister of Japan",
          "how far is the moon", "what is machine learning", "show me pictures of the eiffel tower", "how many calories does running burn",
          "what is the population of Bangladesh", "translate hello to french", "how do vaccines work", "recommend a good movie",
          "how to tie a tie", "what is the speed of light", "show me photos of cats", "how to make a website", "what is bitcoin",
          "ajke weather kemon", "tumi ke", "dhaka theke chittagong koto dur", "ekta golpo bolo", "bangladesher rajdhani ki",
          "আজকের আবহাওয়া কেমন", "তুমি কে", "একটি কবিতা লেখো", "ঢাকা থেকে সিলেট কত দূর", "বাংলাদেশের রাজধানী কি", "ক্রিকেট খেলা কখন"]:
    add(q, detect_intent(q)[0], "neg:non-food", [], True)
SYL = ["bo", "ra", "shi", "ku", "ta", "mo", "li", "pe", "zo", "gha", "tru", "ne", "vex", "plo", "dri"]; made = 0
while made < 70:
    w = "".join(BR.choice(SYL) for _ in range(BR.choice([2, 3])))
    if max(fratio(w, a) for a in ALIAS_LIST) >= 70: continue
    add_neg(w, "neg:invented"); made += 1

bench_all = pd.DataFrame(rows).drop_duplicates("query").reset_index(drop=True)
bench_all["strata"] = bench_all["intent"] + "|" + bench_all["form"]
keep_all = bench_all["should_abstain"] | (bench_all["form"] == "two-dish")
rest = bench_all[~keep_all]; budget = max(CFG["BENCH_MAX"] - int(keep_all.sum()), 0)
frac = min(1.0, budget / max(len(rest), 1))
# NB: groupby().apply drops the grouping column in pandas>=3, so sample by index instead
_pick = []
for _, g in rest.groupby("strata"):
    _pick += g.sample(max(1, int(round(len(g) * frac))), random_state=0).index.tolist()
sampled = rest.loc[_pick]
bench = pd.concat([bench_all[keep_all], sampled]).sort_index().reset_index(drop=True)
bench["split"] = "test"
for _, g in bench.groupby("strata"):
    idx = g.index.tolist(); BR.shuffle(idx)
    if len(idx) > 2: bench.loc[idx[:max(1, round(0.3 * len(idx)))], "split"] = "dev"
print(f"{len(bench_all)} generated -> {len(bench)} kept (BENCH_MAX={CFG['BENCH_MAX']}) |", bench["split"].value_counts().to_dict())
print(pd.crosstab(bench["form"], bench["intent"]).to_string())
print(bench.sample(8, random_state=1)[["query", "intent", "form", "gold", "should_abstain"]].to_string())

## 11. Features once, then tune every system's thresholds on dev in seconds

Objective = **balanced accuracy** (mean of accuracy on answerable and on must-abstain questions; dev is ~80 % answerable, so plain accuracy would ignore safety). Ties (within 0.005) are broken by plain accuracy, so a threshold is not pushed up to catch one extra negative at the cost of many real questions. Link-verification thresholds (evaluator score, retrieval rank, string similarity) are tuned by coordinate search with the evaluator thresholds. Correctness depends only on the decision (dish set / abstention), so tuning replays `decide()` on cached features — no generation, no NLI, no GPU.

In [ ]:
with section("benchmark features (batched GPU pass)"):
    FEATS.update(cached("bench_feats", lambda: query_features(bench["query"].tolist()), (KB_KEY, ENC.kind, GRADER.kind, len(bench), CFG["BENCH_MAX"], "v5")))

def decision_correct(df, S, TH):
    ok = []
    for q, gold, ab in zip(df["query"], df["gold"], df["should_abstain"]):
        pred, abst, _, _ = decide(FEATS[q], S, TH, [])
        ok.append(abst if ab else (not abst and set(pred) == set(gold)))
    return np.array(ok)
def balanced_acc(df, S, TH):
    a = decision_correct(df, S, TH); m = df["should_abstain"].values
    return 0.5 * a[~m].mean() + 0.5 * (a[m].mean() if m.any() else 1.0)

dev, test = bench[bench.split == "dev"], bench[bench.split == "test"]
with section("threshold tuning (all systems)"):
    for name, S in SYSTEMS.items():
        S["thr"] = dict(THR)
        if S["abstain"] in ("base", "dense"):
            conf = np.array([FEATS[q]["conf"][S["retriever"] if S["retriever"] == "base" else "dense"] for q in dev["query"]])
            grid = np.unique(np.quantile(conf, np.linspace(0.0, 0.95, 40))); sc = []
            for t in grid: S["thr"][S["abstain"]] = float(t); sc.append(balanced_acc(dev, S, S["thr"]))
            S["thr"][S["abstain"]] = float(grid[int(np.argmax(sc))])
            print(f"{name:32} {S['abstain']} threshold {S['thr'][S['abstain']]:.3f}  dev balanced acc {max(sc):.3f}")
        elif S["abstain"] == "grader":
            gkey = "g0p" if S.get("probe") else "g0"
            gs = np.array([FEATS[q][gkey].max() for q in dev["query"]])
            cands = np.unique(np.quantile(gs, np.linspace(0.02, 0.98, 17)))
            def score(TH):
                a = decision_correct(dev, S, TH); m = dev["should_abstain"].values
                return (round(0.5 * a[~m].mean() + 0.5 * (a[m].mean() if m.any() else 1.0), 3), a.mean())   # (balanced acc to 0.005-ish, accuracy)
            TH = dict(S["thr"]); best = (score(TH), dict(TH))
            link_grid = ([("link_min", v) for v in [0.0, 0.01, 0.03, 0.05, 0.1, 0.2, 0.3, 1.01]] + [("link_rank", v) for v in [0, 1, 2, 3, 5]] +
                         [("link_floor", v) for v in [0.0, 0.001, 0.003, 0.005, 0.01, 0.02]] + [("link_sim", v) for v in [101, 95, 92, 90, 88]] +
                         [("link_rep", v) for v in [1.01, 0.7, 0.5, 0.3, 0.2, 0.1]] + [("cover_min", v) for v in [0.0, 0.34, 0.5, 0.67]]) if S["link_verify"] else []
            for _round in range(2):                         # coordinate search: evaluator lo/hi, then each link threshold
                for lo in cands:
                    for hi in [h for h in cands if h > lo][:8]:
                        T2 = dict(best[1], grader_lo=float(lo), grader_hi=float(hi)); sc = score(T2)
                        if sc > best[0]: best = (sc, T2)
                for k, v in link_grid:
                    T2 = dict(best[1], **{k: v}); sc = score(T2)
                    if sc > best[0]: best = (sc, T2)
            S["thr"] = best[1]; t = S["thr"]
            print(f"{name:32} evaluator lo/hi {t['grader_lo']:.3f}/{t['grader_hi']:.3f} | link: eval>={t['link_min']:.3f} or (rank<={t['link_rank']} and eval>={t['link_floor']:.3f}) "
                  f"or sim>={t['link_sim']} or repaired>={t.get('link_rep', 1.01)} | cover>={t.get('cover_min', 0)} | dev balanced acc {best[0][0]:.3f} acc {best[0][1]:.3f}")
THR.update(SYSTEMS["FULL v8"]["thr"])

### 11a. The dev set — what it is and what was tuned on it

**Where it comes from.** Section 10 generates the benchmark questions, then splits them **by stratum** (intent × surface form, e.g. *describe × phonetic*): in every stratum with more than 2 questions, a random **30 %** go to **dev** and the rest to **test** (fixed seed, so the split is reproducible). Dev and test therefore have the same mix of forms, but **no question appears in both**.

**What dev is used for.** Only to choose thresholds: the evaluator's CORRECT / INCORRECT cut-offs (`grader_lo / grader_hi`), the link-verification thresholds, and the similarity threshold of the baselines. For every candidate threshold the cheap decision rule `decide()` is replayed on all dev questions, and the candidate with the best **dev balanced accuracy** wins (balanced accuracy = ½ × accuracy on answerable questions + ½ × accuracy on must-decline questions, so declining correctly counts as much as answering). A dev question counts as correct if the system links exactly the gold dish(es), or declines when it should.

**What is reported.** Every accuracy in section 11b and later is on **test** only, with the thresholds frozen from dev. The dev numbers printed above are tuning scores, not results.

This cell shows the dev set and saves it: `dev_set.csv`, `test_set.csv`, `dev_decisions_FULL.csv` (every dev question with the tuned FULL system's decision, correctness and heal trace) and `dev_tuning_curve.png`.

In [ ]:
DEV_VIEW = dev.assign(gold=dev["gold"].map(lambda g: ", ".join(g)))
print(f"dev: {len(dev)} questions ({int(dev['should_abstain'].sum())} must-decline) | test: {len(test)} questions ({int(test['should_abstain'].sum())} must-decline)")
print(f"overlap between dev and test questions: {len(set(dev['query']) & set(test['query']))}")
print("\nDev questions per surface form x intent:"); print(pd.crosstab(dev["form"], dev["intent"], margins=True).to_string())
print("\nThree dev examples per surface form:")
print(DEV_VIEW.groupby("form").head(3).sort_values("form")[["query", "intent", "form", "gold", "should_abstain"]].to_string(max_colwidth=60))

S_ = SYSTEMS["FULL v8"]; TH_ = S_["thr"]; rows_ = []
for q, gold, ab, form in zip(dev["query"], dev["gold"], dev["should_abstain"], dev["form"]):
    tr_ = []; pred, abst, kind, _ = decide(FEATS[q], S_, TH_, tr_)
    rows_.append({"query": q, "form": form, "gold": ", ".join(gold) or "(decline)", "pred": "(declined)" if abst else ", ".join(pred),
                  "correct": bool(abst if ab else (not abst and set(pred) == set(gold))), "trace": " | ".join(tr_)})
DEV_DEC = pd.DataFrame(rows_)
print(f"\nFULL v8 on dev with its tuned thresholds: accuracy {DEV_DEC['correct'].mean():.3f}")
print("dev accuracy by form:", DEV_DEC.groupby("form")["correct"].mean().round(3).to_dict())
print("\nDev questions FULL v8 still gets wrong (first 15):")
print(DEV_DEC[~DEV_DEC["correct"]][["query", "form", "gold", "pred", "trace"]].head(15).to_string(max_colwidth=55))

# how dev balanced accuracy changes with the evaluator's INCORRECT cut-off (other thresholds fixed at their tuned values)
import matplotlib.pyplot as plt
_g = np.array([FEATS[q]["g0p" if S_.get("probe") else "g0"].max() for q in dev["query"]])
_cands = np.unique(np.quantile(_g, np.linspace(0.02, 0.98, 25)))
_curve = [balanced_acc(dev, S_, dict(TH_, grader_lo=float(lo), grader_hi=max(float(lo) + 1e-6, TH_["grader_hi"]))) for lo in _cands]
plt.figure(figsize=(6, 3.2)); plt.plot(_cands, _curve, marker="o", ms=3); plt.axvline(TH_["grader_lo"], ls="--", color="grey")
plt.xlabel("evaluator INCORRECT cut-off (grader_lo)"); plt.ylabel("dev balanced accuracy"); plt.title("Threshold tuning on dev (FULL v8)")
plt.tight_layout(); plt.savefig(os.path.join(WORK, "dev_tuning_curve.png"), dpi=120); plt.show()

DEV_VIEW.to_csv(os.path.join(WORK, "dev_set.csv"), index=False)
test.assign(gold=test["gold"].map(lambda g: ", ".join(g))).to_csv(os.path.join(WORK, "test_set.csv"), index=False)
DEV_DEC.to_csv(os.path.join(WORK, "dev_decisions_FULL.csv"), index=False)
pd.DataFrame([{"system": n, **{k: v for k, v in S["thr"].items()}} for n, S in SYSTEMS.items()]).to_csv(os.path.join(WORK, "tuned_thresholds.csv"), index=False)
print("\nSaved in", WORK, ": dev_set.csv, test_set.csv, dev_decisions_FULL.csv, tuned_thresholds.csv, dev_tuning_curve.png")

### Run every system on the **test** split

Only the first `BENCH_GEN_N` answerable *describe* questions use the LLM (for FULL and FULL − verifier, which share the same raw generations through the cache); all other answers are extractive, which does not change decision accuracy.

In [ ]:
def judge_row(row, res):
    gold, pred = set(row["gold"]), set(res["entities"])
    ok = res["abstained"] if row["should_abstain"] else (not res["abstained"] and pred == gold)
    rank = np.inf
    if len(gold) == 1 and not row["should_abstain"]:
        g = next(iter(gold)); rk = res["ranking"]; rank = rk.index(g) + 1 if g in rk else np.inf
    gi = res.get("gen") or {}
    final_ok = verify(gi["answer"], gi["evidence"], gi["canon"])["ok"] if gi else None
    return {"correct": ok, "abstained": res["abstained"], "halluc": (not res["abstained"]) and (row["should_abstain"] or pred != gold),
            "false_abstain": res["abstained"] and not row["should_abstain"], "rank": rank, "heal_events": len(res["trace"]),
            "raw_ok": gi.get("raw_ok"), "final_ok": final_ok, "gen_stage": gi.get("stage"), "latency_ms": res["latency_ms"],
            "pred": tuple(sorted(pred)), "answer": res["answer"], "trace": " | ".join(res["trace"])}

pool = plates[~te]
llm_rows = set(test[(test.intent == "describe") & ~test.should_abstain].index[:CFG["BENCH_GEN_N"]])
RESULTS = {}
with section("run all systems on test"):
    for name, S in SYSTEMS.items():
        t1 = time.time(); out = []
        use_gen = name in ("FULL v8", "FULL - verifier")
        for i, r in test.iterrows():
            res = answer_query(r["query"], S, use_llm=use_gen and i in llm_rows, pool=pool)
            out.append(judge_row(r, res) | {"system": name, "used_llm": use_gen and i in llm_rows})
        RESULTS[name] = pd.DataFrame(out, index=test.index); print(f"  {name:30} {time.time() - t1:6.1f}s")

## 11b. Accuracy results

* **accuracy** — correct dish set when answerable, abstention when not. **balanced acc** — tuning objective.
* **halluc.** — answered but wrong (wrong dish, or answered an unanswerable question): the main safety metric.
* **abstain P/R/F1** — abstention as the positive class. **coverage** — share of answerable questions answered.
* **R@1 / R@5 / MRR** — rank of the gold dish in the evidence list. **unsupported (gen)** — generated answers still holding an unverified claim at the end.
* 95 % bootstrap CIs and exact McNemar tests (FULL v8 vs each system, same questions).

In [ ]:
from scipy.stats import binomtest
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
def boot_ci(x, n=1000, seed=0):
    x = np.asarray(x, dtype=float); rng = np.random.RandomState(seed)
    return np.percentile([x[rng.randint(0, len(x), len(x))].mean() for _ in range(n)], [2.5, 97.5])
def summarize(name):
    r = RESULTS[name]; ans = ~test["should_abstain"]
    tp = (r["abstained"] & test["should_abstain"]).sum(); fp = (r["abstained"] & ~test["should_abstain"]).sum(); fn = (~r["abstained"] & test["should_abstain"]).sum()
    P = tp / max(tp + fp, 1); R = tp / max(tp + fn, 1)
    rk = r.loc[ans & (test["gold"].map(len) == 1), "rank"]
    lo, hi = boot_ci(r["correct"]); hlo, hhi = boot_ci(r["halluc"]); g = r[r["used_llm"]]
    return {"system": name, "accuracy": r["correct"].mean(), "acc 95% CI": f"[{lo:.3f}, {hi:.3f}]",
            "balanced acc": 0.5 * r.loc[ans, "correct"].mean() + 0.5 * r.loc[~ans, "correct"].mean(),
            "halluc.": r["halluc"].mean(), "halluc 95% CI": f"[{hlo:.3f}, {hhi:.3f}]", "abstain P": P, "abstain R": R,
            "abstain F1": 2 * P * R / max(P + R, 1e-9), "coverage": 1 - r.loc[ans, "false_abstain"].mean(),
            "R@1": (rk <= 1).mean(), "R@5": (rk <= 5).mean(), "MRR": (1 / rk).mean(),
            "unsupported (gen)": (1 - g["final_ok"].astype(float)).mean() if len(g) else np.nan,
            "heal events/q": r["heal_events"].mean(), "median ms": r["latency_ms"].median()}
SUMMARY = pd.DataFrame([summarize(n) for n in SYSTEMS]).set_index("system")
print(f"TEST split: {len(test)} queries ({int(test['should_abstain'].sum())} must-abstain) | data: {'REAL' if not SMOKE else 'SYNTHETIC/SMOKE'}")
print(SUMMARY.round(3).to_string())

print("\nMcNemar exact test, FULL v8 vs ...")
f_ = RESULTS["FULL v8"]["correct"].values.astype(bool); MCNEMAR = {}
for n in SYSTEMS:
    if n == "FULL v8": continue
    o = RESULTS[n]["correct"].values.astype(bool); b, c = int((f_ & ~o).sum()), int((~f_ & o).sum())
    p = binomtest(b, b + c, 0.5).pvalue if b + c else 1.0; MCNEMAR[n] = {"full_only": b, "other_only": c, "p": float(p)}
    print(f"  {n:30} FULL-only correct={b:4d}  other-only correct={c:4d}  p={p:.2g}")

BY_FORM = pd.DataFrame({n: RESULTS[n].groupby(test["form"])["correct"].mean() for n in ["B0 naive dense (v1)", "B1 dense+threshold (v2-base)", "B2 CRAG-style", "B3 v2 linker+threshold", "FULL v8"]})
BY_FORM["n"] = test["form"].value_counts()
print("\nAccuracy by surface form (unseen forms are the honest robustness test):"); print(BY_FORM.round(3).to_string())
BY_INTENT = pd.DataFrame({n: RESULTS[n].groupby(test["intent"])["correct"].mean() for n in ["B2 CRAG-style", "B3 v2 linker+threshold", "FULL v8"]})
print("\nAccuracy by intent:"); print(BY_INTENT.round(3).to_string())

g = RESULTS["FULL v8"][RESULTS["FULL v8"]["used_llm"]]; gn = RESULTS["FULL - verifier"][RESULTS["FULL - verifier"]["used_llm"]]
GEN_EVAL = {}
if len(g):
    GEN_EVAL = {"n": int(len(g)), "raw_unsupported": float((1 - g["raw_ok"].astype(float)).mean()),
                "final_unsupported_full": float((1 - g["final_ok"].astype(float)).mean()), "final_unsupported_no_verifier": float((1 - gn["final_ok"].astype(float)).mean()),
                "stages": g["gen_stage"].value_counts().to_dict()}
    print(f"\nGeneration ({len(g)} answers, generator = {'simulated noisy (SMOKE)' if SMOKE else CFG['GEN_MODEL']}):")
    print(f"  raw generations with an unsupported claim : {GEN_EVAL['raw_unsupported']:.1%}")
    print(f"  after FULL heal loop                      : {GEN_EVAL['final_unsupported_full']:.1%}")
    print(f"  without verifier (ablation)               : {GEN_EVAL['final_unsupported_no_verifier']:.1%}")
    print("  heal stage used:", GEN_EVAL["stages"])

errs, base_err, inside, inside_raw = [], [], [], []
for _, p in plates[te].iterrows():
    r = nutrition_for_items(p["items"], pool)
    if r is None: continue
    m, lo, hi = r["stats"]["Calorie"][:3]; errs.append(abs(m - p["Calorie"])); base_err.append(abs(pool["Calorie"].mean() - p["Calorie"])); inside.append(lo <= p["Calorie"] <= hi)
    if "_raw_kcal" in r["stats"]: inside_raw.append(r["stats"]["_raw_kcal"][2] <= p["Calorie"] <= r["stats"]["_raw_kcal"][3])
NUTR_EVAL = {"plates": len(errs), "lookup_MAE_kcal": float(np.mean(errs)), "mean_baseline_MAE_kcal": float(np.mean(base_err)),
             "range_coverage_calibrated": float(np.mean(inside)), "range_coverage_minmax_v5": float(np.mean(inside_raw)) if inside_raw else None, "target": 0.8}
print("\nNutrition answers on test plates (lookup from non-test plates):", {k: round(v, 3) if isinstance(v, float) else v for k, v in NUTR_EVAL.items()})

print("\nExamples where FULL v8 fails (for error analysis):")
bad = RESULTS["FULL v8"][~RESULTS["FULL v8"]["correct"].astype(bool)].join(test[["query", "form", "gold"]])
print(bad[["query", "form", "gold", "pred", "trace"]].head(12).to_string(max_colwidth=70))

### 11c. Hand-written, free-form questions (not generated from templates)

The main benchmark is generated from templates over the KB, so its "canonical" form is partly circular (the same aliases build the
question and the linker). This small set is written by hand in the way people actually ask — code-mixed Banglish, Bangla script, diet
and recipe questions (including the ones asked in the v6 Kaggle run) — and checks **behaviour**, not only the linked dish:

| behaviour | pass if |
|---|---|
| describe / images | the right dish(es) are linked (pictures shown only if verified) |
| nutrition | right dish(es) and the answer gives numbers from data **or** says there is no data |
| recipe | right dish, the answer says the KB has no step-by-step recipe, and no generated text |
| diet | right dish, suitability comes **only** from a FoodNetBD tag (or "no tag"), and no other sentence makes a health claim |
| abstain | nothing is linked and the answer declines |

It is still small (≈ 55 items) and written by the notebook author, so treat it as a sanity check; replace or extend `HAND_SET` with questions from real users for the paper.

In [ ]:
HAND_SET = [  # (question, gold dishes (empty = must decline), behaviour)
    ("amar ma ajke kacchi biriyani banaise, eta ki jinish?", ["biriyani"], "describe"), ("tehari ar biriyani te parthokko ki?", ["tehari", "biriyani"], "describe"),
    ("panta bhat ki pohela boishakhe khay?", ["panta bhat"], "describe"), ("nehari ki dhakar khabar?", ["nehari"], "describe"),
    ("tell me about kala bhuna from chittagong", ["kala bhuna"], "describe"), ("ilish mach niye kichu bolo", ["hilsha fish"], "describe"),
    ("shorshe ilish kemon khete?", ["shorshe ilish"], "describe"), ("rui macher jhol ki", ["rui fish curry"], "describe"),
    ("dim er torkari ki", ["egg curry"], "describe"), ("চটপটি কী?", ["chotpoti"], "describe"), ("jhal muri ki", ["jhalmuri"], "describe"),
    ("bakorkhani ki puran dhakar?", ["bakorkhani"], "describe"), ("রসমালাই কোথাকার মিষ্টি?", ["roshmalai"], "describe"),
    ("chom chom ki", ["chomchom"], "describe"), ("what is borhani drink", ["borhani"], "describe"),
    ("payesh ar shemai er moddhe parthokko", ["payesh", "shemai"], "describe"), ("bhapa pitha sheet kale khay keno", ["bhapa pitha"], "describe"),
    ("singara r somucha same naki?", ["singara", "somucha"], "describe"), ("biryani ki", ["biriyani"], "describe"),
    ("tell me about khicuri", ["khichuri"], "describe"), ("roshogolla", ["roshgolla"], "describe"), ("fuska ki", ["fuchka"], "describe"),
    ("begun vorta ki", ["begun bhorta"], "describe"), ("বাকরখানি কী দিয়ে তৈরি?", ["bakorkhani"], "recipe"),
    ("beguni kibhabe banay", ["beguni"], "recipe"), ("how do you cook khichuri", ["khichuri"], "recipe"), ("piyaju recipe", ["piyaju"], "recipe"),
    ("halim kibhabe ranna kore", ["haleem"], "recipe"), ("how to make begun bhaji?", ["begun bhaja"], "recipe"),
    ("ek plate bhat ar dal e koto calorie", ["rice", "daal"], "nutrition"), ("porota te koto fat", ["porota"], "nutrition"),
    ("how much protein in chicken curry", ["chicken curry"], "nutrition"), ("রুটি আর সবজির ক্যালরি", ["ruti", "mixed vegetables"], "nutrition"),
    ("roshgolla calories", ["roshgolla"], "nutrition"), ("dim bhaji te kototuku protein", ["egg omlete"], "nutrition"),
    ("porata calorie", ["porota"], "nutrition"), ("aloo bhorta calories", ["mashed potato"], "nutrition"),
    ("fuchkar chobi dekhao", ["fuchka"], "images"), ("show me photos of jilapi", ["jilapi"], "images"), ("ruti ar dal er chobi", ["ruti", "daal"], "images"),
    ("is khichuri ok for diabetes?", ["khichuri"], "diet"), ("ডায়াবেটিস রোগী কি বিরিয়ানি খেতে পারে?", ["biriyani"], "diet"),
    ("high blood pressure e ilish khaoa jabe?", ["hilsha fish"], "diet"), ("kidney rogi daal khete pare?", ["daal"], "diet"),
    ("is ruti gluten free", ["ruti"], "diet"), ("can a diabetic eat roshgolla", ["roshgolla"], "diet"),
    ("what is sushi?", [], "abstain"), ("how to make pizza at home", [], "abstain"), ("masala dosa ki?", [], "abstain"),
    ("who won the cricket match yesterday", [], "abstain"), ("tomar naam ki", [], "abstain"), ("what is the capital of France", [], "abstain"),
    ("pad thai calories", [], "abstain"), ("ramen er chobi dekhao", [], "abstain"), ("what is ml", [], "abstain"),
]
def _canon_now(c): return c if c in DOCS else ALIAS2CANON.get(c, c)          # dishes may have been merged under another canonical name
def hand_check(q, gold, beh):
    r = answer_query(q); pred = set(r["entities"]); g = {_canon_now(c) for c in gold}; a = r["answer"]
    dish_ok = (pred == g) if g else (not pred and r["abstained"])
    lines = a.split("\n")
    if beh == "abstain": beh_ok = dish_ok
    elif beh == "recipe": beh_ok = "no step-by-step recipe" in a
    elif beh == "nutrition": beh_ok = bool(re.search(r"kcal|calorie|protein|fat|\bg\b|no nutrition data|won't give numbers", a, re.I))
    elif beh == "images": beh_ok = r.get("images") is not None
    elif beh == "diet":
        tag_lines = [l for l in lines if l.startswith("FoodNetBD tags") or "no FoodNetBD diet tag" in l or l.startswith("(Diet tags")]
        other = [s for l in lines if l not in tag_lines for s in split_sents(l)]
        beh_ok = bool(tag_lines) and not any(HEALTH_CLAIM.search(s) for s in other)
    else: beh_ok = True
    return {"question": q, "behaviour": beh, "gold": sorted(g), "pred": sorted(pred), "dish_ok": dish_ok, "behaviour_ok": beh_ok,
            "pass": dish_ok and beh_ok, "answer": a[:220].replace("\n", " | ")}
if DOMAIN == "bd":
    with section("hand-written question set"):
        HAND = pd.DataFrame([hand_check(*x) for x in HAND_SET])
    HAND_SUMMARY = HAND.groupby("behaviour")[["dish_ok", "behaviour_ok", "pass"]].mean().round(3).assign(n=HAND.groupby("behaviour").size())
    print(f"Hand-written set: {len(HAND)} questions | pass rate {HAND['pass'].mean():.3f} (dish {HAND['dish_ok'].mean():.3f}, behaviour {HAND['behaviour_ok'].mean():.3f})")
    print(HAND_SUMMARY.to_string())
    print("\nFailures:"); print(HAND[~HAND["pass"]][["question", "behaviour", "gold", "pred", "dish_ok", "behaviour_ok", "answer"]].to_string(max_colwidth=60) if (~HAND["pass"]).any() else "  none")
else:
    HAND, HAND_SUMMARY = pd.DataFrame(), pd.DataFrame()
    print(f"(the hand-written set is about Bangladeshi dishes; skipped for DOMAIN={DOMAIN})")

### Image benchmark

In [ ]:
def ece(conf, ok, bins=10):
    conf, ok = np.asarray(conf), np.asarray(ok, dtype=float); e = 0
    for i in range(bins):
        m = (conf > i / bins) & (conf <= (i + 1) / bins)
        if m.any(): e += m.mean() * abs(ok[m].mean() - conf[m].mean())
    return e
img_rows = []
for p, l in IMG_SPLITS["test"]:
    h = classify_image(p, heal=True); nh = classify_image(p, heal=False); d = h if IMG_HEAL else nh
    v5 = classify_image(p, heal=True, w_fuse=(0.5, 0.5), w_heal=(1 / 3, 1 / 3, 1 / 3))          # v5 equal-weight rule, for comparison
    img_rows.append({"label": l, "source": IMG_SRC.get(p, IMG_SOURCE), "A": h["A"], "B": h["B"], "fused": nh["top"], "fused_raw": nh["raw_conf"], "healed": h["top"], "v5_equal_weights": v5["top"],
                     "deployed": d["top"], "conf": d["conf"], "raw_conf": d["raw_conf"], "top3": [c for c, _ in d["ranked"][:3]],
                     "healed_event": bool(IMG_HEAL) and any(t.startswith("RECOGNISERS") for t in h["trace"])})
IR = pd.DataFrame(img_rows)
IMG_SUMMARY = {"top1_" + k: float((IR[k] == IR["label"]).mean()) for k in ["A", "B", "fused", "healed", "v5_equal_weights", "deployed"]}
IMG_SUMMARY.update({"weights_fuse_AB": W_FUSE, "weights_heal_ABkNN": W_HEAL, "healing_on": bool(IMG_HEAL)})
IMG_SUMMARY["top3_deployed"] = float(np.mean([l in t for l, t in zip(IR["label"], IR["top3"])]))
sel = IR["conf"] >= IMG_TAU
IMG_SUMMARY.update({"tau": IMG_TAU, "selective_coverage": float(sel.mean()),
                    "selective_accuracy": float((IR.loc[sel, "deployed"] == IR.loc[sel, "label"]).mean()) if sel.any() else float("nan"),
                    "ECE_fused_raw": float(ece(IR["fused_raw"], IR["fused"] == IR["label"])), "ECE_deployed_raw": float(ece(IR["raw_conf"], IR["deployed"] == IR["label"])),
                    "ECE_deployed_calibrated": float(ece(IR["conf"], IR["deployed"] == IR["label"])),
                    "heal_triggered": float(IR["healed_event"].mean()), "n_test": len(IR), "plate_photo_nutrition": PLATE})
pr_raw, pr_ver = [], []
for c in sorted(set(P_LAB)):
    s = image_search(c, k=5); pr_raw.append(np.mean([l == c for _, l in s["raw"]])); pr_ver.append(np.mean([l == c for _, l in s["pictures"]]) if s["pictures"] else 0)
IMG_SUMMARY.update({"picture_P@5_raw": float(np.mean(pr_raw)), "picture_P@5_verified": float(np.mean(pr_ver)),
                    "dishes_with_pictures_v5": len({l for p, l in zip(G_PATHS, G_LAB) if IMG_SRC.get(p) != "FoodBD crop"}), "dishes_with_pictures_v6": len(set(P_LAB))})
if PLATE_DET: IMG_SUMMARY["plate_item_detector"] = {k: v for k, v in PLATE_DET.items() if k != "per_item"}
print(f"Image test set ({IMG_SOURCE} + FoodBD crops, n={len(IR)}):")
for k, v in IMG_SUMMARY.items(): print(f"  {k:24} {round(v, 3) if isinstance(v, float) else v}")
# per source: DeshiFoodBD (19 dishes, possible leakage into recogniser A) vs FoodBD test-folder crops (rice, ruti, chicken ... unseen plates)
BY_SRC = IR.assign(ok=IR["deployed"] == IR["label"], okA=IR["A"] == IR["label"], okB=IR["B"] == IR["label"], ans=IR["conf"] >= IMG_TAU).groupby("source").agg(
    n=("ok", "size"), dishes=("label", "nunique"), top1_A=("okA", "mean"), top1_B=("okB", "mean"), top1_deployed=("ok", "mean"), answered=("ans", "mean"))
BY_SRC["selective_acc"] = [IR[(IR.source == s) & (IR.conf >= IMG_TAU)].pipe(lambda d: (d.deployed == d.label).mean() if len(d) else np.nan) for s in BY_SRC.index]
IMG_SUMMARY["by_source"] = BY_SRC.round(3).reset_index().to_dict(orient="records")
print("\nRecognition by image source:"); print(BY_SRC.round(3).to_string())
_fb = IR[IR.source == "FoodBD crop"]
if len(_fb):
    PER_DISH = _fb.assign(ok=_fb.deployed == _fb.label).groupby("label")["ok"].agg(["mean", "size"]).sort_values("size", ascending=False)
    print("\nFoodBD test crops, accuracy per dish (e.g. bhat = rice, ruti):"); print(PER_DISH.head(15).round(3).to_string())
if IMG_SOURCE == "DeshiFoodBD": print("Caveat: the tanziro classifier was trained on pooled DeshiFoodBD images, so recogniser A may have seen test images. Use fresh phone photos for a clean test.")

# ---- Image QA in the SAME metric table as the text benchmark ----
# A photo question ("what is this dish?") is answered correctly if the right dish is named, or - for a photo whose dish is NOT in the
# KB (OOD: other-cuisine classes, held-out dishes) - if the system declines. halluc. = a dish was named but it is wrong (incl. any
# named OOD photo). abstain P/R/F1: declining as a detector of OOD photos. coverage = answered share of in-KB photos. R@k / MRR = rank
# of the true dish in the system's ranking (in-KB photos). unsupported (gen) = generated descriptions with an unsupported claim
# after healing (FULL only, on a sample). heal events/q = recogniser-disagreement repairs + abstentions logged per photo.
def _rank_of(rank_list, lab):
    return rank_list.index(lab) + 1 if lab in rank_list else np.inf
def _order(d): return [c for c, _ in sorted(((c, v) for c, v in d.items() if c != "_unmapped"), key=lambda x: -x[1])]
_cp_ = [p for p, _ in IMG_SPLITS["calib"]]; _cl_ = [l for _, l in IMG_SPLITS["calib"]]
def _maxprob_tau(conf, ok, target):                      # classic selective-classification baseline: threshold on A's max probability
    for t in np.unique(np.round(conf, 3)):
        m = conf >= t
        if m.sum() >= 10 and ok[m].mean() >= target: return float(t)
    return float(conf.max()) if len(conf) else 1.0
_ca = np.array([max([v for c, v in _IMG[p]["A"].items() if c != "_unmapped"] or [0]) for p in _cp_])
_ok_a = np.array([(_order(_IMG[p]["A"]) or [None])[0] == l for p, l in zip(_cp_, _cl_)])
TAU_A = _maxprob_tau(_ca, _ok_a, CFG["TARGET_IMG_PRECISION"])
def _img_system(name, p):
    d = _IMG[p]
    if name.startswith("A:"): return _order(d["A"]), False, 0
    if name.startswith("A + max-prob"): o = _order(d["A"]); m = max([v for c, v in d["A"].items() if c != "_unmapped"] or [0]); return o, m < TAU_A, int(m < TAU_A)
    if name.startswith("B:"): return _order(d["B"]), False, 0
    if name.startswith("kNN"): return _order(d["K"]), False, 0
    if name.startswith("fused"): r = classify_image(p, heal=False); return [c for c, _ in r["ranked"]], False, 0
    r = classify_image(p); return [c for c, _ in r["ranked"]], r["uncertain"], len(r["trace"])
IMG_SYSTEMS = ["A: pretrained classifier (" + str(CFG["CLF_MODEL"]).split("/")[-1] + ")", "A + max-prob threshold (selective baseline)",
               "B: CLIP linear probe", "kNN gallery vote", "fused A+B (no abstention)", "FULL image (fusion + heal + calibrated abstain)"]
_rows = []
for p, l in IMG_SPLITS["test"]:
    for s in IMG_SYSTEMS:
        t0 = time.perf_counter(); o, abst, he = _img_system(s, p); ms = 1000 * (time.perf_counter() - t0) + _IMG[p].get("ms", 0.0)
        top = o[0] if o else None; ood = l == OOD
        _rows.append({"system": s, "path": p, "label": l, "ood": ood, "abstained": bool(abst or top is None),
                      "top": top, "rank": np.inf if ood else _rank_of(o, l), "heal": he, "ms": ms})
IMGQ = pd.DataFrame(_rows)
IMGQ["correct"] = np.where(IMGQ["ood"], IMGQ["abstained"], ~IMGQ["abstained"] & (IMGQ["top"] == IMGQ["label"]))
IMGQ["halluc"] = ~IMGQ["abstained"] & (IMGQ["ood"] | (IMGQ["top"] != IMGQ["label"]))
# generated descriptions for photos answered by FULL (sample): unsupported claims after the heal loop
_gen_ok = []
_full_ans = IMGQ[(IMGQ.system == IMG_SYSTEMS[-1]) & ~IMGQ.abstained & ~IMGQ.ood]
with section("image answers with generated descriptions (sample)"):
    for p in _full_ans["path"].head(max(10, CFG["BENCH_GEN_N"] // 2)):
        rep = multimodal_answer("what is this dish?", image=p); gi = (rep.get("text") or {}).get("gen") or {}
        if gi: _gen_ok.append(verify(gi["answer"], gi["evidence"], gi["canon"])["ok"])
def summarize_img(name):
    r = IMGQ[IMGQ.system == name]; ood = r["ood"].values; ab = r["abstained"].values
    tp, fp, fn = (ab & ood).sum(), (ab & ~ood).sum(), (~ab & ood).sum(); P = tp / max(tp + fp, 1); R = tp / max(tp + fn, 1)
    rk = r.loc[~r.ood, "rank"]; lo, hi = boot_ci(r["correct"]); hlo, hhi = boot_ci(r["halluc"])
    return {"system": name, "accuracy": r["correct"].mean(), "acc 95% CI": f"[{lo:.3f}, {hi:.3f}]",
            "balanced acc": 0.5 * r.loc[~r.ood, "correct"].mean() + 0.5 * (r.loc[r.ood, "correct"].mean() if r.ood.any() else np.nan),
            "halluc.": r["halluc"].mean(), "halluc 95% CI": f"[{hlo:.3f}, {hhi:.3f}]", "abstain P": P, "abstain R": R,
            "abstain F1": 2 * P * R / max(P + R, 1e-9), "coverage": 1 - r.loc[~r.ood, "abstained"].mean(),
            "R@1": (rk <= 1).mean(), "R@5": (rk <= 5).mean(), "MRR": (1 / rk).mean(),
            "unsupported (gen)": (1 - np.mean(_gen_ok)) if (name == IMG_SYSTEMS[-1] and _gen_ok) else np.nan,
            "heal events/q": r["heal"].mean(), "median ms": r["ms"].median()}
IMG_TABLE = pd.DataFrame([summarize_img(s) for s in IMG_SYSTEMS]).set_index("system")
print(f"\nIMAGE QA on {IMG_SOURCE} test photos: {len(IMG_SPLITS['test'])} ({int(sum(l == OOD for _, l in IMG_SPLITS['test']))} must-decline) | "
      f"A max-prob threshold {TAU_A:.2f} and FULL tau {IMG_TAU:.2f} both calibrated for {CFG['TARGET_IMG_PRECISION']:.0%} precision on the calibration photos")
print(IMG_TABLE.round(3).to_string())
IMG_SUMMARY["image_qa_table"] = IMG_TABLE.round(4).reset_index().to_dict(orient="records")

import matplotlib; import matplotlib.pyplot as plt
o = np.argsort(-IR["conf"].values); okc = (IR["deployed"] == IR["label"]).values[o]
cov_ = np.arange(1, len(o) + 1) / len(o); acc_ = np.cumsum(okc) / np.arange(1, len(o) + 1)
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
ax[0].plot(cov_, acc_, color="#2a6f97"); ax[0].axvline(sel.mean(), ls="--", color="#999"); ax[0].set_xlabel("coverage (answered share)"); ax[0].set_ylabel("accuracy of answered")
ax[0].set_title(f"Image risk-coverage (tau={IMG_TAU:.2f})")
S2 = SUMMARY[["accuracy", "halluc."]]; y = np.arange(len(S2))
ax[1].barh(y - 0.2, S2["accuracy"], 0.4, label="accuracy", color="#2a6f97"); ax[1].barh(y + 0.2, S2["halluc."], 0.4, label="hallucination", color="#c8553d")
ax[1].set_yticks(y); ax[1].set_yticklabels(S2.index, fontsize=8); ax[1].invert_yaxis(); ax[1].legend(fontsize=8); ax[1].set_title("Text benchmark (test split)")
plt.tight_layout(); plt.savefig(os.path.join(WORK, "benchmark.png"), dpi=130); plt.show()

## 12. External judge — a JEV-style cascade built from free open models

**What is JEV?** *JEV-as-a-Judge* (arXiv:2609.26550, also shipped in DeepEval as `JevEval`) is a **decision-only judge**: instead of writing a critique, it returns a **probability for each label** (e.g. pass / fail). Its paper's key idea is a *cascade* — **accept the cheap judge's verdict when its confidence ≥ τ, escalate to a strong reasoning judge when it is unsure**, with τ tuned on labelled items. They report near-strong-judge accuracy at a fraction of the cost, but weaker confidence on style-adversarial items and on reference-free prose.

**Is it applicable here?** The hosted JEV model needs a TypeSafe API key (fee/token limits), which this project avoids. The *method* is directly applicable and free:
* **Stage 1 (decision judge)**: `Phi-3.5-mini-instruct` in 4-bit (a different family from the Qwen generator). One forward pass, read P(“Yes”) vs P(“No”) for *"Is the response correct and fully supported by the reference (or a correct refusal)?"* → verdict + confidence, no text generated.
* **Stage 2 (escalation judge)**: `Prometheus-2 7B` (4-bit, **batched generation**) with a 1–5 rubric against a reference; the rubric scores *faithfulness*, not completeness (v5's rubric penalised short but correct answers: FULL "describe" answers had gold pass 1.0 but a Prometheus mean of 3.2). The pass cut (≥ 3 or ≥ 4) is chosen on the calibration half.
* **τ** is chosen on a calibration half of the judged items to **maximise the cascade's own agreement** with the programmatic gold labels (ties → fewer escalations), and reported on the other half. (v5 used a fixed 0.90 target for stage 1 alone; it was unreachable, τ went to 1.0 and 75 % of items were escalated.) Both judges score all items so "stage 1 only", "Prometheus only" and "cascade" can be compared honestly, with escalation rate and judge seconds. `JUDGE_N` is now 120 (fast) so each half has ~60 items.
* Because answers are judged against a **reference** (the KB description or "should decline"), we stay in the regime where the JEV paper found confidence routing works.

**GPU placement.** On Kaggle T4 x2 the judges load on **GPU 1**, which v4 left idle (v4 put everything on GPU 0, so Prometheus hit CUDA OOM and the cascade silently fell back to a heuristic). On a single GPU (Colab) the retrievers, NLI and CLIP are moved to CPU while the 7B judge runs, then moved back.

SMOKE mode uses transparent heuristic judges (not LLMs) — those numbers only test the plumbing.

In [ ]:
RUBRIC = """[Is the response about the right dish, and is every claim it makes supported by the reference? Brevity is fine: a short answer that covers only part of the reference is NOT penalised.]
Score 1: Wrong dish, or answers a question that the reference says to decline, or contradicts the reference.
Score 2: Right dish but contains a clear invented claim (a number, origin, ingredient or method that is not in the reference).
Score 3: Right dish, mostly supported, but with one vague or slightly unsupported statement.
Score 4: Right dish and every claim supported, but unclear or awkward.
Score 5: Right dish and every claim supported by the reference (even if short); or a clean refusal when the reference says to decline."""
def prometheus_prompt(q, a, ref):
    return ("###Task Description:\nAn instruction (might include an Input inside it), a response to evaluate, a reference answer that gets a score of 5, "
            "and a score rubric representing a evaluation criteria are given.\n1. Write a detailed feedback that assess the quality of the response strictly "
            "based on the given score rubric, not evaluating in general.\n2. After writing a feedback, write a score that is an integer between 1 and 5. "
            "You should refer to the score rubric.\n3. The output format should look as follows: \"Feedback: (write a feedback for criteria) [RESULT] "
            "(an integer number between 1 and 5)\"\n4. Please do not generate any other opening, closing, and explanations.\n\n"
            f"###The instruction to evaluate:\n{q}\n\n###Response to evaluate:\n{a}\n\n###Reference Answer (Score 5):\n{ref}\n\n###Score Rubrics:\n{RUBRIC}\n\n###Feedback: ")
def decision_prompt(q, a, ref):
    return (f"You are a strict grader for a Bangladeshi-food assistant.\nQuestion: {q}\nResponse: {a}\nReference: {ref}\n\n"
            "Is the response correct about the dish AND does every claim in it appear in the reference? "
            "If the reference says to decline, the response is correct only if it declines without inventing facts.\nAnswer with Yes or No only.")

def _heur_cov(a, ref):
    toks = [t for t in norm(a).split() if len(t) >= 4 and t not in STOP]; rt = set(norm(ref).split())
    return sum(t in rt for t in toks) / max(len(toks), 1)
def _declined(a): return any(w in a.lower() for w in ["won't guess", "could not find", "did you mean", "won't give numbers", "not confident"])

class DecisionJudge:
    """JEV-style: returns P(pass) from Yes/No logits, no text."""
    def __init__(self):
        self.m, self.kind = None, "heuristic decision judge (SMOKE stand-in)"
        if CFG["USE_JUDGE"] and HAS_TF and HAS_TORCH and not SMOKE:
            _LLM.pop("m", None); free_gpu()
            self.m = try_load("decision-judge:" + CFG["DECISION_JUDGE"], lambda: load_causal_lm(CFG["DECISION_JUDGE"]))
            if self.m:
                tok = self.m[0]; self.kind = CFG["DECISION_JUDGE"]
                ids = lambda ws: sorted({tok.encode(w, add_special_tokens=False)[0] for w in ws})
                self.yes, self.no = ids(["Yes", " Yes", "yes", " yes"]), ids(["No", " No", "no", " no"])
        elif not SMOKE: MODEL_STATUS["decision-judge:" + CFG["DECISION_JUDGE"]] = "FALLBACK - heuristic"
    def p_pass(self, q, a, ref):
        if self.m:
            tok, model = self.m; enc = encode_chat(tok, [{"role": "user", "content": decision_prompt(q, a, ref)}]).to(model.device)
            with torch.no_grad(): lg = model(**enc).logits[0, -1].float()
            return float(torch.sigmoid(torch.logsumexp(lg[self.yes], 0) - torch.logsumexp(lg[self.no], 0)))
        if ref.startswith("DECLINE"): return 0.92 if _declined(a) else 0.08
        if _declined(a): return 0.3
        if not numbers_in(a) <= numbers_in(ref): return 0.15
        return float(1 / (1 + np.exp(-10 * (_heur_cov(a, ref) - 0.75))))
    def unload(self):
        self.m = None; free_gpu()

def move_aux(dev):
    """Move the non-LLM GPU models to `dev` (used on single-GPU machines to make room for the 7B judge)."""
    mods = [getattr(ENC, "model", None), getattr(BASE_ENC, "model", None), getattr(GRADER, "ce", None), getattr(NLI, "model", None), CLIP[0] if CLIP else None]
    for m in mods:
        if m is None: continue
        for target in (m, getattr(m, "model", None)):
            try:
                if target is not None: target.to(dev); break
            except Exception: pass
    free_gpu()

class EscalationJudge:
    def __init__(self):
        self.m, self.kind = None, "heuristic rubric judge (SMOKE stand-in, not an LLM)"
        self.offloaded = False
        if CFG["USE_JUDGE"] and HAS_TF and HAS_TORCH and not SMOKE:
            _LLM.pop("m", None); free_gpu()
            if GPU and N_GPU == 1 and gpu_free_gb() < 6.0:
                print(f"only {gpu_free_gb():.1f} GB free on the single GPU -> moving retrievers/NLI/CLIP to CPU during the judge"); move_aux("cpu"); self.offloaded = True
            for name in (CFG["JUDGE_MODEL"], CFG["JUDGE_FALLBACK"]):
                self.m = try_load("judge:" + name, lambda: load_causal_lm(name))
                if self.m: self.kind = name + (" (same family as generator - possible self-preference)" if "Qwen" in name else ""); break
    @staticmethod
    def parse(out):
        m = re.search(r"\[RESULT\]\s*\(?([1-5])", out) or re.search(r"\b([1-5])\s*\)?\s*$", out.strip())
        return (int(m.group(1)) if m else 3), out[-300:]
    def score_many(self, rows, bs=8):
        if self.m:
            tok, model = self.m
            return [self.parse(o) for o in chat_generate_batch(tok, model, [prometheus_prompt(q, a, ref) for q, a, ref in rows], max_new_tokens=160, bs=bs)]
        return [self.score(q, a, ref) for q, a, ref in rows]
    def score(self, q, a, ref):
        if self.m:
            tok, model = self.m
            return self.parse(chat_generate(tok, model, [{"role": "user", "content": prometheus_prompt(q, a, ref)}], max_new_tokens=160))
        if ref.startswith("DECLINE"): return (5 if _declined(a) else 1), "heuristic"
        if _declined(a): return 2, "heuristic"
        if not numbers_in(a) <= numbers_in(ref): return 2, "heuristic"
        cov = _heur_cov(a, ref); return (5 if cov > 0.9 else 4 if cov > 0.75 else 3 if cov > 0.55 else 2), "heuristic"
    def unload(self):
        self.m = None; free_gpu()
        if self.offloaded: move_aux(DEVICE); self.offloaded = False

# ---- items: paired FULL vs ablation/baseline answers, with programmatic gold labels ----
DECLINE_REF = "DECLINE: this is not a Bangladeshi dish in the knowledge base (or not a food question); the correct response declines without inventing facts."
items, rs = [], np.random.RandomState(0)
desc = test[(test.intent == "describe") & ~test.should_abstain & RESULTS["FULL v8"]["used_llm"]]
neg = test[test.should_abstain]
b0_wrong = test[~test.should_abstain & (test.intent == "describe") & RESULTS["B0 naive dense (v1)"]["halluc"]]
n3 = max(CFG["JUDGE_N"] // 6, 1)
for idx_set, systems, kind in [(desc, ("FULL v8", "FULL - verifier"), "describe"), (neg, ("FULL v8", "B0 naive dense (v1)"), "must-abstain"),
                               (b0_wrong, ("B0 naive dense (v1)", "FULL v8"), "baseline-wrong-dish")]:
    for i in idx_set.sample(min(len(idx_set), n3), random_state=0).index:
        ref = DECLINE_REF if test.loc[i, "should_abstain"] else DOC_TEXT[test.loc[i, "gold"][0]]
        for sysname in systems:
            r = RESULTS[sysname].loc[i]
            gold_ok = bool(r["correct"]) and (r["final_ok"] is None or bool(r["final_ok"]) or test.loc[i, "should_abstain"])
            items.append({"idx": i, "system": sysname, "kind": kind, "query": test.loc[i, "query"], "answer": r["answer"].split("\n")[0], "reference": ref, "gold_ok": gold_ok})
JI = pd.DataFrame(items)
print(f"{len(JI)} judge items | gold pass rate {JI['gold_ok'].mean():.2f} | by kind:", JI["kind"].value_counts().to_dict())

with section("stage-1 decision judge"):
    _JKEY = hashlib.md5(str(list(zip(JI["query"], JI["answer"], JI["reference"]))).encode()).hexdigest()
    def _stage1():
        DJ_ = DecisionJudge(); t0 = time.time(); p = [DJ_.p_pass(q, a, ref) for q, a, ref in zip(JI["query"], JI["answer"], JI["reference"])]
        out = (p, (time.time() - t0) / max(len(JI), 1), DJ_.kind); DJ_.unload(); return out
    # v7: judge outputs are cached, so a re-run with the previous run's output attached skips the ~12 min of judging
    _p, T_STAGE1, _k1 = cached("judge_stage1", _stage1, (_JKEY, CFG["DECISION_JUDGE"], SMOKE), store_if=lambda o: SMOKE or "heuristic" not in o[2])
    JI["p_pass"] = _p; DJ = type("J", (), {"kind": _k1})(); print("decision judge:", DJ.kind)
with section("stage-2 escalation judge (scores every item for the comparison)"):
    def _stage2():
        EJ_ = EscalationJudge(); t0 = time.time(); sc_ = EJ_.score_many(list(zip(JI["query"], JI["answer"], JI["reference"])))
        out = (sc_, (time.time() - t0) / max(len(JI), 1), EJ_.kind); EJ_.unload(); return out
    sc, T_STAGE2, _k2 = cached("judge_stage2", _stage2, (_JKEY, CFG["JUDGE_MODEL"], RUBRIC, SMOKE), store_if=lambda o: SMOKE or "heuristic" not in o[2])
    JI["judge_score"], JI["judge_raw"] = [s for s, _ in sc], [r for _, r in sc]
    EJ = type("J", (), {"kind": _k2})(); print("escalation judge:", EJ.kind)

def kappa(a, b):
    a, b = np.asarray(a, bool), np.asarray(b, bool); po = (a == b).mean(); pe = a.mean() * b.mean() + (1 - a.mean()) * (1 - b.mean())
    return float((po - pe) / (1 - pe)) if pe < 1 else float("nan")
JI["half"] = np.where(JI["idx"] % 2 == 0, "calib", "eval")          # split by question, so both answers to a question land in the same half
cal = JI[JI.half == "calib"]
# Prometheus pass cut (score >= 3 or >= 4) and the cascade threshold tau are both chosen on the calibration half.
# v5 picked the smallest tau whose stage-1 accuracy reached a fixed 0.90 target; Phi never reached it, so tau went to 1.0 and 75 % of items
# were escalated for no gain. Now: tau maximises the calibration agreement of the *cascade itself*; ties -> fewer escalations.
PASS_CUT = max((3, 4), key=lambda k: ((cal["judge_score"] >= k) == cal["gold_ok"]).mean())
JI["s1_pass"] = JI["p_pass"] >= 0.5; JI["s1_conf"] = np.maximum(JI["p_pass"], 1 - JI["p_pass"]); JI["s2_pass"] = JI["judge_score"] >= PASS_CUT
cal = JI[JI.half == "calib"]; best_t = (-1.0, 0.0, 1.01)
for t in list(np.sort(cal["s1_conf"].unique())) + [1.01]:
    casc = np.where(cal["s1_conf"] >= t, cal["s1_pass"], cal["s2_pass"]); agr = float((casc == cal["gold_ok"]).mean()); esc = float((cal["s1_conf"] < t).mean())
    if (agr, -esc) > (best_t[0], -best_t[1]): best_t = (agr, esc, float(t))
tau = best_t[2]
JI["cascade_pass"] = np.where(JI["s1_conf"] >= tau, JI["s1_pass"], JI["s2_pass"]); JI["escalated"] = JI["s1_conf"] < tau
ev = JI[JI.half == "eval"]
JUDGE_TABLE = pd.DataFrame({
    "judge": ["stage 1 only (" + DJ.kind.split("/")[-1] + ")", f"Prometheus only, pass>={PASS_CUT} (" + EJ.kind.split("/")[-1][:30] + ")", f"JEV-style cascade (tau={tau:.2f})"],
    "agreement with gold": [(ev["s1_pass"] == ev["gold_ok"]).mean(), (ev["s2_pass"] == ev["gold_ok"]).mean(), (ev["cascade_pass"] == ev["gold_ok"]).mean()],
    "Cohen kappa": [kappa(ev["s1_pass"], ev["gold_ok"]), kappa(ev["s2_pass"], ev["gold_ok"]), kappa(ev["cascade_pass"], ev["gold_ok"])],
    "escalated": [0.0, 1.0, ev["escalated"].mean()],
    "judge s/item": [T_STAGE1, T_STAGE2, T_STAGE1 + ev["escalated"].mean() * T_STAGE2]}).set_index("judge")
def _boot_ci(ok, n_boot=2000, seed=0):
    ok = np.asarray(ok, float); r = np.random.RandomState(seed); b = [ok[r.randint(0, len(ok), len(ok))].mean() for _ in range(n_boot)]
    return f"[{np.percentile(b, 2.5):.3f}, {np.percentile(b, 97.5):.3f}]"
JUDGE_TABLE.insert(1, "95% CI", [_boot_ci(ev[c] == ev["gold_ok"]) for c in ("s1_pass", "s2_pass", "cascade_pass")])   # v7: n is small (~60)
print(f"\nJudge validation on the held-out half (n={len(ev)}):"); print(JUDGE_TABLE.round(3).to_string())
print("\nMean Prometheus score (1-5) and cascade pass rate by item kind and system:")
print(JI.groupby(["kind", "system"]).agg(prometheus_mean=("judge_score", "mean"), cascade_pass=("cascade_pass", "mean"), gold_pass=("gold_ok", "mean"), n=("gold_ok", "size")).round(3).to_string())
JUDGE_SUMMARY = {"decision_judge": DJ.kind, "escalation_judge": EJ.kind, "tau": tau, "prometheus_pass_cut": PASS_CUT, "n": len(JI), "table": JUDGE_TABLE.round(4).reset_index().to_dict(orient="records")}
JI.assign(human_label="")[["kind", "system", "query", "answer", "reference", "gold_ok", "p_pass", "judge_score", "cascade_pass", "human_label"]].to_csv(
    os.path.join(WORK, "judge_items_for_human_check.csv"), index=False)
print("\nExported judge_items_for_human_check.csv - label 30-50 rows by hand to validate the judges against people, not only against programmatic labels.")

### How to read the results
* Look first at **unseen forms** (`typo`, `phonetic`, `bangla_variant`) and the three `neg:*` rows — where v2's alias table and the English embedder break and where the heal steps should help.
* **Ablations** answer "does this heal step earn its cost?": `FULL - phonetic repair`, `FULL - link verification`, `FULL - evaluator`, `FULL - verifier`. A large McNemar p-value means the step is not shown to help on this data.
* Accuracy and hallucination trade off through abstention; always report both, plus coverage.
* `unsupported (gen)` and the judge rows are the generation-side safety metrics; compare FULL with `FULL - verifier`.
* **SMOKE mode** numbers (synthetic data, fallback models, simulated generator, heuristic judges) only prove the pipeline logic. Report numbers come from the Kaggle/Colab run with real data; check `MODEL_STATUS`.
* **Comparison with other work**: published CRAG / Self-RAG / RAGAS numbers come from English open-domain benchmarks and are not comparable to this closed-domain Bangla benchmark. The fair comparison is the one above: their *mechanisms* re-implemented on the same questions (B0 naive RAG, B1 similarity-threshold RAG, B2 CRAG-style evaluator + rewrite + decline, B3 your v2) against FULL v8.

## 13. Report: model status, timings, files

In [ ]:
print("MODEL_STATUS (what actually ran):")
for k, v in MODEL_STATUS.items(): print(f"  {k:72} {v}")
print("\nMODEL SOURCES (attached Kaggle/Colab input vs. Hugging Face download):")
for k, v in MODEL_SOURCES.items(): print(f"  {k:60} {v}")
print("DATA:", {"FoodBD": csv_path if REAL_FOODBD else "synthetic", "FoodBD_Meta_data": META_PATH, "DeshiFoodBD": desh_train, "FoodNetBD": FNB_INFO.get("path"),
               "previous-run caches": INPUT_CACHES})
print("\nTIMINGS (s):"); [print(f"  {k:55} {v:8.1f}") for k, v in TIMINGS.items()]
report = {"mode": "SMOKE (synthetic data, fallback models)" if SMOKE else "REAL", "profile": PROFILE, "n_test_queries": len(test),
          "thresholds": {n: S["thr"] for n, S in SYSTEMS.items()}, "text_summary": SUMMARY.round(4).reset_index().to_dict(orient="records"),
          "mcnemar_vs_full": MCNEMAR, "by_form": BY_FORM.round(4).to_dict(), "generation": GEN_EVAL,
          "nutrition_estimator": NUTRITION_RESULTS, "nutrition_answers": NUTR_EVAL, "image": IMG_SUMMARY, "judge": JUDGE_SUMMARY,
          "foodbd_metadata": {**META_INFO, "condition_analysis": COND_ANALYSIS, "bangla_names": len(META_BN),
                              "plate_mondrian": None if PLATE_COND is None else PLATE_COND["table"].to_dict(orient="records")},
          "foodnetbd": FNB_INFO, "kb": {"dishes": len(DOCS), "aliases": len(ALIAS2CANON), "merged_duplicates": len(KB_MERGES), "foodbd_fact_classes": len(FOODBD_FACTS)},
          "models": MODEL_STATUS, "timings_s": TIMINGS}
report["domain"] = {"name": DOMAIN, "label": DOMAIN_LABEL, "kb_file": DOMAIN_KB_PATH, "images": IMG_SOURCE, "held_out": HELDOUT}
json.dump(report, open(os.path.join(WORK, "shrag_report.json"), "w"), indent=2, default=str, ensure_ascii=False)
json.dump(report, open(os.path.join(WORK, f"shrag_report_{DOMAIN}.json"), "w"), indent=2, default=str, ensure_ascii=False)
_tag = f"{DOMAIN}{'_SMOKE' if SMOKE else ''}"
SUMMARY.assign(domain=DOMAIN, mode="SMOKE" if SMOKE else "REAL", n_test=len(test), n_must_decline=int(test["should_abstain"].sum())).to_csv(os.path.join(WORK, f"metrics_text_{_tag}.csv"))
IMG_TABLE.assign(domain=DOMAIN, mode="SMOKE" if SMOKE else "REAL", n_test=len(IMG_SPLITS["test"]),
                 n_must_decline=int(sum(l == OOD for _, l in IMG_SPLITS["test"]))).to_csv(os.path.join(WORK, f"metrics_image_{_tag}.csv"))
pd.concat([RESULTS[n].assign(query=test["query"], form=test["form"], intent=test["intent"]) for n in SYSTEMS]).to_csv(os.path.join(WORK, "benchmark_rows.csv"), index=False)
bench.to_csv(os.path.join(WORK, "benchmark_queries.csv"), index=False)
print("\nSaved in", WORK, ":", ", ".join(sorted(os.path.basename(p) for p in glob.glob(os.path.join(WORK, "*.*")))))

## 14. Demo: English, Banglish, Bangla, pictures, a photo

In [ ]:
_DEMO_Q = ["kacchi biriyani ki?", "ইলিশ মাছের ক্যালরি কত?", "show me pictures of fuchka", "khichuri ar daal er calorie koto", "what is a croissant?",
          "tell me about shorshe ilsh", "show me picture of ruti and murgir mangsho", "show me pictures of bhat", "বেগুনী কি?",
          "what is usually eaten with khichuri?", "describe beef curry plate", "how many calories in kala bhuna?", "masala dosa ki?"] if DOMAIN == "bd" else (list(test[~test.should_abstain]["query"].head(6)) + list(test[test.should_abstain]["query"].head(3)))
for q in _DEMO_Q:
    res = answer_query(q); show(res)
    if res.get("images") and res["images"]["pictures"]: show_images(res["images"]["pictures"][:6], f"verified pictures for {res['entities']}")
    print()
p, l = IMG_SPLITS["test"][0]
rep = multimodal_answer("how many calories are in this?", image=p)
print(f"Photo of (hidden label) {l} -> candidates {rep.get('image_candidates')}")
print(rep["text"]["answer"]); print("Healing trace:", rep["trace"] or "none needed")
show_images(list(rep.get("similar", []))[:5], "similar pictures")

### Try your own photo
Set `MY_IMAGE` to any image path (a Kaggle input, or a file uploaded to Colab). For a **whole-plate** photo with a nutrition question, the answer adds the photo-regression estimate. Note: FoodBD `train/` photos were used to fit that regression, so use `test/` photos or your own for a fair check.

In [ ]:
MY_IMAGE = None                      # e.g. "/kaggle/input/foodbd/FoodBD/test/images/FoodBD-1700.jpg"
MY_QUESTION = "what is this dish and how many calories are in it?"
def try_photo(img, question, note=""):
    rep = multimodal_answer(question, image=img)
    print(f"\n{note}\nQ: {question}\nAnswer: {rep['text']['answer']}\nHealing trace: {rep['trace']}")
    show_images([(img, "photo")] + list(rep.get("similar", []))[:4], "photo + most similar verified pictures")

if MY_IMAGE and os.path.exists(MY_IMAGE.split("#")[0]):
    try_photo(MY_IMAGE, MY_QUESTION, "your photo")
else:
    # examples from FoodBD's TEST folder (never used for training): one whole meal plate, and single-dish crops of bhat (rice) and ruti
    _ex = INST[(INST.split == "test") & INST.path.notna()]
    if len(_ex):
        _plate = _ex.groupby("stem").filter(lambda g: g["cls"].nunique() >= 2).drop_duplicates("stem").head(1)
        for r in _plate.itertuples():
            try_photo(r.path, MY_QUESTION, f"FoodBD test plate, annotated items: {sorted(set(_ex[_ex.stem == r.stem]['cls']))}")
    for want in ("rice", "ruti", "chicken"):
        hit = [p for p, l in CROPS["test"] if l == want or FOODBD_MAP.get(l, ("", ""))[0] == want][:1]
        if hit: try_photo(hit[0], "what is this?", f"FoodBD test crop, true dish: {want}")

## 16. Cross-dataset results — the same pipeline on other food datasets

Set `DOMAIN` in section 1 (or the environment variable `SHRAG_DOMAIN`) and *Run All* once per domain:

| `DOMAIN` | text KB (benchmark questions are generated from it) | photos | recogniser A (pretrained) | must-decline cases |
|---|---|---|---|---|
| `"bd"` | tanziro + seed KB + FoodBD + FoodNetBD | DeshiFoodBD + FoodBD crops | `tanziro/...efficientnetv2s` | foreign, near-miss, non-food, invented |
| `"indian"` | **Indian Food 101** (`indian_food.csv`, 255 dishes) — Add input → Kaggle *nehaprabhavalkar/indian-food-101* | Kaggle *Indian Food Images* (80 classes) or, if not attached, HF `rajistics/indian_food_images` (20 classes) | `rajistics/finetuned-indian-food` (ViT; trained on that HF set → treat as optimistic) | + photos of classes not in the KB (pizza, burger …), **held-out dishes**, Bangladeshi dishes |
| `"food101"` | **Food-101** class names + their **Wikipedia** summaries (Internet On) | Kaggle Food-101 folder (`images/`, `meta/test.txt`) or HF `ethz/food101` validation | `nateraw/food` (ViT, 89.1 %; trained on the train split, tested here on unseen images) | + held-out classes, Bangladeshi dishes |
| `"custom"` | `CUSTOM_KB_CSV` (any table with a name column) | `CUSTOM_IMG_ROOT` (folder per class) | `CUSTOM_CLF` (any HF image classifier) or none | as above |

Each run writes `metrics_text_<domain>.csv` and `metrics_image_<domain>.csv` with exactly the columns below (accuracy, acc 95 % CI, balanced acc, halluc., halluc 95 % CI, abstain P / R / F1, coverage, R@1, R@5, MRR, unsupported (gen), heal events/q, median ms). This cell collects every such file in this run's output **and in attached inputs** — so attach the outputs of your other domain runs (Add input → Notebook output) and the comparison table fills in. Thresholds are always re-tuned on that domain's own dev split; nothing is tuned on test.

In [ ]:
_mfiles = sorted(set(glob.glob(os.path.join(WORK, "metrics_*_*.csv")) + [p for p in IDX_FILES if re.search(r"metrics_(text|image)_[^/]+\.csv$", p)]))
_tabs = {"text": [], "image": []}
for p in _mfiles:
    kind = "text" if "metrics_text_" in p else "image"
    try: _tabs[kind].append(pd.read_csv(p, index_col=0).assign(file=os.path.basename(p)))
    except Exception as e: print("could not read", p, e)
for kind, lst in _tabs.items():
    if not lst: continue
    T = pd.concat(lst).reset_index().drop_duplicates(subset=["system", "domain", "mode"], keep="last")
    cols = ["domain", "mode", "system", "accuracy", "acc 95% CI", "balanced acc", "halluc.", "halluc 95% CI", "abstain P", "abstain R", "abstain F1",
            "coverage", "R@1", "R@5", "MRR", "unsupported (gen)", "heal events/q", "median ms", "n_test", "n_must_decline"]
    T = T[[c for c in cols if c in T.columns]]
    keep = T["system"].str.startswith(("B0", "B1", "B2", "B3", "FULL v8", "FULL image", "A:", "A +", "B:", "fused"))
    print(f"\n=== {kind.upper()} QA — main systems per domain ({T['domain'].nunique()} domain(s): {sorted(T['domain'].unique())}) ===")
    print(T[keep].set_index(["domain", "mode", "system"]).round(3).to_string())
    T.to_csv(os.path.join(WORK, f"cross_domain_{kind}.csv"), index=False)
if not any(_tabs.values()): print("no metrics files yet")
print("\nSaved: cross_domain_text.csv, cross_domain_image.csv in", WORK)

## 15. This project next to related work (auto-filled from this run)

Published numbers come from each paper's **own** benchmark (English open-domain QA, medical VQA, …), so they cannot be compared with this closed-domain Bangla/Banglish benchmark number-for-number. What *can* be compared is the **shape** of the result: how much each self-correcting method gains over its own plain-RAG baseline, and which failure it targets. The second table is filled from this run.

In [ ]:
PUBLISHED = pd.DataFrame([
    ("CRAG (Yan et al., 2024)", "PopQA, accuracy", "RAG 52.8 (SelfRAG-LLaMA2-7b)", "CRAG 59.8 · Self-CRAG 61.8", "+7.0 / +9.0 pts"),
    ("CRAG (Yan et al., 2024)", "PubHealth, accuracy", "RAG 39.0", "CRAG 75.6", "+36.6 pts"),
    ("CRAG (Yan et al., 2024)", "ARC-Challenge, accuracy", "RAG 53.2", "CRAG 68.6", "+15.4 pts"),
    ("CRAG (Yan et al., 2024)", "Biography, FactScore", "RAG 59.2", "CRAG 74.1 · Self-CRAG 86.2", "+14.9 / +27.0"),
    ("Self-RAG 7B (as reported in CRAG)", "PopQA / PubHealth / ARC", "RAG 52.8 / 39.0 / 53.2", "54.9 / 72.4 / 67.3", "+2.1 / +33.4 / +14.1"),
    ("MRAG-Bench (Hu et al., ICLR 2025)", "1,353 vision-centric MCQ", "GPT-4o, no retrieval 68.68", "retrieved images 68.96 · ground-truth images 74.50", "+0.28 with retrieval"),
    ("MMed-RAG (Xia et al., ICLR 2025)", "5 medical VQA / report sets", "Med-LVLM", "factual accuracy", "+43.8 % (avg., relative)"),
    ("JEV-as-a-Judge (Li et al., 2026)", "1,610 held-out preference pairs", "GPT-6 judge 92.5", "JEV cascade (tau 0.9) 93.4, 31.5 % escalated, ~41 % of the fee", "+0.9 pts at lower cost"),
    ("Polygon-aware nutrition (Yasin et al., 2026)", "FoodBD v2 meal nutrition", "box / whole-image features", "polygon features + XGBoost", "+11.63 % avg; instance F1 0.87"),
], columns=["work", "benchmark", "baseline", "method", "gain"])
print("Published results (their benchmarks):"); print(PUBLISHED.to_string(index=False))

def _row(n): r = SUMMARY.loc[n]; return r["accuracy"], r["halluc."], r["abstain F1"]
names = ["B0 naive dense (v1)", "B1 dense+threshold (v2-base)", "B2 CRAG-style", "B3 v2 linker+threshold", "FULL v8"]
OURS = pd.DataFrame([(n, *_row(n)) for n in names], columns=["system (same test questions)", "accuracy", "hallucination", "abstain F1"]).round(3)
a0, a2, af = OURS["accuracy"].iloc[0], OURS["accuracy"].iloc[2], OURS["accuracy"].iloc[4]
print(f"\nThis run ({'REAL' if not SMOKE else 'SMOKE'} data, {len(test)} test questions):"); print(OURS.to_string(index=False))
print(f"\n  CRAG-style mechanism vs naive RAG : {100 * (a2 - a0):+.1f} pts   (CRAG paper on PopQA: +7.0)")
print(f"  FULL v8 vs CRAG-style             : {100 * (af - a2):+.1f} pts   McNemar p = {MCNEMAR.get('B2 CRAG-style', {}).get('p', float('nan')):.2g}")
for f_ in ["phonetic", "typo", "bangla_variant", "neg:near-miss"]:
    if f_ in BY_FORM.index: print(f"  {f_:16} CRAG-style {BY_FORM.loc[f_, 'B2 CRAG-style']:.3f} -> FULL v8 {BY_FORM.loc[f_, 'FULL v8']:.3f}")
print(f"  image: recogniser top-1 {IMG_SUMMARY['top1_deployed']:.3f} | picture P@5 raw {IMG_SUMMARY['picture_P@5_raw']:.3f} -> label-verified "
      f"{IMG_SUMMARY['picture_P@5_verified']:.3f} (MRAG-Bench: retrieved images barely help, +0.28) | ECE {IMG_SUMMARY['ECE_deployed_raw']:.3f} -> {IMG_SUMMARY['ECE_deployed_calibrated']:.3f}")
print(f"  generation: unsupported claims {GEN_EVAL.get('raw_unsupported', float('nan')):.1%} raw -> {GEN_EVAL.get('final_unsupported_full', float('nan')):.1%} after the heal loop")
print(f"  judge: {JUDGE_TABLE.index[2]} agreement {JUDGE_TABLE.iloc[2, 0]:.3f} vs Prometheus-only {JUDGE_TABLE.iloc[1, 0]:.3f}, "
      f"escalating {JUDGE_TABLE.iloc[2, 2]:.0%} of items")

## Appendix: changelog of earlier versions

### What changed in v6 (after the real Kaggle run of v5)
| v5 finding on real data | cause | v6 fix |
|---|---|---|
| `FoodBD_Meta_data.csv` unused | — | **Section 2b**: the CSV is profiled, column roles are detected and printed; Bangla class names → aliases; photo conditions → per-condition error analysis + **condition-aware (Mondrian) conformal intervals** for photo nutrition; numeric per-instance columns (e.g. weight) → per-dish KB facts. All 3,523 annotated images (not only the 1,837 with nutrition) give **co-occurrence facts** ("khichuri is most often served with …") |
| "show me pictures of ruti / bhat / murgir mangsho" → *no verified pictures*; a photo of ruti or bhat could never be named | picture **and recognition** gallery = DeshiFoodBD's 19 classes only | **FoodBD polygon crops** (train folder) join the recognition + picture gallery, valid-folder crops join calibration, test-folder crops join the image test set (results per source and per dish); CLIP **ViT-L/14** (fp16) replaces ViT-B/32; multi-dish picture questions show every dish |
| FoodBD test plates (rice + daal + fish …) → "I'm not confident which dish this is" | a single-label recogniser cannot name a multi-dish meal | **plate item detector** (multi-label, trained on all annotated FoodBD photos, per-dish thresholds tuned on valid, P/R/F1 on test) + a **plate-vs-single-dish router**; a plate photo is answered with every detected dish and the nutrition of that combination |
| FoodNetBD not used | — | per-100 g nutrients **and diet tags** (safe / restrict / avoid for 12 conditions) linked to KB dishes; "is khichuri ok for diabetes?" / "ডায়াবেটিস রোগী কি বিরিয়ানি খেতে পারে?" answered only from the stored tag (no tag → no claim); new *health* benchmark form |
| judge cascade: τ = 1.00, 75 % escalated, no gain over either judge (0.792 all three); Prometheus 797 s | fixed 0.90 target unreachable for stage 1; rubric penalised short faithful answers; one item at a time | τ maximises the **cascade's** calibration agreement; faithfulness-only rubric; Prometheus pass cut tuned; **batched** Prometheus generation; 120 judged items (60 per half) |
| near-miss / foreign negatives: FULL abstain recall 0.68 / 0.82 (worse than baselines) | "dish is in the retrieval top-3" accepted repairs, but lexical retrieval is not independent of the misspelled string | the rank signal now also needs a small evaluator floor (`link_floor`, tuned on dev) |
| image "healing" lowered accuracy 0.983 → 0.973 | A (97 %), B (83 %) and kNN averaged equally, so two CLIP-based votes out-voted the strong classifier | fusion / heal weights **learned on the calibration split**; healing switched off automatically if it does not help there |
| বেগুনী → *begun* (not *beguni*); dundol vs dhundul; "beef curry plate" → *beef curry* | long/short vowel signs; duplicate tanziro entries; "plate" is a stop word | **Bangla spelling folding** (ী/ি, ূ/ু, ণ/ন, শ/ষ/স, ড়/র …); phonetic+string **near-duplicate merge**; full multi-word names may end in a stop word |
| nutrition ranges covered 65 % of test plates | min–max of the matching plates | median ± conformally calibrated half-width (validation plates) — target 80 % |
| "The evidence describes daal as a lentil soup." | generator talked about its context | meta-talk sentences count as unsupported and are dropped |
| uncertain photo → no plate estimate | early return | the whole-plate estimate does not need the dish name, so it is still given |
| KB load 299 s | 13 parquet files read one after another | parallel ranged reads |
| `kala bhuna` calories = FoodBD *beef* plates | proxy class | optional **FoodNetBD** per-100 g values for the exact dish (raw ingredients are never mapped to cooked dishes) |


### What changed in v5 (after the first real Kaggle run of v4)
| v4 finding on real data | cause | v5 fix |
|---|---|---|
| Removing link verification *raised* accuracy (0.889 → 0.952, McNemar p = 3e-11); typo/phonetic repairs like *aarh → aar*, *ybaim → baim* were rejected | the cross-encoder scored the **misspelled** question against the repaired dish; it cannot know "aarh" means "aar", so it said 0.00 | a repair is accepted when **any independent signal agrees**: evaluator score, the dish appearing in the hybrid top-k for the question/rewrites, or a very close string match. All three thresholds are tuned on dev |
| *ইলিশ মাছের ক্যালরি কত?* abstained; nutrition intent had the lowest CRAG accuracy (0.58) | the evaluator judged "how many calories…" against a *description* passage that has no numbers → irrelevant | **intent-stripped grading**: the evaluator grades the dish mention (`content_query`), because numbers come from FoodBD, not the KB. Bangla/Banglish **genitive suffixes** (-er, -এর, -ের …) are stripped before linking. New ablation row |
| thresholds over-fit (FULL lo/hi = 0.61/0.73) | only 62 must-abstain questions (≈ 19 in dev) | ≈ 230 negatives incl. **hard near-miss** South-Asian dishes; ties in balanced accuracy are broken by plain accuracy |
| Prometheus-2 and Qwen-3B judges: CUDA OOM on GPU 0, GPU 1 idle | everything was pinned to GPU 0; OOM then retried in fp16 (2× bigger) | LLMs go to GPU 1 when there are two GPUs; on one GPU, retrievers are moved to CPU while the 7B judge runs; OOM never retries in fp16 |
| CLIP recogniser B: 9 % top-1, ECE 0.47 | zero-shot over all 600 KB names with ViT-B/32 | B = CLIP **few-shot prototypes + prompts over classes that have gallery photos**; fused confidence is **Platt-calibrated** on the calibration split before τ is chosen |
| plate-photo regression took 320 s | full-resolution phone JPEGs decoded one by one | JPEG draft decoding at reduced size + threaded loading |
| picture questions got LLM text like "search an image engine" | the generator ran for picture intents | picture answers are templated (identity sentence + verified-picture count) |
| a whole-plate photo got one dish's lookup | `plate=True` had to be passed by hand | nutrition questions with a FoodBD-style plate photo get the photo-regression estimate automatically |
| generator Qwen2.5-1.5B: 62.5 % raw answers had an unsupported claim | small model | Qwen2.5-3B-Instruct (4-bit) on GPU 1 when two GPUs exist; 1.5B otherwise |
| NLI verifier, CLIP and the EfficientNet classifier ran in fp32 | no precision setting | **fp16** (autocast / fp16 weights). Quantization map: LLMs (Qwen generator, Phi-3.5 decision judge, Prometheus-2 7B) = **4-bit NF4 + double quant, fp16 compute** (T4 has no bf16); encoders/reranker/NLI/vision = **fp16** — 4/8-bit bitsandbytes is slower than fp16 for these small encoders and shifts the score thresholds |

### What changed from v3 (and why the tuning cell ran for 4+ hours)
| v3 problem | cause | v4 fix |
|---|---|---|
| "tune each system's thresholds" ran > 4 h | for **every threshold candidate** (≈ 70–80 per system) the full pipeline was re-run on all 1 284 dev queries, including GPU NLI verification of every answer; the benchmark had also grown to 4 280 queries after 588 tanziro dishes joined the KB | each query is split into **expensive features** (linking, retrieval, evaluator scores — computed once, in GPU batches, cached to disk) and a **cheap decision rule**. Tuning replays only the rule: ~seconds. Benchmark is capped by stratified sampling (`BENCH_MAX`) |
| results cell crashed (`summarize` used before definition) | ordering bug | fixed |
| "what is the capital of France" → *choi pitha + firni* | phonetic repair on the rewritten query matched English words | every fuzzy/phonetic link is **verified by the evaluator** (cross-encoder); rejected links are a logged heal event and a new ablation |
| "shorshe ilsh" → *shorshe* (mustard seed) + *ilish* | exact single-word alias consumed the token before a better two-word match was tried | linker does **greedy longest-span matching** (exact > fuzzy > phonetic within the same span length) |
| tanziro metadata downloaded ~6 GB | `load_dataset(columns=...)` still fetches whole parquet files | ranged reads with `HfFileSystem` + `pyarrow`: only the text column chunks are fetched |
| verifier ran NLI on sentences copied verbatim from evidence | — | verbatim sentences are supported by definition; NLI is batched per answer and cached |
| image cell ran the recognisers twice per image, one image at a time | — | one batched pass, cached; heal/no-heal are derived from the same predictions |
| one 7B judge, slow | — | **JEV-style cascade judge** (see section 12): a cheap decision-only judge accepts confident verdicts, only uncertain items go to Prometheus-2 7B |

## 17. Export a deploy bundle (free web app: Hugging Face Space backend + GitHub Pages frontend)

This cell writes `/kaggle/working/deploy_bundle/`: everything the web app needs **at serving time**, taken from this run — the KB, the
**dev-tuned thresholds of FULL v8**, nutrition models, the retrieval / evaluator caches, the image gallery embeddings + trained heads
(linear probe, plate detector, router, Platt calibration) and small thumbnails of the gallery photos. No dataset is needed on the server.
The backend (`shrag-app` repo) runs the **same functions as this notebook** (generated from it by `tools/extract_core.py`), so the app
behaves like the benchmarked FULL v8. A parity file (questions + this notebook's answers) lets the backend check itself on start-up.

Optional but recommended: put a Hugging Face **write** token in Kaggle *Add-ons → Secrets* as `HF_TOKEN` and set
`HF_BUNDLE_REPO = "your-hf-name/bd-food-rag-bundle"`: the bundle is uploaded to that Hugging Face **dataset** repo, and the Space
(env `SHRAG_BUNDLE_REPO`) downloads it at start-up. Re-running this notebook and this cell updates the live app.

In [ ]:
HF_BUNDLE_REPO = None                 # e.g. "fatemahossain0987/bd-food-rag-bundle" -> uploaded as a HF dataset repo (needs the HF_TOKEN secret)
BUNDLE = os.path.join(WORK, "deploy_bundle")
def export_bundle(out=BUNDLE, thumb=320):
    import shutil
    from PIL import Image as _PIL
    shutil.rmtree(out, ignore_errors=True); os.makedirs(os.path.join(out, "thumbs"), exist_ok=True); os.makedirs(os.path.join(out, "cache"), exist_ok=True)
    # gallery thumbnails (crops are rendered with their polygon mask); gallery paths become thumbnail ids
    tid = {}
    for i, p in enumerate(G_PATHS):
        name = f"thumbs/{i:05d}.jpg"
        try:
            im = _open(p, size=thumb); im.thumbnail((thumb, thumb)); im.save(os.path.join(out, name), quality=82); tid[p] = name
        except Exception: tid[p] = None
    gp = [tid.get(p) or f"missing/{i}" for i, p in enumerate(G_PATHS)]
    state = {
        "DOMAIN": DOMAIN, "DOMAIN_LABEL": DOMAIN_LABEL, "CFG": {k: v for k, v in CFG.items()},
        "DOCS": DOCS, "ALIASES": {k: set(v) for k, v in ALIASES.items()}, "ALIAS2CANON": ALIAS2CANON, "ALIAS_FOLD": ALIAS_FOLD,
        "ALIAS_LIST": ALIAS_LIST, "ALIAS_KEYS": {k: set(v) for k, v in ALIAS_KEYS.items()}, "ALIAS_KEY_NW": ALIAS_KEY_NW, "ALIAS_KEY_LIST": ALIAS_KEY_LIST,
        "FOODBD_MAP": FOODBD_MAP, "DOC_FACTS": {k: list(v) for k, v in DOC_FACTS.items()}, "NUTR100": NUTR100, "HEALTH": HEALTH,
        "DOC_IDS": DOC_IDS, "DOC_POS": DOC_POS, "DOC_TEXT": DOC_TEXT, "INDEX_TEXT": INDEX_TEXT, "KB_KEY": KB_KEY,
        "NAMES": list(NAMES), "NC": NC, "LOOKUP_S": LOOKUP_S, "Q_PRES": Q_PRES, "ridge_p": ridge_p,
        "plates": {"items": [sorted(x) for x in plates["items"]], "split": list(plates["split"]), **{c: plates[c].astype(float).tolist() for c in NUTR}},
        "FULL": {k: v for k, v in FULL.items()},
        # image branch
        "G_EMB": G_EMB.astype(np.float32), "G_LAB": np.array(G_LAB), "G_PATHS": gp, "IMG_CLASSES": IMG_CLASSES, "PROBE": PROBE,
        "W_FUSE": W_FUSE, "W_HEAL": W_HEAL, "IMG_HEAL": IMG_HEAL, "IMG_CAL": IMG_CAL, "IMG_TAU": IMG_TAU, "CAT_TAU": CAT_TAU, "IMG_SOURCE": IMG_SOURCE,
        "_rA": globals().get("_rA"), "PLATE_DET": None if PLATE_DET is None else {k: v for k, v in PLATE_DET.items() if k != "per_item"},
        "DET_CLASSES": globals().get("DET_CLASSES"), "DET_THR": globals().get("DET_THR"), "_heads": globals().get("_heads"),
        "ROUTER": globals().get("ROUTER"), "ROUTER_T": ROUTER_T, "PLATE": PLATE, "_e2e": globals().get("_e2e"), "_Qimg": globals().get("_Qimg"),
        "PLATE_COND": None if PLATE_COND is None else {k: v for k, v in PLATE_COND.items() if k != "table"}, "_cclf": globals().get("_cclf"),
        "MODEL_STATUS": MODEL_STATUS, "SMOKE": SMOKE,
        "results": {"text": SUMMARY.round(4).reset_index().to_dict(orient="records"), "image": IMG_TABLE.round(4).reset_index().to_dict(orient="records"),
                    "n_test": int(len(test)), "hand_set_pass": float(HAND["pass"].mean()) if len(HAND) else None},
    }
    with open(os.path.join(out, "state.pkl"), "wb") as f: pickle.dump(state, f, protocol=4)
    # precomputed retrieval + evaluator scores (same cache keys as here -> the server does not re-embed the KB)
    for pat in ("doc_emb_*.pkl", "grader_pairs_*.pkl"):
        for p in glob.glob(os.path.join(CACHE, pat)): shutil.copy(p, os.path.join(out, "cache"))
    GRADER.save(); [shutil.copy(p, os.path.join(out, "cache")) for p in glob.glob(os.path.join(CACHE, "grader_pairs_*.pkl"))]
    # parity samples: the server must reproduce these decisions (extractive answers, no LLM)
    par = []
    for q in list(test["query"].sample(min(60, len(test)), random_state=0)) + (list(_DEMO_Q) if "_DEMO_Q" in globals() else []):
        r = answer_query(q, use_llm=False); par.append({"q": q, "entities": r["entities"], "abstained": bool(r["abstained"]), "answer": r["answer"]})
    json.dump(par, open(os.path.join(out, "parity.json"), "w"), ensure_ascii=False, indent=1)
    import sklearn, platform
    vers = {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__, "scikit-learn": sklearn.__version__}
    json.dump(vers, open(os.path.join(out, "versions.json"), "w"), indent=1)
    open(os.path.join(out, "requirements-pinned.txt"), "w").write(f"numpy=={np.__version__}\nscikit-learn=={sklearn.__version__}\n")
    mb = sum(os.path.getsize(os.path.join(dp, f)) for dp, _, fs in os.walk(out) for f in fs) / 2**20
    print(f"deploy bundle: {out} | {len(gp)} thumbnails | {len(par)} parity questions | {mb:.1f} MB | versions {vers}")
    return out
with section("export deploy bundle"):
    export_bundle()
if HF_BUNDLE_REPO:
    try:
        from huggingface_hub import HfApi
        _api = HfApi(token=os.environ.get("HF_TOKEN"))
        _api.create_repo(HF_BUNDLE_REPO, repo_type="dataset", exist_ok=True)
        _api.upload_folder(folder_path=BUNDLE, repo_id=HF_BUNDLE_REPO, repo_type="dataset", delete_patterns=["thumbs/*", "cache/*"],
                           commit_message=f"bundle from Kaggle run ({DOMAIN}, {time.strftime('%Y-%m-%d %H:%M')})")
        print(f"uploaded to https://huggingface.co/datasets/{HF_BUNDLE_REPO} -> restart your Space (Settings -> Restart) to load it")
    except Exception as e: print("upload failed:", type(e).__name__, str(e)[:200], "-> download deploy_bundle from the notebook output and upload it by hand")